# LAB 05 Capstone Project — Multiple Linear Regression

**Project:** Historical Property Valuation (King County House Sales)
**Primary Method:** Ordinary Least Squares (OLS) Multiple Linear Regression
**Total Duration:** 6 sessions × 150 minutes ≈ 15 hours
**Audience:** LAB02–LAB06'yı tamamlamış katılımcılar

## Bu Capstone, LAB05'in Yerine Geçmez

Bu çalışma, `labs/lab05/lab05_supervised_regression.ipynb` dosyasının **devamı veya
güncellemesi değildir** — tamamen bağımsız, ayrı bir capstone projesidir. LAB05'te Northwind
verisiyle regression metodolojisinin (baseline → model → değerlendirme → karşılaştırma → seçim
→ tek seferlik test) temellerini öğrenmiştiniz. Bu capstone'da **aynı disiplin**, çok daha büyük
bir veri kümesi ve **istatistiksel çıkarıma (statistical inference)** dayalı bir yöntemle
(Ordinary Least Squares) derinlemesine uygulanacaktır.

## Bu Capstone'un Amacı

Bu capstone'u sıradan bir regression tutorial'ı olarak değil, bir **Applied Statistics ve Senior
Engineering egzersizi** olarak ele alın. Yalnızca bir model eğitmeyeceksiniz — veriyi
derinlemesine anlayacak, istatistiksel hipotezler geliştirecek, multicollinearity problemlerini
teşhis edecek, OLS varsayımlarını sistematik olarak inceleyecek ve nihai modelin sonuçlarını
**teknik olarak savunabilecek** düzeye geleceksiniz.

## Business Scenario — Historical Property Valuation

King County (Seattle bölgesi, ABD) için 2014-2015 arası gerçekleşmiş konut satışlarından
öğrenerek, konut özelliklerine dayalı bir **satış fiyatı modeli** geliştireceğiz.

> **Önemli sınırlama:** Bu çalışma **gerçek zamanlı bir production valuation sistemi değildir.**
> Model, geçmiş (historical) satış verisinden öğrenir ve yine geçmişe ait, modelin görmediği bir
> döneme ait satışları tahmin eder. Günümüz piyasa koşullarına genellenebilirliği
> **doğrulanmamıştır.**

## Dataset — King County House Sales

- **Kaynak:** [Kaggle — House Sales in King County, USA](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction)
- **Dosya:** `data/king_county_house_sales/kc_house_data.csv` (repository'ye commit edilmez,
  manuel olarak yerleştirilmelidir)
- **Target:** `price` (satış fiyatı, USD)

## Oturum Planı (Toplam ~15 Saat)

| Session | İçerik | Section'lar |
|---|---|---|
| 1 | Problem Definition & Data Quality | 01–05 |
| 2 | Univariate EDA & Distribution Analysis | 06–09 |
| 3 | Bivariate & Multivariate EDA | 10–14 |
| 4 | Feature Engineering & Multicollinearity | 15–19 |
| 5 | OLS Regression & Statistical Diagnostics | 20–24 |
| 6 | Model Refinement & Final Evaluation | 25–30 |

Her oturum 150 dakikadır (140 dakika aktif çalışma + 10 dakika ara).

## Kapsam Dışı (Out of Scope)

MLOps, Model Registry, Deployment, FastAPI/REST API, Production Monitoring, CI/CD, MLflow, Cloud
Infrastructure — bunlar bu capstone'un kapsamında **değildir**. Bu bir eğitim projesidir, ancak
bu durum kod kalitesinden veya istatistiksel doğruluktan ödün verilmesi anlamına gelmez.


---
# SESSION 1 — Problem Definition & Data Quality
**Toplam Süre: 150 dakika (140 dakika aktif + 10 dakika ara)**


---
## Section 01 — Business Problem & Modeling Contract

### Learning Objective

Bu capstone'un iş problemini, hedef değişkenini ve istatistiksel modelleme hedeflerini net bir
şekilde tanımlamak; OLS'in bir **predictive modeling** aracından farklı olarak aynı zamanda bir
**statistical inference** aracı olduğunu kavramak.

### Concept Explanation

- **Business Problem:** King County'deki geçmiş konut satışlarından öğrenerek, konut
  özelliklerine (büyüklük, konum, yapı kalitesi vb.) dayalı bir satış fiyatı tahmini üretmek.
- **Target Definition:** `price` — bir konutun gerçekleşen satış fiyatı (USD, sürekli, pozitif
  bir sayı).
- **OLS vs Predictive Modeling:** LAB05'te `scikit-learn` ile yalnızca **tahmin (prediction)**
  odaklı çalıştınız. Bu capstone'da `statsmodels` ile **OLS (Ordinary Least Squares)**
  kullanacağız — bu, hem tahmin üretir hem de her bir feature'ın katsayısı için **istatistiksel
  anlamlılık (p-value), güven aralığı ve varsayım temelli çıkarım** sağlar. Bu iki hedef (iyi
  tahmin vs güvenilir çıkarım) her zaman birbiriyle örtüşmez — bu capstone boyunca bu gerilimi
  defalarca ele alacağız.

### Engineering Motivation

Gerçek dünyada bir regression modeli yalnızca "ne kadar doğru tahmin ediyor" sorusuyla
değerlendirilmez — "bu katsayıya neden güvenebiliriz" sorusu da sorulur. Bir L5 mühendisin işi,
modelin hem tahmin performansını hem de istatistiksel geçerliliğini savunabilmektir.

### Statistical Foundations

Multiple Linear Regression'ın temel formu:

$$price_i = \beta_0 + \sum_{j=1}^{p}\beta_j x_{ij} + \varepsilon_i$$

- `price_i`: i'inci konutun satış fiyatı
- `β₀`: Intercept
- `βⱼ`: j'inci feature'ın katsayısı
- `x_{ij}`: i'inci konutun j'inci feature değeri
- `ε_i`: Error term (gözlenemeyen etkiler + ölçüm hatası)

OLS, $\sum_i \varepsilon_i^2$'yi (kareli hataların toplamını) minimize eden $\hat{\beta}$
katsayılarını bulur.

### Prediction-Time Assumptions & Feature Availability

Bu capstone'da feature'ların **"satış anında zaten bilinen, konuta ait fiziksel/konumsal
özellikler"** olduğu varsayılır (örn. oda sayısı, büyüklük, yapı yılı, konum). Bu varsayımın iki
**istisnası** vardır ve bu iki değişken için bilgi üretim zamanı belirsizliği **açıkça
belirtilir**:

- `sqft_living15`, `sqft_lot15`: Kaggle dokümantasyonuna göre bu alanlar "en yakın 15 komşunun
  güncel (2015) sqft büyüklüğü"nü temsil eder. Bu değerlerin, **her satışın gerçekleştiği tarihte
  mi** yoksa **veri kümesinin derlendiği (daha sonraki) bir anda mı** ölçüldüğü dataset'ten
  **kesin olarak çıkarılamaz**. Bu nedenle bu iki değişken, tahmin anında gerçekten bilinip
  bilinmediği doğrulanmamış bir varsayımla kullanılacaktır — Section 05'te bu risk ayrıca ele
  alınacaktır.

### Evaluation Criteria

Bu capstone'da model değerlendirmesi iki eksende yapılacaktır:

1. **Predictive accuracy** (Session 6): Validation/Test MAE, RMSE, R².
2. **Statistical validity** (Session 5): OLS varsayımlarının (linearity, homoscedasticity,
   residual normality, multicollinearity yokluğu) ne ölçüde sağlandığı.

Yüksek tahmin performansı, istatistiksel geçerliliği garanti etmez — bu capstone'un merkezi
derslerinden biri budur.

### Checkpoint

- [ ] İş problemi ve target değişken tanımlandı.
- [ ] OLS'in hem predictive hem inferential bir araç olduğu anlaşıldı.
- [ ] `sqft_living15`/`sqft_lot15` için prediction-time belirsizliği not edildi.


---
## Section 02 — Dataset Loading & Schema Validation

### Learning Objective

Dataset'i proje standartlarına uygun biçimde (config-driven, `pathlib` tabanlı) yüklemek ve
gerçek şemanın beklenen şemayla eşleştiğini **doğrulamak** — varsayımla kabul etmemek.

### Hands-On Implementation


In [ ]:
# Same environment setup pattern as LAB02-LAB06: resolve the project root, add src/ to sys.path,
# then load the shared configuration.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


In [ ]:
from data_loading import load_config, load_king_county_house_sales

config = load_config()
RANDOM_STATE = config["lab05_capstone"]["random_seed"]
TARGET = config["lab05_capstone"]["target"]

df = load_king_county_house_sales(config)
print("Dataset shape:", df.shape)
print("Target column:", TARGET)
print("Random state (from config/settings.yaml):", RANDOM_STATE)


### Expected Output

Kaggle dokümantasyonuna göre bu dataset'in beklenen profili: **21,613 gözlem, 21 kolon**. Bu
rakamları referans alıyoruz, ancak aşağıda **gerçek dosyayı okuyarak doğruluyoruz** — varsayımla
kabul etmiyoruz.

### Hands-On Implementation — Schema Inspection


In [ ]:
print("Columns:", df.columns.tolist())
df.dtypes


In [ ]:
expected_columns = [
    "id", "date", "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot",
    "floors", "waterfront", "view", "condition", "grade", "sqft_above",
    "sqft_basement", "yr_built", "yr_renovated", "zipcode", "lat", "long",
    "sqft_living15", "sqft_lot15",
]

missing_expected = set(expected_columns) - set(df.columns)
unexpected_extra = set(df.columns) - set(expected_columns)

print("Missing expected columns:", missing_expected if missing_expected else "None")
print("Unexpected extra columns:", unexpected_extra if unexpected_extra else "None")
print("Shape check -> expected (21613, 21), actual:", df.shape)

assert df.shape == (21613, 21), f"Unexpected shape: {df.shape}"
assert not missing_expected, f"Missing columns: {missing_expected}"
print("CP01 PASSED — required columns present, shape matches expected profile.")


### Hands-On Implementation — Data Type & Invalid-Type Checks


In [ ]:
numeric_expected = [c for c in expected_columns if c not in ("date",)]
non_numeric = [c for c in numeric_expected if not pd.api.types.is_numeric_dtype(df[c])]
print("Columns expected numeric but NOT numeric:", non_numeric if non_numeric else "None")

print()
print("'date' column dtype:", df["date"].dtype, "| sample value:", df["date"].iloc[0])


### Output Interpretation

`date` sütunu şu an metin (`str`) olarak okunuyor — formatı `"20141013T000000"` gibi, standart
ISO formatında değil. Bu, otomatik olarak `datetime`'a parse edilmemiştir; Section 04'te bu
alanı açıkça parse edeceğiz. Diğer tüm sütunlar (price dahil) beklenen biçimde sayısaldır.

### Hands-On Implementation — Target Validity (CP02)


In [ ]:
target_is_numeric = pd.api.types.is_numeric_dtype(df[TARGET])
target_is_finite = np.isfinite(df[TARGET]).all()
target_is_positive = (df[TARGET] > 0).all()

print(f"Target numeric : {target_is_numeric}")
print(f"Target finite  : {target_is_finite}")
print(f"Target positive: {target_is_positive}")
print()
print(df[TARGET].describe())

assert target_is_numeric and target_is_finite and target_is_positive
print("CP02 PASSED — target is numeric, finite, and strictly positive.")


### Critical Engineering Notes

- `id`, bir identifier'dır — Section 15'e kadar yalnızca property overlap analizlerinde
  kullanılacak, **doğrudan bir model feature'ı olarak kullanılmayacaktır**.
- `price`'ın kesinlikle pozitif olması (`min > 0`), Session 6'da deneyeceğimiz **log
  transformation**'ı matematiksel olarak geçerli kılar (log(0) tanımsızdır).

### Student Exercise

`df.describe(include="all")` çalıştırarak her sütunun temel istatistiklerini inceleyin.
`waterfront`, `view`, `condition`, `grade` gibi sütunların **sayısal olarak okunduğunu ama
kavramsal olarak kategorik/ordinal olabileceğini** not edin — bu konuya Section 08'de
döneceğiz.


In [ ]:
# Student Exercise:
# TODO: df.describe(include="all").T çalıştırın.
# TODO: "waterfront" ve "view" sütunlarının unique() değerlerine bakıp, bunların gerçekten
#       sürekli sayısal mı yoksa sınırlı sayıda kategori mi içerdiğini belirleyin.

# print(df.describe(include="all").T)
# print(df["waterfront"].unique(), df["view"].unique())


### Checkpoint

- [x] **CP01** — Required columns present, shape matches expected profile (21613, 21).
- [x] **CP02** — Target (`price`) is numeric, finite, and strictly positive.
- [ ] `date` sütununun parse edilmesi gerektiği not edildi (Section 04'e aktarıldı).


---
## Section 03 — Data Quality & Anomaly Assessment

### Learning Objective

Dataset'teki potansiyel veri kalitesi sorunlarını (missing values, duplicate'ler, tekrarlanan
property ID'leri, domain-invalid değerler, temporal anomaliler, structural zero'lar) sistematik
olarak **değerlendirmek** — otomatik olarak silmek değil.

### Hands-On Implementation — Missing Values & Duplicates


In [ ]:
missing_summary = df.isna().sum()
print("Missing value summary (count per column):")
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else "No missing values in any column.")
print()
print("Exact duplicate rows (all columns):", df.duplicated().sum())


### Output Interpretation

Bu dataset'te **gerçek missing value bulunmamaktadır** ve **tam (exact) duplicate satır
yoktur**. Bu, prompt'ta belirtilen "Missing Data" bölümünün açıkça belirtilmesini gerektiren bir
durumdur: **orijinal veri kümesinde gerçek missing value yoktur.** Missing-value imputation
tekniklerini (LAB04'te öğrendiğiniz `SimpleImputer` stratejileri) bu capstone'da **gerçek bir
ihtiyaç olarak göstermeyeceğiz** — bunun yerine Section 09'da kontrollü, açıkça etiketlenmiş bir
synthetic missingness demonstration'ı kullanacağız; orijinal CSV **değiştirilmeyecektir**.

### Hands-On Implementation — Repeated Property IDs


In [ ]:
n_total_rows = len(df)
n_unique_ids = df["id"].nunique()
n_repeated_id_rows = df["id"].duplicated().sum()

print(f"Total rows        : {n_total_rows}")
print(f"Unique property IDs: {n_unique_ids}")
print(f"Repeated ID rows   : {n_repeated_id_rows}  (i.e. rows beyond each property's first occurrence)")


In [ ]:
# Inspect one repeated property to understand what a "repeat" means in this dataset.
repeated_ids = df.loc[df["id"].duplicated(keep=False), "id"].unique()
sample_id = repeated_ids[0]
df[df["id"] == sample_id][["id", "date", "price", "sqft_living", "grade"]]


### Output Interpretation

`id` sütunu **21,436 benzersiz** değer içeriyor, ama toplam **21,613 satır** var — yani **177
satır**, daha önce dataset'te görülmüş bir `id`'nin **tekrarı**. Yukarıdaki örnekte görüldüğü
gibi, bu genellikle **aynı fiziksel mülkün farklı tarihlerde yeniden satılması**dır (farklı
`date`, farklı `price`, aynı `sqft_living`/`grade`) — bir veri kalitesi hatası değil, dataset'in
doğal bir özelliğidir. Bu kayıtları **silmiyoruz**; ancak bu durum Section 04'te chronological
split sırasında bir "property ID overlap across splits" riski oluşturur ve orada ayrıca ele
alınacaktır.

### Hands-On Implementation — Domain Validity: Bedrooms & Bathrooms


In [ ]:
print("Bedrooms value counts:")
print(df["bedrooms"].value_counts().sort_index())
print()
print("bedrooms == 0 count :", (df["bedrooms"] == 0).sum())
print("bedrooms == 33 count:", (df["bedrooms"] == 33).sum())
print("bathrooms == 0 count:", (df["bathrooms"] == 0).sum())


In [ ]:
df[df["bedrooms"] == 33][["id", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "price"]]


### Output Interpretation

- **`bedrooms = 0`** (13 satır): Bu, muhtemelen stüdyo daireler veya ticari/özel amaçlı
  birimlerdir (örn. "studio" tipi konutlar gerçekten 0 yatak odası olarak kodlanabilir) — ancak
  bu, dataset'ten **kesin olarak doğrulanamaz**.
- **`bedrooms = 33`** (1 satır, `id=2402100895`): `sqft_living=1620`, `sqft_lot=6000` ile
  birlikte 33 yatak odası **fiziksel olarak orantısız**dır (bu büyüklükteki bir konut için
  gerçekçi değil). En olası açıklama bir **veri girişi hatası**dır (muhtemelen `3` yerine
  yanlışlıkla `33` girilmiştir), ama bu dataset'ten **kesin olarak doğrulanamaz**. **Bu kaydı
  otomatik silmiyoruz**; aşağıda bir treatment strategy tanımlıyoruz.
- **`bathrooms = 0`** (10 satır): `bedrooms=0` ile benzer bir yapısal örüntü olabilir.

### Critical Engineering Notes — Treatment Strategy (Not Automatic Deletion)

Bu capstone, şüpheli kayıtları **otomatik silmeme** kuralına uyar. Aşağıdaki treatment stratejisi
**gerekçeli** biçimde uygulanacaktır:

| Anomali | Satır Sayısı | Strateji | Gerekçe |
|---|---|---|---|
| `bedrooms = 33` | 1 | **Flag + exclude from training** (Section 04'te, training split'e has özel bir filtre ile) | `sqft_living` (1620) ile orantısız; muhtemelen veri girişi hatası, ama kesin değil — tamamen silmek yerine modelden ayrı tutup Section 25'te influence analizinde yeniden ele alacağız. |
| `bedrooms = 0` | 13 | **Keep, flag** | Stüdyo/özel birim olabilir; `sqft_living` değerleri makul aralıkta. |
| `bathrooms = 0` | 10 | **Keep, flag** | Benzer gerekçe. |

**Bu kayıtları bu noktada silmiyoruz** — yalnızca bir `data_quality_flag` sütunu ile
işaretleyeceğiz (Section 04). Nihai treatment kararı (örn. training'den çıkarma), Section 25'teki
influence diagnostics sonuçlarına göre **yeniden gözden geçirilecektir**.

### Hands-On Implementation — Price & Lot Size Extremes


In [ ]:
print("Price describe:")
print(df["price"].describe())
print()
print("sqft_lot describe:")
print(df["sqft_lot"].describe())


### Output Interpretation

`price`, 75,000 USD ile 7,700,000 USD arasında değişiyor — geniş bir aralık, ancak hepsi pozitif
ve finansal olarak mantıklı (negatif veya sıfır fiyat yok). `sqft_lot`, 520 ile 1,651,359 arasında
— bu uç değer (maksimum), muhtemelen büyük bir arazi parçasıdır, bir veri hatası olduğu
**kesin değildir**. Bu LAB'da outlier threshold'ları **training verisinden öğrenilecek**
(Section 04/19), **önceden sabit bir eşik uydurulmayacaktır**.

### Hands-On Implementation — Temporal Anomalies


In [ ]:
df_temp = df.copy()
df_temp["date_parsed"] = pd.to_datetime(df_temp["date"], format="%Y%m%dT%H%M%S")
df_temp["sale_year"] = df_temp["date_parsed"].dt.year

yr_built_anomaly = df_temp[df_temp["yr_built"] > df_temp["sale_year"]]
yr_renovated_anomaly = df_temp[(df_temp["yr_renovated"] > 0) & (df_temp["yr_renovated"] > df_temp["sale_year"])]

print(f"Rows where yr_built > sale_year     : {len(yr_built_anomaly)}")
print(f"Rows where yr_renovated > sale_year : {len(yr_renovated_anomaly)}")
print()
print("yr_built - sale_year, for the anomalous rows:")
print((yr_built_anomaly["yr_built"] - yr_built_anomaly["sale_year"]).describe())


### Output Interpretation

**12 satırda** `yr_built`, satış yılından **tam olarak 1 yıl sonraya** denk geliyor (örn. satış
2014'te, `yr_built=2015`); **6 satırda** aynı durum `yr_renovated` için geçerli. Bu sapma her
zaman **tam olarak +1 yıl** — rastgele bir veri bozulması gibi görünmüyor. Olası bir açıklama:
bu konutlar **inşaat/renovasyon tamamlanmadan önce ("off-plan") satılmış** olabilir — `yr_built`,
satış anında planlanan tamamlanma yılını temsil edebilir. **Bu yorum doğrulanmış bir gerçek
değildir** — dataset bu açıklamayı kesin olarak doğrulamaz. Bu kayıtları **silmiyoruz**; bir
`temporal_anomaly_flag` ile işaretleyip raporluyoruz.

### Hands-On Implementation — Structural Zero Analysis


In [ ]:
print("yr_renovated == 0 count:", (df["yr_renovated"] == 0).sum(), f"({(df['yr_renovated']==0).mean():.1%} of rows)")
print("sqft_basement == 0 count:", (df["sqft_basement"] == 0).sum(), f"({(df['sqft_basement']==0).mean():.1%} of rows)")
print()
mismatch = (df["sqft_above"] + df["sqft_basement"] != df["sqft_living"]).sum()
print("Rows where sqft_above + sqft_basement != sqft_living:", mismatch)


### Output Interpretation

- **`yr_renovated = 0`** satırların **%95.8**'inde görülüyor. Bunu **missing value olarak
  yorumlamıyoruz** — dataset dokümantasyonuyla ve mantıkla tutarlı biçimde, bu **"hiç
  renove edilmemiş"** anlamına gelen bir **structural zero**'dur.
- **`sqft_basement = 0`** satırların **%60.7**'sinde görülüyor — benzer şekilde, bu **"bodrum
  bulunmuyor"** anlamına gelen bir structural zero'dur, eksik veri değildir.
- `sqft_above + sqft_basement == sqft_living` kimliği **21,613 satırın tamamında (%100)** tam
  olarak doğrulanıyor — bu, Section 18'de **perfect multicollinearity** örneği olarak doğrudan
  kullanılacak kritik bir bulgudur.

### Hands-On Implementation — Data Quality Summary


In [ ]:
data_quality_summary = pd.DataFrame([
    {"Check": "Missing values (any column)", "Count": int(df.isna().sum().sum()), "Treatment": "None needed"},
    {"Check": "Exact duplicate rows", "Count": int(df.duplicated().sum()), "Treatment": "None needed"},
    {"Check": "Repeated property IDs (extra rows)", "Count": int(df["id"].duplicated().sum()), "Treatment": "Keep; monitor split overlap (Section 04)"},
    {"Check": "bedrooms == 0", "Count": int((df["bedrooms"] == 0).sum()), "Treatment": "Keep, flag"},
    {"Check": "bedrooms == 33", "Count": int((df["bedrooms"] == 33).sum()), "Treatment": "Flag, exclude from training fit"},
    {"Check": "bathrooms == 0", "Count": int((df["bathrooms"] == 0).sum()), "Treatment": "Keep, flag"},
    {"Check": "yr_built > sale_year", "Count": int(len(yr_built_anomaly)), "Treatment": "Keep, flag (temporal anomaly)"},
    {"Check": "yr_renovated > sale_year (where renovated)", "Count": int(len(yr_renovated_anomaly)), "Treatment": "Keep, flag (temporal anomaly)"},
    {"Check": "yr_renovated == 0 (structural zero)", "Count": int((df["yr_renovated"] == 0).sum()), "Treatment": "Interpret as 'never renovated', not missing"},
    {"Check": "sqft_basement == 0 (structural zero)", "Count": int((df["sqft_basement"] == 0).sum()), "Treatment": "Interpret as 'no basement', not missing"},
])
data_quality_summary


### Critical Engineering Notes

Hiçbir kayıt bu section'da **silinmedi**. Tüm işaretlemeler (`data_quality_flag`,
`temporal_anomaly_flag`) Section 04'te gerçek DataFrame'e eklenecek ve ileriki section'larda
(özellikle Section 25 — Influence Diagnostics) bu flag'li kayıtların model üzerindeki etkisi
**ayrıca** incelenecektir.

### Student Exercise

**Session 1 Exercise:** Yukarıdaki `data_quality_summary` tablosundaki şüpheli kayıt
gruplarından birini (örn. `bathrooms == 0`) seçin. Bu kayıtların `price` ve `sqft_living`
dağılımının, şüpheli olmayan kayıtlarla karşılaştırıldığında anlamlı bir farklılık gösterip
göstermediğini inceleyin ve kendi treatment önerinizi (keep / flag / exclude) gerekçelendirerek
yazın.


In [ ]:
# Student Exercise:
# TODO: df["bathrooms"] == 0 olan satırların "price" ve "sqft_living" istatistiklerini
#       (.describe()) hesaplayıp, bathrooms > 0 olan satırlarla karşılaştırın.
# TODO: Kendi treatment öneriniz (keep / flag / exclude) için 2-3 cümlelik bir gerekçe yazın.

# suspicious = df[df["bathrooms"] == 0]
# normal = df[df["bathrooms"] > 0]
# print(suspicious[["price", "sqft_living"]].describe())
# print(normal[["price", "sqft_living"]].describe())


### Checkpoint

- [x] Missing values ve duplicate'ler incelendi — **gerçek missing value yok**.
- [x] Repeated property ID'ler sayıldı (177 satır, 21436 unique id).
- [x] Domain-invalid değerler (`bedrooms=0/33`, `bathrooms=0`) tespit edildi, **silinmedi**.
- [x] Temporal anomaliler (`yr_built`/`yr_renovated` > `sale_year`) tespit edildi ve
  gerekçeli bir yorumla not edildi.
- [x] Structural zero'lar (`yr_renovated=0`, `sqft_basement=0`) missing value'dan ayırt edildi.
- [x] `sqft_above + sqft_basement == sqft_living` kimliği %100 doğrulandı (Section 18 için not
  edildi).


---
## Section 04 — Chronological Data Splitting

### Learning Objective

Satış tarihine göre **kronolojik** bir train/validation/test ayrımı oluşturmak; aynı tarihteki
kayıtların split sınırlarında bölünmemesini sağlamak; property ID overlap riskini ölçmek.

### Engineering Motivation

LAB05'te (Northwind) kronolojik split'i zaten uygulamıştınız. Burada ek bir incelik var:
**günlük** çok sayıda satış olduğu için, satır-indeksi bazlı bir `%70` noktası genellikle bir
günün **ortasına** denk gelir — bu, aynı güne ait kayıtları farklı split'lere dağıtabilir ve
günlük piyasa koşullarına dayalı (ölçülmeyen) ortak etkilerin train ve validation arasında
sızmasına (mild bir leakage biçimine) yol açabilir. Bu nedenle split noktasını **gün
sınırlarına hizalayacağız**.

### Hands-On Implementation — Date Parsing & Chronological Sorting


In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%S")
df["sale_year"] = df["date"].dt.year

date_parse_failures = df["date"].isna().sum()
print("Date parse failures:", date_parse_failures)
print("Date range:", df["date"].min().date(), "->", df["date"].max().date())

assert date_parse_failures == 0
print("CP03 PASSED — date parsing valid, no failures.")


In [ ]:
df_sorted = df.sort_values("date").reset_index(drop=True)
df_sorted[["id", "date", "price"]].head(3)


### Hands-On Implementation — Date-Aligned Split Boundaries

Naif bir `%70`'lik satır-indeksi noktasının bir günün ortasına denk gelip gelmediğini kontrol
edelim.


In [ ]:
n = len(df_sorted)
naive_train_end_idx = int(n * 0.70)
boundary_date = df_sorted.loc[naive_train_end_idx, "date"]
rows_on_boundary_date = (df_sorted["date"] == boundary_date).sum()

print(f"Naive 70% row index: {naive_train_end_idx}")
print(f"Date at that index : {boundary_date.date()}")
print(f"Rows sharing that exact date: {rows_on_boundary_date}")


### Output Interpretation

Doğrulandığı gibi, naif `%70` satır noktası **53 kaydın paylaştığı bir günün ortasına** denk
geliyor — bu günü bölmek, aynı güne ait satışları farklı split'lere dağıtır. Bunun yerine, **gün
bazında** kümülatif sayım kullanarak `%70`/`%85` hedeflerine en yakın **tam gün sınırını**
seçeceğiz.

### Hands-On Implementation — Day-Aligned Chronological Split


In [ ]:
daily_counts = df_sorted.groupby("date").size().sort_index()
cumulative_counts = daily_counts.cumsum()

train_boundary_date = cumulative_counts[cumulative_counts <= n * 0.70].index[-1]
val_boundary_date = cumulative_counts[cumulative_counts <= n * 0.85].index[-1]

train_df = df_sorted[df_sorted["date"] <= train_boundary_date].copy()
val_df = df_sorted[(df_sorted["date"] > train_boundary_date) & (df_sorted["date"] <= val_boundary_date)].copy()
test_df = df_sorted[df_sorted["date"] > val_boundary_date].copy()

print(f"Train: {len(train_df):5d} rows ({len(train_df)/n:.4f}) | {train_df['date'].min().date()} -> {train_df['date'].max().date()}")
print(f"Val  : {len(val_df):5d} rows ({len(val_df)/n:.4f}) | {val_df['date'].min().date()} -> {val_df['date'].max().date()}")
print(f"Test : {len(test_df):5d} rows ({len(test_df)/n:.4f}) | {test_df['date'].min().date()} -> {test_df['date'].max().date()}")


### Hands-On Implementation — Split Validation (CP05, CP06)


In [ ]:
# CP06: date boundaries must not overlap (no shared date across splits).
train_dates = set(train_df["date"].unique())
val_dates = set(val_df["date"].unique())
test_dates = set(test_df["date"].unique())

date_overlap_train_val = len(train_dates & val_dates)
date_overlap_val_test = len(val_dates & test_dates)
date_overlap_train_test = len(train_dates & test_dates)

print(f"Date overlap train/val : {date_overlap_train_val}")
print(f"Date overlap val/test  : {date_overlap_val_test}")
print(f"Date overlap train/test: {date_overlap_train_test}")

assert date_overlap_train_val == 0 and date_overlap_val_test == 0 and date_overlap_train_test == 0
print("CP06 PASSED — chronological split boundaries are clean (no shared dates across splits).")


In [ ]:
# CP05: row-level overlap (should never happen with disjoint date ranges, but verify explicitly).
row_overlap_train_val = len(set(train_df.index) & set(val_df.index))
row_overlap_val_test = len(set(val_df.index) & set(test_df.index))
row_overlap_train_test = len(set(train_df.index) & set(test_df.index))

print(f"Row index overlap train/val : {row_overlap_train_val}")
print(f"Row index overlap val/test  : {row_overlap_val_test}")
print(f"Row index overlap train/test: {row_overlap_train_test}")

assert row_overlap_train_val == 0 and row_overlap_val_test == 0 and row_overlap_train_test == 0
print("CP05 PASSED — no row overlap between splits.")


### Hands-On Implementation — Property ID Overlap Analysis

Aynı fiziksel mülkün (aynı `id`) farklı split'lerde bulunması, **satır** düzeyinde bir overlap
değildir (her satır tek bir split'e aittir), ama **mülk** düzeyinde bir örtüşmedir — Section
03'te belirlediğimiz 177 tekrarlanan satıştan bazıları, kronolojik sıralama nedeniyle farklı
split'lere düşebilir. Bunu ölçelim.


In [ ]:
train_ids = set(train_df["id"])
val_ids = set(val_df["id"])
test_ids = set(test_df["id"])

overlap_train_val = train_ids & val_ids
overlap_val_test = val_ids & test_ids
overlap_train_test = train_ids & test_ids
all_overlapping_ids = overlap_train_val | overlap_val_test | overlap_train_test

print(f"Property IDs in both train and val : {len(overlap_train_val)}")
print(f"Property IDs in both val and test  : {len(overlap_val_test)}")
print(f"Property IDs in both train and test: {len(overlap_train_test)}")
print(f"Total distinct properties appearing in more than one split: {len(all_overlapping_ids)}")


### Output Interpretation

**134 farklı mülk**, birden fazla split'te görünüyor (69 train/val, 8 val/test, 57 train/test) —
bunlar, gözlem penceresi içinde **iki kez satılmış** ve ikinci satışı kronolojik olarak farklı
bir split'e düşmüş mülklerdir. Bu, **satır düzeyinde bir leakage değildir** (her satır farklı bir
satış olayını temsil eder, aynı satır iki kez görünmüyor), ama şu riski taşır: eğer bir mülkün
**fiziksel özellikleri** (örn. `sqft_living`, `grade`) satışlar arasında değişmemişse, model
training'de o mülkü "görmüş" olabilir ve validation'da aynı mülkü (farklı bir satış tarihiyle)
yeniden karşılaştığında yapay olarak iyi performans gösterebilir.

Bu, **chronological split'in yerine property-disjoint bir split'e otomatik geçmemizi
gerektirmez** — prompt'ta da belirtildiği gibi, kronolojik split'in asıl amacı zaman
boyutundaki genellemeyi test etmektir. Bunun yerine bu örtüşmeyi **açıkça raporluyoruz** ve
isteğe bağlı bir **property-disjoint sensitivity analysis**'i Session 4/6'da (gerekirse) ayrı bir
kontrol olarak ele alabiliriz.

### Critical Engineering Notes

- Chronological split, **134/21613 (%0.62)** mülk için hafif bir örtüşme taşıyor — bu oran
  düşüktür ve chronological split'in temel mantığını geçersiz kılmaz.
- Bu noktadan sonra `train_df`, `val_df`, `test_df` bu capstone'un **tüm sonraki section'larında
  tekrar kullanılacaktır**.

### Student Exercise

**Session 1 Exercise (devam):** `overlap_train_test` kümesindeki mülklerden birini seçip,
`df[df["id"] == <seçilen_id>]` ile iki satışını yan yana inceleyin. Satış fiyatı, aynı mülk için
zaman içinde nasıl değişmiş? Bu, modelin zaman içindeki fiyat değişimini (appreciation/
depreciation) tek başına yakalayıp yakalayamayacağı hakkında ne söylüyor?


In [ ]:
# Student Exercise:
# TODO: overlap_train_test kümesinden bir id seçin (örn. list(overlap_train_test)[0]).
# TODO: df[df["id"] == secilen_id][["date", "price", "sqft_living", "grade"]] ile iki satışı karşılaştırın.

# secilen_id = list(overlap_train_test)[0]
# print(df[df["id"] == secilen_id][["date", "price", "sqft_living", "grade"]])


### Checkpoint

- [x] **CP03** — Date parsing valid (0 failures).
- [x] **CP05** — No row-level overlap between train/validation/test.
- [x] **CP06** — Chronological split boundaries are date-aligned (0 shared dates across splits).
- [x] Property ID overlap across splits measured and reported (134 properties, %0.62) — **not**
  used to override the chronological split strategy.


---
## Section 05 — Leakage & Integrity Checkpoint

### Learning Objective

Session 1'i kapatmadan önce, target separation, feature availability ve temporal integrity'yi
tek bir sistematik checkpoint turunda doğrulamak.

### Hands-On Implementation — Target Separation


In [ ]:
feature_candidate_columns = [c for c in df.columns if c not in ("id", "date", "price", "sale_year")]
print("Candidate feature columns (target and identifiers excluded):")
print(feature_candidate_columns)
print()
print(f"'{TARGET}' in candidate features: {TARGET in feature_candidate_columns}")

assert TARGET not in feature_candidate_columns
print("CP07 (partial) PASSED — target is not present in the candidate feature set.")


### Hands-On Implementation — Feature Availability Review

Section 01'de belirttiğimiz prediction-time varsayımını somut bir tabloya dökelim.


In [ ]:
feature_availability = pd.DataFrame([
    {"Feature": "bedrooms, bathrooms, sqft_living, sqft_lot, floors", "Known at Sale Time?": "Yes (physical attributes)", "Leakage Risk": "None"},
    {"Feature": "waterfront, view, condition, grade", "Known at Sale Time?": "Yes (assessed property attributes)", "Leakage Risk": "None"},
    {"Feature": "yr_built, yr_renovated", "Known at Sale Time?": "Yes (historical facts about the property)", "Leakage Risk": "None"},
    {"Feature": "zipcode, lat, long", "Known at Sale Time?": "Yes (location)", "Leakage Risk": "None"},
    {"Feature": "sqft_above, sqft_basement", "Known at Sale Time?": "Yes (physical attributes)", "Leakage Risk": "None"},
    {"Feature": "sqft_living15, sqft_lot15", "Known at Sale Time?": "UNVERIFIED — may reflect neighbor sqft as of dataset compilation, not as of this sale's date", "Leakage Risk": "Possible (documented assumption, not proven)"},
])
feature_availability


### Critical Engineering Notes

`sqft_living15`/`sqft_lot15` için **"UNVERIFIED"** etiketi kasıtlıdır — bu capstone bu iki
değişkeni Session 4'te (Section 15-19) feature engineering/selection sürecine dahil edecek, ancak
her kullanımda bu belirsizliği **tekrar hatırlatacaktır**. Bu, Section 01'de verilen sözün somut
bir takibidir.

### Hands-On Implementation — Prediction Contract Validation


In [ ]:
prediction_contract_checks = {
    "Target separated from features": TARGET not in feature_candidate_columns,
    "Identifier (id) excluded from features": "id" not in feature_candidate_columns,
    "Date excluded from numeric features (handled separately)": "date" not in feature_candidate_columns,
    "No row-level split overlap": (row_overlap_train_val == 0) and (row_overlap_val_test == 0) and (row_overlap_train_test == 0),
    "Chronological split boundaries are date-clean": (date_overlap_train_val == 0) and (date_overlap_val_test == 0) and (date_overlap_train_test == 0),
}

for check, passed in prediction_contract_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {check}")

assert all(prediction_contract_checks.values())
print()
print("CP07 PASSED — no unintended target leakage detected at this stage.")


### Output Interpretation

Session 1 sonunda elimizde: temiz (gerçek missing value'suz, duplicate'siz), işaretlenmiş
(flagged ama silinmemiş) bir dataset; tarih sınırlarına hizalı, satır ve tarih düzeyinde
örtüşmeyen bir train/validation/test ayrımı; ve feature availability konusunda açıkça
belgelenmiş bir varsayım seti var. Bu, Session 2'deki EDA çalışmasının üzerine inşa edileceği
sağlam bir temel oluşturuyor.

### Student Exercise

Session 1'i tamamlayan bir katılımcı olarak: bu dataset'teki **en riskli 3 kaydı** (Section
03-05'teki bulgulara göre) kendi gerekçenizle sıralayın ve her biri için bir treatment stratejisi
önerin (keep / flag / exclude-from-training). Bu, Session 1'in resmi alıştırmasıdır.

### Checkpoint

- [x] **CP07** — No unintended target leakage detected (target separation, identifier
  exclusion, split integrity all verified).
- [x] Feature availability table documented, including the explicit `sqft_living15`/
  `sqft_lot15` uncertainty.
- [x] Session 1 complete — `df`, `train_df`, `val_df`, `test_df` ready for Session 2 (EDA).

---

## Session 1 Özeti

Bu oturumda:

- İş problemini (**Historical Property Valuation**) ve hedef değişkeni (`price`) tanımladık.
- Dataset'i config-driven bir loader ile yükledik ve gerçek şemayı (21613×21) doğruladık.
- Veri kalitesi sorunlarını (tekrarlanan ID'ler, domain-invalid değerler, temporal anomaliler,
  structural zero'lar) **sildirmeden** sistematik olarak değerlendirdik.
- Veriyi **gün-sınırına hizalı** kronolojik train/validation/test kümelerine ayırdık (%69.9/
  %15.0/%15.1) ve property ID overlap riskini ölçtük.
- Target leakage ve feature availability konusunda bir Leakage & Integrity Checkpoint
  gerçekleştirdik.

Session 2'de bu temiz, bölünmüş veri üzerinde **Univariate EDA**'ya başlayacağız.


---
# SESSION 2 — Univariate EDA & Distribution Analysis
**Toplam Süre: 150 dakika (140 dakika aktif + 10 dakika ara)**

### EDA Veri Kapsamı Hakkında Önemli Bir Tasarım Kararı

Session 1'de `train_df`/`val_df`/`test_df` ayrımını oluşturduk. **Bu noktadan itibaren, Session
2 ve Session 3'teki tüm EDA (univariate ve bivariate/multivariate) yalnızca `train_df` üzerinde
yapılacaktır.** `val_df` ve `test_df`'e bu session'larda **bakmıyoruz**.

**Gerekçe:** Eğer feature dağılımlarını, outlier eşiklerini veya feature-target ilişkilerini
validation/test verisine bakarak keşfedersek, bu bilgi dolaylı olarak feature engineering ve
model seçim kararlarımıza sızabilir — bu, Section 01'de tanımladığımız "Data-driven outlier
thresholds training verisinden öğrenilir" kuralının EDA aşamasına kadar genişletilmiş hâlidir.
Validation ve test setleri, Session 5 ve Session 6'ya kadar **görülmemiş** kalacaktır.


---
## Section 06 — Target Distribution

### Learning Objective

`price` değişkeninin dağılımını (merkezi eğilim, yayılım, çarpıklık, kurtosis) hem sayısal hem
görsel olarak analiz etmek; log transformation'ın dağılım şeklini nasıl etkiledigini gözlemlemek.

### Hands-On Implementation — Descriptive Statistics


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Color palette: fixed categorical/semantic roles reused across the ml-labs notebook series.
BAR_COLOR = "#2a78d6"
HIGHLIGHT_COLOR = "#1baf7a"
WARNING_COLOR = "#eb6834"
GRID_COLOR = "#e1e0d9"


def style_axes(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)


In [ ]:
price = train_df["price"]
print(price.describe())
print()
print(f"Mean  : {price.mean():,.2f}")
print(f"Median: {price.median():,.2f}")
print(f"Mean - Median: {price.mean() - price.median():,.2f}  (mean > median -> right-skewed signal)")


In [ ]:
print(f"Skewness: {price.skew():.3f}")
print(f"Kurtosis (excess): {price.kurtosis():.3f}")


### Output Interpretation

`price`'ın ortalaması (≈539,064) medyanından (450,000) belirgin biçimde yüksek — bu, dağılımın
**sağa çarpık** olduğuna işaret eder. Skewness (≈4.26) ve excess kurtosis (≈39.5) bu izlenimi
doğruluyor: **çok sağa çarpık, çok ağır kuyruklu** bir dağılım. Bu, konut fiyatları için tipik bir
örüntüdür — çoğu konut orta fiyat aralığında, az sayıda konut çok yüksek fiyatlarda.

### Hands-On Implementation — Histogram, KDE, Boxplot


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].hist(price, bins=50, color=BAR_COLOR)
axes[0].axvline(price.mean(), color=WARNING_COLOR, linewidth=2, label=f"Mean ({price.mean():,.0f})")
axes[0].axvline(price.median(), color=HIGHLIGHT_COLOR, linewidth=2, label=f"Median ({price.median():,.0f})")
axes[0].set_title("Price — Histogram")
axes[0].set_xlabel("Price (USD)")
axes[0].legend(frameon=False, fontsize=8)
style_axes(axes[0])

sns.kdeplot(price, ax=axes[1], color=BAR_COLOR, fill=True)
axes[1].set_title("Price — KDE")
axes[1].set_xlabel("Price (USD)")
style_axes(axes[1])

axes[2].boxplot(price, orientation="vertical", patch_artist=True, boxprops={"facecolor": BAR_COLOR, "alpha": 0.6})
axes[2].set_title("Price — Boxplot")
axes[2].set_ylabel("Price (USD)")
style_axes(axes[2])

plt.tight_layout()
plt.show()


### Hands-On Implementation — Quantiles & Extreme Values


In [ ]:
quantiles = price.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
print("Quantiles:")
print(quantiles)


In [ ]:
q1, q3 = price.quantile(0.25), price.quantile(0.75)
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr

n_extreme_high = (price > upper_bound).sum()
n_extreme_low = (price < lower_bound).sum()

print(f"IQR bounds: [{lower_bound:,.0f}, {upper_bound:,.0f}]")
print(f"Extreme-high count (price > upper bound): {n_extreme_high} ({n_extreme_high/len(price):.2%})")
print(f"Extreme-low count  (price < lower bound): {n_extreme_low} ({n_extreme_low/len(price):.2%})")


### Output Interpretation

IQR yöntemine göre, training verisindeki fiyatların **%5.25'i** ("yüksek uç") bu kurala göre
"outlier" sayılıyor; alt sınır negatif olduğu için (fiyat her zaman pozitif olduğundan) hiçbir
"düşük uç" outlier yok. Bu kadar yüksek bir oranın "outlier" sayılması, **IQR kuralının çok sağa
çarpık dağılımlar için mekanik olarak uygulanmasının sorunlu olabileceğini** gösterir — bu uç
değerler gerçek, geçerli, yüksek fiyatlı satışlardır; otomatik olarak **silinmeyecektir**.

### Hands-On Implementation — Log-Price Distribution & Q-Q Plot


In [ ]:
import numpy as np
from statsmodels.graphics.gofplots import qqplot

log_price = np.log(price)
print(f"log(price) skewness: {log_price.skew():.3f}  (vs raw price: {price.skew():.3f})")
print(f"log(price) kurtosis: {log_price.kurtosis():.3f}  (vs raw price: {price.kurtosis():.3f})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.histplot(log_price, bins=50, color=BAR_COLOR, kde=True, ax=axes[0])
axes[0].set_title("log(Price) — Histogram + KDE")
axes[0].set_xlabel("log(Price)")
style_axes(axes[0])

qqplot(log_price, line="s", ax=axes[1], markerfacecolor=BAR_COLOR, markeredgecolor=BAR_COLOR, alpha=0.4)
axes[1].set_title("log(Price) — Q-Q Plot vs Normal")
style_axes(axes[1])

plt.tight_layout()
plt.show()


### Output Interpretation

Log transformation, skewness'i **4.26'dan 0.43'e**, excess kurtosis'i **39.5'ten 0.71'e**
düşürüyor — dağılım şekli normale **çok daha yakın** hale geliyor. Q-Q plot'ta noktalar, orta
kısımda referans çizgisine (45°) oldukça yakın, ancak kuyruklarda hafif sapmalar var (tam normal
değil, ama ham fiyata kıyasla çok daha iyi).

### Critical Engineering Notes

**Target Normality vs Residual Normality:** Burada `price`'ın kendisinin dağılımını inceliyoruz
— bu, OLS'in bir **ön koşulu değildir**. OLS'in klasik varsayımı **residual'ların (hata
terimlerinin)** normal dağılması gerektiğidir, **target'ın değil**. Bu ayrım Section 23'te
ayrıntılı olarak ele alınacaktır. Log-price'ın burada daha normal görünmesi, Session 6'da
deneyeceğimiz **log-target OLS specification**'ı için motive edici bir ön bulgudur, ama henüz
bir OLS modeli eğitmedik — bu yalnızca target'ın kendi dağılımına dair bir gözlemdir.

### Student Exercise

`price.quantile(0.99)` değerinin üzerindeki satışları (`train_df[train_df["price"] >
price.quantile(0.99)]`) inceleyin. Bu satışların `sqft_living`, `grade` ve `zipcode`
dağılımlarında, dataset'in genelinden farklı bir örüntü var mı?


In [ ]:
# Student Exercise:
# TODO: p99 = price.quantile(0.99) hesaplayın.
# TODO: train_df[train_df["price"] > p99][["sqft_living", "grade", "zipcode"]].describe(include="all") ile
#       bu en yüksek %1'lik dilimi inceleyin.

# p99 = price.quantile(0.99)
# print(train_df[train_df["price"] > p99][["sqft_living", "grade", "zipcode"]].describe(include="all"))


### Checkpoint

- [x] `price`'ın merkezi eğilim, yayılım, skewness ve kurtosis değerleri hesaplandı.
- [x] Histogram, KDE, boxplot ile görsel olarak doğrulandı.
- [x] Log transformation'ın dağılım şeklini normale yaklaştırdığı gösterildi (henüz bir OLS
  modeli eğitilmedi — bu yalnızca bir EDA bulgusudur).
- [x] IQR tabanlı "outlier" oranının (%5.25) çarpık dağılımlarda mekanik olarak güvenilmez
  olabileceği not edildi.


---
## Section 07 — Numerical Feature Analysis

### Learning Objective

Tüm uygun sayısal feature'lar için standart bir dağılım analizi (count, missing, unique, mean,
median, variance, std, min/max, quantiles, IQR, skewness, kurtosis) üretmek; bunu **tekrar
kullanılabilir fonksiyonlarla** yapmak.

### Engineering Motivation

13 sayısal feature için aynı analizi 13 kez elle tekrarlamak hem hataya açıktır hem de kod
tekrarını artırır (LAB04'teki "preprocessing için tekrar kullanılabilir fonksiyonlar" prensibiyle
aynı mantık). Bu nedenle iki küçük, tek-sorumluluklu fonksiyon tanımlıyoruz: biri özet
istatistikleri hesaplar, diğeri standart bir üçlü grafik (histogram + KDE, boxplot) çizer.

### Hands-On Implementation — Reusable Functions


In [ ]:
def summarize_numeric_feature(series: pd.Series) -> dict:
    # Compute the standard univariate numeric summary used throughout this capstone.
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = int(((series < lower_bound) | (series > upper_bound)).sum())
    return {
        "count": series.count(),
        "missing": int(series.isna().sum()),
        "unique": series.nunique(),
        "mean": series.mean(),
        "median": series.median(),
        "variance": series.var(),
        "std": series.std(),
        "min": series.min(),
        "max": series.max(),
        "IQR": iqr,
        "skewness": series.skew(),
        "kurtosis": series.kurtosis(),
        "n_outliers_iqr": n_outliers,
        "pct_outliers_iqr": n_outliers / len(series) * 100,
    }


In [ ]:
def plot_numeric_feature(series: pd.Series, feature_name: str, axes) -> None:
    # Draw a histogram+KDE and a boxplot for one numeric feature on the given pair of axes.
    ax_hist, ax_box = axes
    sns.histplot(series, bins=40, color=BAR_COLOR, kde=True, ax=ax_hist)
    ax_hist.set_title(f"{feature_name} — Histogram + KDE")
    style_axes(ax_hist)

    ax_box.boxplot(series, orientation="horizontal", patch_artist=True, boxprops={"facecolor": BAR_COLOR, "alpha": 0.6})
    ax_box.set_title(f"{feature_name} — Boxplot")
    ax_box.set_yticks([])
    style_axes(ax_box)


### Hands-On Implementation — Summary Table for All Numerical Features


In [ ]:
numerical_features_raw = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "sqft_above", "sqft_basement", "yr_built", "yr_renovated",
    "sqft_living15", "sqft_lot15", "lat", "long",
]

numeric_summary = pd.DataFrame(
    {col: summarize_numeric_feature(train_df[col]) for col in numerical_features_raw}
).T.round(3)
numeric_summary


### Output Interpretation

Bu tabloda dikkat çeken bulgular:

- **`sqft_lot`** (skewness ≈ 10.80, kurtosis ≈ 180.5) ve **`sqft_lot15`** (skewness ≈ 9.15),
  dataset'teki **en çarpık** sayısal feature'lardır — IQR kuralına göre sırasıyla **%11.4** ve
  **%10.4**'ü "outlier".
- **`yr_renovated`** (skewness ≈ 4.37, %4.53 outlier): Bu yüksek skewness, Section 03'te zaten
  belirlediğimiz **structural zero** örüntüsünden kaynaklanıyor (satırların %95.8'i tam olarak
  0) — IQR kuralı burada "0 olmayan her değeri" outlier gibi işaretliyor, bu **yanıltıcıdır**.
  `yr_renovated`'a klasik outlier mantığını uygulamak anlamlı değildir; bu feature Section 15'te
  `is_renovated_at_sale` (binary) ve `years_since_renovation` gibi türetilmiş feature'lara
  dönüştürülecektir.
- **`floors`** ve **`yr_built`**: Düşük skewness, 0 IQR-outlier — nispeten düzenli dağılımlar.
- **`lat`**: Hemen hemen simetrik (skewness ≈ -0.52), ihmal edilebilir outlier oranı (%0.02).

### Hands-On Implementation — Distribution Plots (Grid)

Tüm 13 feature için ayrı ayrı büyük grafikler yerine, okunabilirliği korumak amacıyla
**kompakt bir grid** kullanıyoruz.


In [ ]:
fig, axes = plt.subplots(len(numerical_features_raw), 2, figsize=(11, 3 * len(numerical_features_raw)))

for row_idx, feature in enumerate(numerical_features_raw):
    plot_numeric_feature(train_df[feature], feature, (axes[row_idx, 0], axes[row_idx, 1]))

plt.tight_layout()
plt.show()


### Output Interpretation

Grid'deki görseller, yukarıdaki sayısal tabloyu doğruluyor: `sqft_lot`/`sqft_lot15` için
histogram neredeyse tek bir çubukta toplanmış görünüyor (çünkü çok az sayıda aşırı büyük değer,
x eksenini domine ediyor) — bu, Section 16'daki **log transformation**'ın motivasyonudur.
`bedrooms`'ın boxplot'unda, Section 03'te bulduğumuz `33` değeri tek bir uç nokta olarak
görülebilir.

### Hands-On Implementation — Outlier Screening Summary


In [ ]:
outlier_screening = numeric_summary[["skewness", "kurtosis", "n_outliers_iqr", "pct_outliers_iqr"]].sort_values(
    "pct_outliers_iqr", ascending=False
)
outlier_screening


### Critical Engineering Notes

Bu section'da hesaplanan IQR-tabanlı outlier sayıları **hiçbir kaydı silmek için
kullanılmamaktadır** — yalnızca bir **screening (tarama)** aracıdır. Section 01'deki kurala
uygun olarak, gerçek outlier/transformation kararları Session 4'te (Section 16, 19) **training
verisinden öğrenilen** eşiklerle, gerekçeli biçimde verilecektir.

### Student Exercise

**Session 2 Exercise:** `summarize_numeric_feature()` fonksiyonunu `sqft_above` feature'ına
uygulayıp, `sqft_living` ile karşılaştırın. Bu iki feature'ın birbirine çok benzer dağılımlara
sahip olup olmadığını (bu, Section 18'deki multicollinearity tartışmasının bir ön işaretidir)
yorumlayın.


In [ ]:
# Student Exercise:
# TODO: summarize_numeric_feature(train_df["sqft_above"]) ve summarize_numeric_feature(train_df["sqft_living"])
#       sonuçlarını yan yana bir DataFrame'de karşılaştırın.
# TODO: Bu iki feature'ın skewness/kurtosis/IQR değerlerinin ne kadar benzer olduğunu not edin.

# comparison = pd.DataFrame({
#     "sqft_above": summarize_numeric_feature(train_df["sqft_above"]),
#     "sqft_living": summarize_numeric_feature(train_df["sqft_living"]),
# })
# print(comparison)


### Checkpoint

- [x] İki tekrar kullanılabilir fonksiyon (`summarize_numeric_feature`, `plot_numeric_feature`)
  tanımlandı ve ne yaptıkları açıklandı.
- [x] 13 sayısal feature için özet istatistik tablosu ve dağılım grafikleri üretildi.
- [x] `sqft_lot`/`sqft_lot15` (aşırı çarpık) ve `yr_renovated` (structural-zero kaynaklı yanıltıcı
  outlier oranı) özellikle not edildi.


---
## Section 08 — Categorical & Ordinal Analysis

### Learning Objective

Sayısal olarak depolanan ama kavramsal olarak kategorik/ordinal olan feature'ları (`waterfront`,
`view`, `condition`, `grade`, `zipcode`) doğru biçimde analiz etmek; cardinality ve nadir
kategorileri tespit etmek.

### Concept Explanation

- **Binary Feature**: Yalnızca iki değer alan feature (`waterfront`: 0/1).
- **Ordinal Feature**: Değerleri arasında doğal bir **sıralama** olan, ama aralarındaki
  "uzaklığın" eşit olması gerekmeyen feature (`view`, `condition`, `grade`).
- **Cardinality**: Bir kategorik feature'ın kaç farklı değer aldığı (`zipcode`: yüksek
  cardinality — 70 farklı değer).
- **Rare Category**: Dataset'te çok az örnekte görülen bir kategori değeri — model eğitiminde
  (özellikle One-Hot Encoding ile) instabiliteye yol açabilir.

### Hands-On Implementation — Binary Feature: `waterfront`


In [ ]:
waterfront_counts = train_df["waterfront"].value_counts().sort_index()
waterfront_pct = (train_df["waterfront"].value_counts(normalize=True).sort_index() * 100).round(2)
print(pd.DataFrame({"count": waterfront_counts, "pct": waterfront_pct}))


### Output Interpretation

`waterfront=1` (deniz kenarı), training verisinin yalnızca **%0.79**'unu oluşturuyor — bu, çok
nadir bir kategori. Bu durum Section 17'de (Categorical Encoding) ve Section 25'te (Influence
Diagnostics) akılda tutulmalıdır: bu kadar az örnekli bir kategorinin katsayısı, birkaç aşırı
değerli gözlem tarafından orantısız şekilde etkilenebilir.

### Hands-On Implementation — Ordinal Features: `view`, `condition`, `grade`


In [ ]:
def ordinal_countplot(series: pd.Series, feature_name: str, ax) -> None:
    # Draw a count bar chart with categories in their natural (already-ordered) order.
    counts = series.value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color=BAR_COLOR)
    ax.set_title(f"{feature_name} — Ordered Count Distribution")
    ax.set_xlabel(feature_name)
    ax.set_ylabel("Count")
    style_axes(ax)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, feature in zip(axes, ["view", "condition", "grade"]):
    ordinal_countplot(train_df[feature], feature, ax)
plt.tight_layout()
plt.show()


In [ ]:
for feature in ["view", "condition", "grade"]:
    print(f"=== {feature} ===")
    counts = train_df[feature].value_counts().sort_index()
    pct = (train_df[feature].value_counts(normalize=True).sort_index() * 100).round(2)
    print(pd.DataFrame({"count": counts, "pct": pct}))
    print()


### Output Interpretation

- **`view`**: Gözlemlerin **%90.2**'si `view=0` (manzara puanı yok) — kalan değerler (1-4)
  nadir ve kendi aralarında da dengesiz dağılmış.
- **`condition`**: **%64.1**'i `condition=3` (orta/standart) — `condition=1` yalnızca **%0.14**
  (21 gözlem), çok nadir bir kategori.
- **`grade`**: Çoğunluk `grade=7` (%41.1) ve `grade=8` (%28.4) etrafında kümelenmiş; `grade=1`
  ve `grade=3` yalnızca **1'er gözlem** — bunlar aşırı nadir kategorilerdir.

### Hands-On Implementation — High-Cardinality Categorical: `zipcode`


In [ ]:
zipcode_counts = train_df["zipcode"].value_counts()
print(f"zipcode unique value count: {train_df['zipcode'].nunique()}")
print()
print("Top 5 most frequent zipcodes:")
print(zipcode_counts.head(5))
print()
print("5 rarest zipcodes (still present in training data):")
print(zipcode_counts.tail(5))


### Output Interpretation

`zipcode`, **70 farklı değer** alan, nominal (sıralaması olmayan) bir konum tanımlayıcısıdır. En
sık görülen zipcode (`98103`) training verisinde 436 kez, en nadir görülen (`98039`) ise 37 kez
geçiyor — hiçbir zipcode training verisinde **sıfır** gözlemle temsil edilmiyor, ama bu dağılım
oldukça dengesiz. Bu, Section 17'de **One-Hot Encoding ile `zipcode`'u kullanmanın 69 ek dummy
değişken** anlamına geleceğini (yüksek boyutluluk) ve Section 18'de ciddi bir
multicollinearity/design-matrix-boyutu riskini işaret ediyor.

### Critical Engineering Notes

Nadir kategoriler (`condition=1`, `grade=1`/`3`, hatta `waterfront=1`) kendi başına bir hata
değildir, ancak şu riskleri taşır:

1. **Train/validation/test arasında bir kategorinin hiç görülmemesi** (Section 17'de
   `handle_unknown` stratejisiyle ele alınacak).
2. **Katsayı instabilitesi** — çok az gözlemli bir kategori için tahmin edilen katsayının standart
   hatası büyük olur (Section 21'de bu konuya döneceğiz).

### Student Exercise

**Session 2 Exercise (devam):** `train_df["grade"].value_counts()` sonucuna bakarak, `grade <= 4`
olan kategorileri **"nadir"** olarak işaretleyin (örn. toplam 20 gözlemden az). Bu kategorilerin
birleştirilmesi (örn. tek bir "grade_low" kategorisi) gerekip gerekmediğini, kendi gerekçenizle
tartışın — bu kararı burada **uygulamıyoruz**, yalnızca tartışıyoruz; nihai karar Section 17'de
verilecektir.


In [ ]:
# Student Exercise:
# TODO: train_df["grade"].value_counts().sort_index() çalıştırıp, 20'den az gözlemli grade
#       değerlerini listeleyin.
# TODO: Bu kategorileri birleştirmenin (örn. "grade_le_5") avantaj/dezavantajını 2-3 cümlede tartışın.

# rare_grades = train_df["grade"].value_counts()[train_df["grade"].value_counts() < 20]
# print(rare_grades)


### Checkpoint

- [x] Binary feature (`waterfront`) analiz edildi — nadir kategori (%0.79) not edildi.
- [x] Ordinal feature'lar (`view`, `condition`, `grade`) sıralı countplot'larla incelendi.
- [x] Yüksek cardinality'li nominal feature (`zipcode`, 70 kategori) analiz edildi.
- [x] Nadir kategorilerin encoding/katsayı instabilitesi riskleri not edildi (Section 17/21'e
  aktarıldı).


---
## Section 09 — EDA Workshop

### Learning Objective

Section 06-08'de öğrendiğiniz araçları (özet istatistikler, histogram/KDE/boxplot, ordinal
countplot) **bağımsız olarak** bir feature'a uygulayarak, kendi başınıza bir statistical
distribution report hazırlamak.

### Workshop Instructions

Aşağıdaki feature'lardan **birini** seçin (daha önce detaylı incelemediğimiz bir tanesini
seçmeniz önerilir): `sqft_above`, `bathrooms`, `floors`, `yr_built`, `condition`.

Seçtiğiniz feature için aşağıdaki **Statistical Distribution Report** şablonunu doldurun:

1. **Tip:** Sayısal mı, ordinal mi, binary mi?
2. **Özet istatistikler:** `summarize_numeric_feature()` (sayısal ise) veya
   `value_counts()` (kategorik/ordinal ise).
3. **Görsel:** `plot_numeric_feature()` veya `ordinal_countplot()` ile uygun grafiği üretin.
4. **Yorum:** Dağılım şekli (simetrik/çarpık), outlier oranı, dikkat çeken herhangi bir örüntü.
5. **Sonraki adıma not:** Bu feature'ın Session 3'te (feature-target ilişkisi) veya Session 4'te
  (feature engineering) nasıl ele alınabileceğine dair bir öngörü.

### Hands-On Implementation — Workshop Scaffold


In [ ]:
# Workshop Exercise:
# TODO: Aşağıdaki "selected_feature" değişkenini kendi seçtiğiniz feature ile değiştirin.
# TODO: Feature sayısal ise summarize_numeric_feature + plot_numeric_feature kullanın.
# TODO: Feature ordinal/binary ise value_counts + ordinal_countplot kullanın.
# TODO: Yukarıdaki 5 maddelik şablonu bir Markdown hücresinde (bu hücrenin altına ekleyeceğiniz
#       yeni bir hücrede) doldurun.

selected_feature = "sqft_above"  # <- kendi seçiminizle değiştirin

if pd.api.types.is_numeric_dtype(train_df[selected_feature]) and train_df[selected_feature].nunique() > 10:
    print(summarize_numeric_feature(train_df[selected_feature]))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    plot_numeric_feature(train_df[selected_feature], selected_feature, axes)
    plt.tight_layout()
    plt.show()
else:
    print(train_df[selected_feature].value_counts().sort_index())
    fig, ax = plt.subplots(figsize=(6, 4))
    ordinal_countplot(train_df[selected_feature], selected_feature, ax)
    plt.show()


### Critical Engineering Notes

Bu hücre, hangi feature'ı seçerseniz seçin **çalışacak şekilde** (sayısal/ordinal ayrımını
otomatik tespit ederek) tasarlanmıştır — bu nedenle bu Workshop hücresi, varsayılan
`selected_feature = "sqft_above"` değeriyle notebook'un ana akışını **bozmadan** çalışır. Kendi
seçiminizle değiştirip yeniden çalıştırmanız beklenir.

### Checkpoint

- [x] Workshop scaffold'u çalıştı (varsayılan `sqft_above` ile), herhangi bir feature adıyla
  değiştirilebilir biçimde doğrulandı.
- [ ] Katılımcı kendi seçtiği feature için 5 maddelik raporu tamamlamalıdır (bu, bir sonraki
  derste tartışılacaktır).

---

## Session 2 Özeti

Bu oturumda, `train_df` üzerinde:

- `price`'ın dağılımını (çok sağa çarpık, skewness ≈4.26) detaylı inceledik ve log
  transformation'ın bu çarpıklığı önemli ölçüde azalttığını (skewness ≈0.43) gösterdik — ama
  henüz hiçbir model eğitmedik.
- 13 sayısal feature için tekrar kullanılabilir fonksiyonlarla özet istatistik ve dağılım
  grafikleri ürettik; `sqft_lot`/`sqft_lot15` (aşırı çarpık) ve `yr_renovated` (yanıltıcı IQR
  outlier oranı) özellikle not edildi.
- Binary/ordinal/yüksek-cardinality kategorik feature'ları (`waterfront`, `view`, `condition`,
  `grade`, `zipcode`) inceledik ve nadir kategorilerin encoding riskini belirledik.
- Bağımsız bir EDA Workshop'uyla bu araçları kendi başınıza uygulama fırsatı sundu.

Session 3'te bu univariate bulguları, **feature-target** ve **feature-feature** ilişkilerine
genişleteceğiz.


---
# SESSION 3 — Bivariate & Multivariate EDA
**Toplam Süre: 150 dakika (140 dakika aktif + 10 dakika ara)**

Bu oturumda da Session 2'deki kural geçerlidir: tüm analizler **yalnızca `train_df` üzerinde**
yapılmaktadır.


---
## Section 10 — Feature–Target Scatter Analysis

### Learning Objective

Seçilmiş feature'ların `price` ile ilişkisini scatter plot'larla görsel olarak incelemek;
overplotting, doğrusal trend ve LOWESS (yerel ağırlıklı regresyon) eğrileri arasındaki farkı
anlamak.

### Concept Explanation

- **Overplotting**: Çok sayıda nokta üst üste bindiğinde (bu dataset'te 15,100 nokta), ham
  scatter plot yanıltıcı olabilir — **transparency (alpha)** kullanımı bunu hafifletir.
- **Linear Trendline**: Basit bir doğrusal (OLS) uyum çizgisi — ilişkinin **ortalama doğrusal
  yönünü** gösterir, ama nonlinear örüntüleri gizleyebilir.
- **LOWESS (Locally Weighted Scatterplot Smoothing)**: Her noktada yerel bir ağırlıklı regresyon
  uygulayarak **nonparametrik, esnek** bir trend eğrisi üretir — doğrusal olmayan örüntüleri
  ortaya çıkarabilir.

### Hands-On Implementation — `sqft_living` vs `price`


In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess


def scatter_with_trend(x: pd.Series, y: pd.Series, feature_name: str, ax, lowess_frac: float = 0.3) -> None:
    ax.scatter(x, y, color=BAR_COLOR, alpha=0.15, s=12, linewidths=0)

    coeffs = np.polyfit(x, y, deg=1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, np.polyval(coeffs, x_line), color=WARNING_COLOR, linewidth=2, label="Linear trend")

    lowess_result = lowess(y, x, frac=lowess_frac)
    ax.plot(lowess_result[:, 0], lowess_result[:, 1], color=HIGHLIGHT_COLOR, linewidth=2, label="LOWESS")

    ax.set_title(f"{feature_name} vs price")
    ax.set_xlabel(feature_name)
    ax.set_ylabel("price")
    ax.legend(frameon=False, fontsize=8)
    style_axes(ax)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scatter_with_trend(train_df["sqft_living"], train_df["price"], "sqft_living", ax)
plt.show()


### Output Interpretation

Linear trend ve LOWESS eğrileri büyük ölçüde örtüşüyor — `sqft_living`/`price` ilişkisi, orta
aralıkta **büyük ölçüde doğrusal**. Ancak çok yüksek `sqft_living` değerlerinde (>8000) LOWESS
eğrisi doğrusal çizgiden hafifçe ayrılıyor ve nokta sayısı seyrekleşiyor — bu bölgede tahminler
daha az güvenilir olacaktır.

### Hands-On Implementation — `grade` vs `price` (Discrete Ordinal Feature)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
jittered_grade = train_df["grade"] + np.random.RandomState(RANDOM_STATE).uniform(-0.15, 0.15, size=len(train_df))
scatter_with_trend(jittered_grade, train_df["price"], "grade (jittered)", ax, lowess_frac=0.4)
plt.show()


### Output Interpretation

`grade`, ayrık bir ordinal değişken olduğu için noktalar dikey şeritler halinde kümeleniyor;
görselleştirmeyi kolaylaştırmak için küçük bir rastgele **jitter** eklendi (yalnızca
görselleştirme amaçlı — orijinal `grade` değerleri değiştirilmedi). LOWESS eğrisi, `grade`
arttıkça `price`'ın **giderek hızlanan (konveks)** bir artış gösterdiğine işaret ediyor —
doğrusal trendden farklı olarak, yüksek grade'lerde fiyat artışı daha dik. Bu, Section 16'da
**quadratic/polynomial** bir `grade` teriminin değerlendirilmesini motive eden bir bulgudur.

### Hands-On Implementation — `yr_built` vs `price`


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scatter_with_trend(train_df["yr_built"], train_df["price"], "yr_built", ax, lowess_frac=0.3)
plt.show()


### Output Interpretation

`yr_built` ile `price` arasındaki ilişki **belirgin biçimde doğrusal değildir**: LOWESS eğrisi
U-benzeri bir örüntü gösteriyor — çok eski (1900'ler civarı) ve çok yeni (2010'lar) konutlar,
orta yaşlı (1950-1980) konutlara kıyasla daha yüksek ortalama fiyata sahip. Bu, Section 15'te
türeteceğimiz `house_age_at_sale` feature'ı için **doğrusal olmayan bir etki** (örn. quadratic
terim) ihtiyacını işaret ediyor — bu konuya Section 16'da döneceğiz.

### Critical Engineering Notes — Outlier Effects

Scatter plot'larda `bedrooms=33` gibi Section 03'te işaretlediğimiz uç noktalar, **tek bir nokta
olarak** diğer binlerce noktanın arasında kayboluyor — scatter plot'lar bu tür tekil anomalileri
tespit etmek için **ideal bir araç değildir** (boxplot bu amaçla daha uygundur, Section 07).
Ancak LOWESS gibi yerel yöntemler, az sayıda uç nokta tarafından **orantısız şekilde** etkilenen
bölgelerde (örn. veri seyrekleştiğinde) daha hassastır.

### Student Exercise

`bathrooms` vs `price` scatter'ını `scatter_with_trend()` fonksiyonuyla üretin. Linear trend ve
LOWESS eğrileri arasında `sqft_living`'e kıyasla daha fazla mı yoksa daha az mı ayrışma
görüyorsunuz? Yorumunuzu yazın.


In [ ]:
# Student Exercise:
# TODO: scatter_with_trend(train_df["bathrooms"], train_df["price"], "bathrooms", ax) çağırın.
# TODO: Linear ve LOWESS eğrileri arasındaki ayrışmayı sqft_living ile karşılaştırın.

# fig, ax = plt.subplots(figsize=(8, 5))
# scatter_with_trend(train_df["bathrooms"], train_df["price"], "bathrooms", ax)
# plt.show()


### Checkpoint

- [x] Tekrar kullanılabilir bir `scatter_with_trend()` fonksiyonu tanımlandı (overplotting için
  alpha, linear trend, LOWESS).
- [x] `sqft_living` (büyük ölçüde doğrusal), `grade` (konveks nonlinear), `yr_built` (U-şekilli
  nonlinear) ile `price` ilişkileri karşılaştırıldı.
- [x] Nonlinear bulgular Section 15/16'ya (feature engineering/transformation) yönlendirildi.


---
## Section 11 — Correlation Analysis

### Learning Objective

Pearson ve Spearman korelasyonları hesaplamak, bir correlation heatmap oluşturmak ve
**korelasyonun sınırlamalarını** (özellikle nominal bir değişkeni sayısal gibi ele almanın
riskini) somut bir örnekle göstermek.

### Statistical Foundations

Pearson korelasyonu, iki değişken arasındaki **doğrusal** ilişkinin gücünü ölçer:

$$r_{xy}=\frac{\sum_i (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_i (x_i-\bar{x})^2}\sqrt{\sum_i (y_i-\bar{y})^2}}$$

Spearman korelasyonu, değerler yerine **sıralamalar (rank)** üzerinden hesaplanır — bu nedenle
monoton ama doğrusal olmayan ilişkileri de yakalayabilir.

### Hands-On Implementation — Feature–Target Correlations


In [ ]:
numeric_cols_for_corr = [
    "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "grade", "sqft_above", "sqft_basement",
    "yr_built", "yr_renovated", "zipcode", "lat", "long", "sqft_living15", "sqft_lot15",
]

pearson_with_target = train_df[numeric_cols_for_corr].corr(method="pearson")["price"].drop("price").sort_values(ascending=False)
spearman_with_target = train_df[numeric_cols_for_corr].corr(method="spearman")["price"].drop("price").sort_values(ascending=False)

correlation_comparison = pd.DataFrame({"Pearson": pearson_with_target, "Spearman": spearman_with_target})
correlation_comparison


### Output Interpretation

- **`sqft_living`** (Pearson ≈0.712) ve **`grade`** (Pearson ≈0.671) `price` ile en güçlü
  doğrusal ilişkiye sahip feature'lar.
- **`lat`**'ın Spearman korelasyonu (≈0.445), Pearson korelasyonundan (≈0.298) belirgin biçimde
  yüksek — bu, `lat` ile `price` arasındaki ilişkinin **monoton ama tam doğrusal olmadığını**
  gösteriyor (Section 10'daki `yr_built` örneğine benzer bir sinyal).
- **`condition`**'ın hem Pearson (≈0.039) hem Spearman (≈0.027) korelasyonu **neredeyse sıfır**
  — bu şaşırtıcı görünebilir ("durum iyiyse fiyat yüksek olmalı" beklentisine aykırı). Bu bulguyu
  Section 13'te (Conditional Relationships) **confounding** açısından inceleyeceğiz.
- **`zipcode`**'un korelasyonu (Pearson ≈ -0.056) da neredeyse sıfır — ama bu **yanıltıcıdır**,
  aşağıda ayrıca ele alınıyor.

### Hands-On Implementation — Feature–Feature Correlation Heatmap


In [ ]:
corr_matrix = train_df[numeric_cols_for_corr].corr(method="pearson")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False, square=True, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Feature-Feature Pearson Correlation Heatmap (training data)")
plt.tight_layout()
plt.show()


In [ ]:
high_corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=abs, ascending=False)
)
print("Top 10 highest-magnitude feature-feature correlations:")
print(high_corr_pairs.head(10))


### Output Interpretation

En yüksek feature-feature korelasyonları arasında **`sqft_living`-`sqft_above`** (≈0.876),
**`sqft_living`-`grade`** (≈0.763), **`sqft_living`-`bathrooms`** (≈0.753) ve
**`sqft_living`-`sqft_living15`** (≈0.758) bulunuyor. Bu yüksek korelasyonlar, Section 18'de ele
alacağımız **multicollinearity** probleminin ilk somut işaretleridir — bunlardan biri
(`sqft_living`-`sqft_above`) hatta **structural/perfect** bir multicollinearity'nin
(`sqft_above + sqft_basement = sqft_living`) bir sonucudur.

### Hands-On Implementation — Correlation Limitations: The `zipcode` Trap


In [ ]:
zip_median_price = train_df.groupby("zipcode")["price"].median().sort_values(ascending=False)

print(f"Raw numeric correlation(zipcode, price): {train_df['zipcode'].corr(train_df['price']):.4f}")
print()
print(f"Highest-median zipcode: {zip_median_price.index[0]} -> median price {zip_median_price.iloc[0]:,.0f}")
print(f"Lowest-median zipcode : {zip_median_price.index[-1]} -> median price {zip_median_price.iloc[-1]:,.0f}")
print(f"Ratio (highest/lowest): {zip_median_price.iloc[0] / zip_median_price.iloc[-1]:.2f}x")


### Critical Engineering Notes — Correlation Limitations

`zipcode`'un `price` ile ham sayısal korelasyonu **≈ -0.056** — pratikte **sıfıra çok yakın**. Bu,
"konum fiyatı etkilemiyor" anlamına **gelmez**. `zipcode`, posta kodlarının **nominal (sıralaması
olmayan) bir kimliğidir** — `98004`'ün `98001`'den "sayısal olarak büyük" olması, konut
fiyatlarıyla hiçbir mantıksal ilişki taşımaz; bu rastgele bir kodlama kararıdır. Gerçekte, en
yüksek medyan fiyatlı zipcode'un medyanı (**2,100,000 USD**, zipcode `98039`), en düşük medyanlı
zipcode'dan (**233,250 USD**, zipcode `98168`) **≈9 kat** daha yüksek — konum, fiyat üzerinde **çok
güçlü** bir etkiye sahip, ama bu etki **Pearson/Spearman korelasyonuyla doğru biçimde
ölçülemez**. Bu, **Correlation vs Causation** değil, daha temel bir **"yanlış ölçek/tip ile
korelasyon hesaplama"** hatasıdır. `zipcode`'u Section 17'de **kategorik** (One-Hot Encoded)
olarak ele alacağız — sayısal bir feature olarak değil.

### Student Exercise

`waterfront` (binary, 0/1) ile `price` arasındaki Pearson korelasyonunu (≈0.276) `view`
(ordinal, 0-4) ile karşılaştırın. Bu iki korelasyon katsayısının **doğrudan karşılaştırılabilir
olup olmadığını** (ikisi de farklı ölçeklerde değişkenler) tartışın.


In [ ]:
# Student Exercise:
# TODO: train_df["waterfront"].corr(train_df["price"]) ve train_df["view"].corr(train_df["price"])
#       değerlerini yazdırın.
# TODO: Binary bir feature ile ordinal bir feature'ın korelasyon katsayılarının yorumlanma
#       farkını 2-3 cümlede tartışın.

# print(train_df["waterfront"].corr(train_df["price"]))
# print(train_df["view"].corr(train_df["price"]))


### Checkpoint

- [x] Pearson ve Spearman korelasyonları hesaplandı ve karşılaştırıldı (`lat` için belirgin
  fark gözlendi).
- [x] Feature-feature correlation heatmap üretildi; en yüksek 10 çift raporlandı (Section 18
  için not edildi).
- [x] `zipcode` örneğiyle, nominal bir değişkeni sayısal korelasyonla ölçmenin **yanıltıcı
  olabileceği** somut sayılarla (9x fiyat farkı, ama ≈0 korelasyon) gösterildi.


---
## Section 12 — Multivariate & Spatial Analysis

### Learning Objective

İkiden fazla değişkeni aynı anda görselleştirmek (pairplot, faceted scatter, spatial scatter) ve
`zipcode`/`lat`/`long` üzerinden konum etkisini keşfetmek.

### Hands-On Implementation — Pairplot (Seçili Feature'lar)

Tüm 18 sayısal feature'ı bir pairplot'a sığdırmak okunaksız olurdu (`18x18 = 324` panel);
bunun yerine Section 11'de en yüksek korelasyona sahip olduğunu gördüğümüz **4 feature**'ı
seçiyoruz.


In [ ]:
pairplot_features = ["price", "sqft_living", "grade", "bathrooms"]
pairplot_sample = train_df[pairplot_features].sample(n=2000, random_state=RANDOM_STATE)

sns.pairplot(pairplot_sample, plot_kws={"color": BAR_COLOR, "alpha": 0.3, "s": 15})
plt.suptitle("Pairplot — price, sqft_living, grade, bathrooms (2000-row sample)", y=1.02)
plt.show()


### Output Interpretation

Okunabilirlik için **2000 satırlık rastgele bir örneklem** kullandık (tüm 15,100 satırı
çizdirmek hem çok yavaş hem de aşırı overplotting yaratırdı — bu, "tüm scatter kombinasyonlarını
körlemesine üretme" kuralının bir uygulamasıdır). Pairplot, `sqft_living`-`grade`,
`sqft_living`-`bathrooms` ve `grade`-`bathrooms` arasında da pozitif ilişkiler olduğunu
doğruluyor — bu üç feature'ın **birbirleriyle de** ilişkili olduğu, yalnızca `price` ile değil
(Section 18'e not).

### Hands-On Implementation — Spatial Analysis (`lat`/`long`)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    train_df["long"], train_df["lat"], c=train_df["price"], cmap="viridis",
    s=8, alpha=0.5, vmax=train_df["price"].quantile(0.95),
)
plt.colorbar(scatter, ax=ax, label="price (capped at 95th percentile for color scale)")
ax.set_title("Spatial Distribution of Price (King County, training data)")
ax.set_xlabel("long")
ax.set_ylabel("lat")
plt.show()


### Output Interpretation

Renk skalası, aşırı yüksek fiyatlı birkaç satışın tüm haritayı "boğmaması" için **95. yüzdelik
dilimde sınırlandırıldı** — aksi halde çoğu nokta tek bir renkte görünür, konumsal örüntü
kaybolurdu. Haritada belirli bölgelerde (özellikle göl kıyısı bölgeleri — Section 11'de en
yüksek medyan fiyatlı zipcode olan `98039` bu bölgede) **koyu/parlak renkli kümeler** görülüyor —
bu, `lat`/`long`'un (ve dolaylı olarak `zipcode`'un) fiyatla güçlü bir **mekansal (spatial)**
ilişkisi olduğunu doğruluyor.

### Hands-On Implementation — Price by Zipcode (Group-Based Distribution)


In [ ]:
top_bottom_zipcodes = (
    list(train_df.groupby("zipcode")["price"].median().sort_values(ascending=False).head(5).index)
    + list(train_df.groupby("zipcode")["price"].median().sort_values(ascending=False).tail(5).index)
)
subset = train_df[train_df["zipcode"].isin(top_bottom_zipcodes)].copy()
subset["zipcode"] = subset["zipcode"].astype(str)

order = subset.groupby("zipcode")["price"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=subset, x="zipcode", y="price", order=order, color=BAR_COLOR, ax=ax)
ax.set_title("Price Distribution — Top 5 vs Bottom 5 Zipcodes (by median price)")
ax.tick_params(axis="x", rotation=45)
style_axes(ax)
plt.tight_layout()
plt.show()


### Output Interpretation

En yüksek 5 zipcode'un fiyat dağılımları, en düşük 5 zipcode'unkilerle **neredeyse hiç
örtüşmüyor** — bu, `zipcode`'un (nominal olmasına karşın) fiyatı açıklamada son derece güçlü bir
grup-tabanlı sinyal taşıdığını gösteriyor. Bu bulgu Section 17'de `zipcode`'un **One-Hot
Encoding** ile modele dahil edilmesi kararını doğrudan destekliyor.

### Hands-On Implementation — Group-Based Distribution: Price by Grade Bins


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=train_df, x="grade", y="price", color=BAR_COLOR, ax=ax)
ax.set_title("Price Distribution by Grade")
style_axes(ax)
plt.show()


### Output Interpretation

`grade` arttıkça medyan fiyat **monoton olarak** artıyor ve yayılım (interquartile range) da
büyüyor — bu, Section 23'te ele alacağımız **heteroscedasticity** (hata varyansının feature
değerine bağlı değişmesi) olasılığına dair bir ön işarettir: yüksek grade'li konutların fiyat
tahmini doğal olarak daha büyük bir mutlak hata aralığına sahip olabilir.

### Student Exercise

`train_df`'te `waterfront=1` olan ve olmayan konutların `price` dağılımlarını bir boxplot ile
karşılaştırın. `waterfront`'un fiyat üzerindeki etkisinin, `grade`'in etkisiyle kıyaslandığında
ne kadar büyük olduğunu yorumlayın.


In [ ]:
# Student Exercise:
# TODO: sns.boxplot(data=train_df, x="waterfront", y="price") ile iki grubu karşılaştırın.
# TODO: Medyan farkını grade'in medyan farkıyla (en yüksek vs en düşük grade) kıyaslayın.

# fig, ax = plt.subplots(figsize=(6, 5))
# sns.boxplot(data=train_df, x="waterfront", y="price", ax=ax)
# plt.show()


### Checkpoint

- [x] Pairplot ile `price`/`sqft_living`/`grade`/`bathrooms` arasındaki ikili ilişkiler
  (okunabilir bir örneklemle) görselleştirildi.
- [x] `lat`/`long` üzerinde renk-kodlu bir spatial price haritası üretildi.
- [x] En yüksek/en düşük 5 zipcode arasındaki fiyat dağılımı farkı görsel olarak doğrulandı.
- [x] `grade` arttıkça hem medyan fiyatın hem fiyat yayılımının arttığı not edildi (Section
  23'e, heteroscedasticity tartışmasına referansla).


---
## Section 13 — Conditional Relationships

### Learning Objective

Section 11'de bulduğumuz **"`condition`'ın `price` ile korelasyonu neredeyse sıfır"** bulgusunu,
başka bir feature'ı **kontrol ederek (controlling for)** yeniden incelemek; confounding
kavramını gerçek bir örnekle göstermek.

### Concept Explanation

- **Confounding**: Üçüncü bir değişkenin (burada: `grade`), incelenen iki değişken
  (`condition`, `price`) arasındaki görünen ilişkiyi **gizlediği veya bozduğu** durumdur.
- **Controlled Comparison**: Bir feature'ın etkisini, başka bir feature'ın değerini **sabit
  tutarak (veya bir alt kümesine odaklanarak)** incelemek.
- **Partial Relationship**: Bir feature'ın, diğer feature'ların etkisi "çıkarıldıktan sonra"
  kalan ilişkisi — tam olarak OLS'in multiple regression'da yaptığı şeydir (Section 21'de
  katsayıları bu şekilde yorumlayacağız).

### Hands-On Implementation — Unconditional `condition` vs `price`


In [ ]:
unconditional = train_df.groupby("condition")["price"].agg(["mean", "median", "count"])
unconditional


### Output Interpretation

Dikkat: `condition=3`'ün ortalama fiyatı (≈540,490) `condition=4`'ten (≈519,660) **daha
yüksek** — yani fiyat, `condition` arttıkça **monoton olarak artmıyor**. Bu, Section 11'deki
"neredeyse sıfır korelasyon" bulgusuyla tutarlı, ama şaşırtıcı: sağduyu, daha iyi durumdaki bir
evin daha pahalı olmasını beklerdi.

### Hands-On Implementation — Controlled Comparison (within `grade == 7`)


In [ ]:
grade_7_subset = train_df[train_df["grade"] == 7]
conditional_within_grade7 = grade_7_subset.groupby("condition")["price"].agg(["mean", "median", "count"])
conditional_within_grade7


### Output Interpretation

`grade == 7` (en sık görülen grade) içinde **sabit tutulduğunda**, `condition` arttıkça medyan
fiyat **tutarlı biçimde artıyor**: condition 3 → 4 → 5 için medyan fiyat **350,000 → 387,673 →
453,500**'e yükseliyor. Yani `condition`'ın `price` ile **gerçek bir pozitif ilişkisi var**, ama
bu ilişki `grade`'i kontrol etmeden bakıldığında **gizleniyor**.

### Hands-On Implementation — Identifying the Confound


In [ ]:
confound_check = train_df.groupby("condition")[["grade", "yr_built"]].mean()
confound_check


### Output Interpretation

Confound'un kaynağı burada görülüyor: **`condition=3`** grubunun ortalama `grade`'i (≈7.86),
`condition=4` (≈7.41) ve `condition=5` (≈7.35) gruplarından **daha yüksek**. Yani
`condition=3` evler, ortalamada daha yüksek yapı kalitesine (grade) sahip — bu da onların
fiyatını yukarı çekiyor ve `condition`'ın kendi (düşük) etkisini **maskeliyor**. Bu, klasik bir
**confounding (Simpson's paradox'a benzer bir örüntü)** örneğidir: `grade` kontrol edilmeden
`condition`'ın gerçek etkisi yanlış yorumlanabilir.

### Critical Engineering Notes

Bu bulgu, Section 21'de OLS katsayılarını yorumlarken **neden kritik** olacak: Multiple
Regression, tam olarak burada elle yaptığımız şeyi **matematiksel olarak tüm feature'lar için
aynı anda** yapar — her katsayı, "diğer tüm feature'lar sabit tutulduğunda" o feature'ın
etkisini tahmin eder. Bu nedenle OLS'te `condition`'ın katsayısının, bu section'daki bulgularla
tutarlı biçimde **pozitif ve anlamlı** çıkmasını (ham korelasyonun aksine) **bekliyoruz** — bu,
Section 21'de gerçek model sonucuyla karşılaştırılacak bir **ön tahmindir**, henüz doğrulanmış
bir sonuç değildir.

### Student Exercise

Aynı controlled comparison'ı `grade == 8` alt kümesi için tekrarlayın. Bulgu, `grade == 7` için
bulduğumuzla tutarlı mı?


In [ ]:
# Student Exercise:
# TODO: train_df[train_df["grade"] == 8] alt kümesini oluşturup, "condition" bazında
#       price mean/median/count hesaplayın.
# TODO: Bu sonucu grade==7 bulgusuyla karşılaştırıp tutarlılığını yorumlayın.

# grade_8_subset = train_df[train_df["grade"] == 8]
# print(grade_8_subset.groupby("condition")["price"].agg(["mean", "median", "count"]))


### Checkpoint

- [x] `condition`'ın ham (unconditional) fiyat ilişkisinin monoton olmadığı doğrulandı.
- [x] `grade == 7` içinde kontrollü karşılaştırma yapıldığında `condition`'ın **gerçek, pozitif**
  bir ilişkisi ortaya çıktı.
- [x] Confound kaynağı (`condition=3` grubunun daha yüksek ortalama `grade`'e sahip olması)
  somut sayılarla tespit edildi.
- [x] Bu bulgunun Section 21'deki OLS katsayı yorumlamasına nasıl bağlanacağı not edildi.


---
## Section 14 — EDA Findings & Hypotheses

### Learning Objective

Session 2-3'teki bulguları, test edilebilir (ama henüz test edilmemiş) bir **hypothesis
register**'a dönüştürmek.

### Critical Engineering Notes

Aşağıdaki hipotezler, Session 2-3'teki **gözlemsel EDA bulgularına dayanmaktadır** — bunlar
**doğrulanmış sonuçlar değildir**. Her biri, Session 5'te (OLS Statistical Diagnostics)
istatistiksel olarak **test edilecektir** (t-test, F-test, güven aralığı). Burada yalnızca
hipotezleri **kayda geçiriyoruz**.

### Hands-On Implementation — Hypothesis Register


In [ ]:
hypothesis_register = pd.DataFrame([
    {
        "ID": "H1",
        "Hypothesis": "Living area (sqft_living) and price are positively associated.",
        "EDA Evidence": "Pearson r ≈ 0.712 (Section 11); largely linear scatter with LOWESS overlap (Section 10).",
        "To Be Tested In": "Section 21-22 (t-test on sqft_living coefficient)",
    },
    {
        "ID": "H2",
        "Hypothesis": "Building grade (grade) is associated with price, and the relationship may be nonlinear (convex).",
        "EDA Evidence": "Pearson r ≈ 0.671; LOWESS vs linear divergence at high grade (Section 10); monotonic median increase (Section 12).",
        "To Be Tested In": "Section 16 (polynomial term), Section 21-22",
    },
    {
        "ID": "H3",
        "Hypothesis": "Location (captured via zipcode/lat/long) is important in explaining price, despite near-zero raw numeric correlation with zipcode.",
        "EDA Evidence": "9x median price ratio between top and bottom zipcode (Section 11-12); spatial clustering in lat/long scatter (Section 12).",
        "To Be Tested In": "Section 17 (encoding), Section 21-22 (joint F-test on zipcode dummies)",
    },
    {
        "ID": "H4",
        "Hypothesis": "House age effect on price may be nonlinear (non-monotonic).",
        "EDA Evidence": "U-shaped LOWESS pattern for yr_built vs price (Section 10).",
        "To Be Tested In": "Section 15-16 (house_age_at_sale feature, possible quadratic term)",
    },
    {
        "ID": "H5",
        "Hypothesis": "Feature interactions may change how a feature relates to price (e.g., condition's effect depends on grade).",
        "EDA Evidence": "Simpson's-paradox-like confounding between condition and grade (Section 13).",
        "To Be Tested In": "Section 27 (interaction model specification)",
    },
    {
        "ID": "H6",
        "Hypothesis": "A raw-price OLS model may produce heteroscedastic errors (error variance increasing with price level).",
        "EDA Evidence": "Price distribution is highly right-skewed (skew ≈4.26) with spread increasing by grade group (Section 06, Section 12).",
        "To Be Tested In": "Section 23 (Breusch-Pagan test, Scale-Location plot)",
    },
])
hypothesis_register


### Output Interpretation

Bu altı hipotez, prompt'ta örnek olarak verilen hipotezlerle uyumludur ve her biri **gerçek EDA
bulgusuna** dayanmaktadır (uydurma değildir). Her hipotez, Session 4-6'daki belirli bir section'a
açıkça bağlanmıştır — bu, capstone'un ileriki bölümlerinin **rastgele değil, bu EDA bulgularının
doğrudan takibi** olduğunu gösterir.

### Student Exercise

**Session 3 Exercise:** Kendi gözlemlediğiniz (yukarıdaki 6 hipotezde yer almayan) bir
feature-target ilişkisi için **7. bir hipotez** yazın. Hangi EDA kanıtına dayandığını ve hangi
section'da test edilebileceğini belirtin.


In [ ]:
# Student Exercise:
# TODO: Kendi hipotezinizi (H7) hypothesis_register'a eklemek için yeni bir satır DataFrame'i
#       oluşturup pd.concat ile birleştirin.

# new_hypothesis = pd.DataFrame([{
#     "ID": "H7", "Hypothesis": "...", "EDA Evidence": "...", "To Be Tested In": "...",
# }])
# hypothesis_register_extended = pd.concat([hypothesis_register, new_hypothesis], ignore_index=True)
# print(hypothesis_register_extended)


### Checkpoint

- [x] Altı hipotez, gerçek EDA bulgularına referansla kayda geçirildi — **doğrulanmış sonuç
  olarak sunulmadı**.
- [x] Her hipotez, Session 4-6'da test edileceği spesifik section'a bağlandı.

---

## Session 3 Özeti

Bu oturumda, yine yalnızca `train_df` üzerinde:

- `sqft_living`, `grade`, `yr_built` için feature-target scatter analizleri yaptık; `grade`'in
  konveks, `yr_built`'in U-şekilli nonlinear örüntüler gösterdiğini bulduk.
- Pearson/Spearman korelasyonlarını karşılaştırdık; `zipcode` örneğiyle korelasyonun nominal
  değişkenler için **yanıltıcı** olabileceğini (9x fiyat farkına karşın ≈0 korelasyon) somut
  olarak gösterdik.
- Pairplot, spatial scatter ve grup-tabanlı boxplot'larla multivariate ilişkileri inceledik.
- `condition`-`price` ilişkisinde gerçek bir **confounding** örneği (grade kontrolü ile ortaya
  çıkan ters bulgu) keşfettik.
- Altı EDA bulgusunu, ileriki section'larla açıkça ilişkilendirilmiş bir **hypothesis
  register**'a dönüştürdük.

Session 4'te bu bulguları **feature engineering** ve **multicollinearity analizi**ne
dönüştüreceğiz.


---
# SESSION 4 — Feature Engineering & Multicollinearity
**Toplam Süre: 150 dakika (140 dakika aktif + 10 dakika ara)**

Session 2-3'teki kural geçerliliğini sürdürüyor: feature engineering ve multicollinearity
analizi **yalnızca `train_df` üzerinde** yapılır. Türetilen feature'lar, Section 19'un sonunda
`val_df`/`test_df`'e **aynı formüllerle** (yeniden fit edilmeden) uygulanacaktır.


---
## Section 15 — Domain-Based Feature Engineering

### Learning Objective

Ham sütunlardan, domain bilgisine dayanan, her biri gerekçeli ve leakage riski değerlendirilmiş
sekiz yeni feature türetmek.

### Engineering Motivation

Ham sütunlar (`yr_built`, `yr_renovated`, `sqft_basement` vb.), OLS'e doğrudan verildiğinde
anlamlı olmayan veya yorumlanması zor katsayılar üretebilir (örn. `yr_built`'in katsayısı "yıl
arttıkça fiyat X kadar değişir" der, ama Section 10'da bu ilişkinin **U-şekilli** olduğunu
gördük — ham `yr_built` bunu yakalayamaz). Domain-based feature engineering, bu ham bilgiyi
**daha anlamlı bir forma** dönüştürür.

### Hands-On Implementation — Feature Definitions


In [ ]:
train_df["sale_year"] = train_df["date"].dt.year
train_df["sale_month"] = train_df["date"].dt.month
train_df["sale_quarter"] = train_df["date"].dt.quarter

train_df["house_age_at_sale"] = train_df["sale_year"] - train_df["yr_built"]
train_df["is_renovated_at_sale"] = (train_df["yr_renovated"] > 0).astype(int)
train_df["years_since_renovation"] = np.where(
    train_df["yr_renovated"] > 0, train_df["sale_year"] - train_df["yr_renovated"], np.nan
)
train_df["has_basement"] = (train_df["sqft_basement"] > 0).astype(int)
train_df["basement_ratio"] = train_df["sqft_basement"] / train_df["sqft_living"]
train_df["living_area_per_bedroom"] = train_df["sqft_living"] / train_df["bedrooms"].replace(0, np.nan)

print("New columns added:", [c for c in train_df.columns if c not in df.columns])


### Hands-On Implementation — Feature-by-Feature Specification

| Feature | Formula | Source Columns | Data Type | Domain Justification |
|---|---|---|---|---|
| `house_age_at_sale` | `sale_year - yr_built` | `date`, `yr_built` | int | Konutun satış anındaki yaşı — Section 10'daki U-şekilli ilişki ham `yr_built`'ten daha yorumlanabilir bir biçime taşınır. |
| `is_renovated_at_sale` | `1` if `yr_renovated > 0` else `0` | `yr_renovated` | binary | `yr_renovated`'ın structural-zero örüntüsünü (Section 03) açık bir binary sinyale çevirir. |
| `years_since_renovation` | `sale_year - yr_renovated` (renove edilmişse) | `date`, `yr_renovated` | float (NaN if not renovated) | Yalnızca renove edilmiş konutlar için anlamlı bir "ne kadar süre önce" bilgisi. |
| `has_basement` | `1` if `sqft_basement > 0` else `0` | `sqft_basement` | binary | `sqft_basement`'ın structural-zero örüntüsünü açık bir sinyale çevirir. |
| `basement_ratio` | `sqft_basement / sqft_living` | `sqft_basement`, `sqft_living` | float [0,1) | Bodrumun toplam yaşam alanına oranı — ham `sqft_basement`'tan farklı, ölçekten bağımsız bir bilgi. |
| `living_area_per_bedroom` | `sqft_living / bedrooms` | `sqft_living`, `bedrooms` | float | Oda başına büyüklük — yalnızca toplam büyüklüğün değil, "ferahlığın" bir göstergesi. |
| `sale_month` | `date.month` | `date` | int (1-12) | Mevsimsel etkiler için. |
| `sale_quarter` | `date.quarter` | `date` | int (1-4) | `sale_month`'tan daha az granüler bir mevsimsellik alternatifi. |

### Hands-On Implementation — Validation Rules (Her Feature İçin)


In [ ]:
print("house_age_at_sale: negative count (yr_built > sale_year anomaly propagates here):",
      (train_df["house_age_at_sale"] < 0).sum())
print("house_age_at_sale range:", train_df["house_age_at_sale"].min(), "-", train_df["house_age_at_sale"].max())
print()
print("years_since_renovation: negative count (yr_renovated > sale_year anomaly propagates here):",
      (train_df["years_since_renovation"] < 0).sum())
print("years_since_renovation: missing (not renovated) count:", train_df["years_since_renovation"].isna().sum(),
      f"({train_df['years_since_renovation'].isna().mean():.1%})")
print()
print("living_area_per_bedroom: missing (bedrooms == 0) count:", train_df["living_area_per_bedroom"].isna().sum())
print("living_area_per_bedroom: min/max:", train_df["living_area_per_bedroom"].min(), "/", train_df["living_area_per_bedroom"].max())


### Output Interpretation

- **`house_age_at_sale`**'de **12 negatif değer** var — bunlar Section 03'te zaten tespit
  ettiğimiz `yr_built > sale_year` anomalisinin **doğrudan sonucu**dur (bu anomali burada yeni
  bir sorun olarak "ortaya çıkmıyor", önceden işaretlediğimiz bir bulgunun doğal bir
  yansımasıdır). Bu değerler **silinmiyor veya sıfıra kırpılmıyor** — flag'lenmiş durumda
  kalıyor; Section 25'te (Influence Diagnostics) bu 12 kaydın model üzerindeki etkisi ayrıca
  incelenecektir.
- **`years_since_renovation`**'ın **%95.47'si eksik (NaN)** — bu **gerçek bir missing value
  değildir**, "hiç renove edilmemiş" konutlar için **tanımsız** bir büyüklüktür (0 koymak,
  "az önce renove edildi" ile "hiç renove edilmedi" anlamlarını birbirine karıştırır). Bu
  feature'ın bu yüksek eksiklik oranı nedeniyle Section 19'da **tasarım matrisine dahil
  edilmemesi** ve yalnızca `is_renovated_at_sale` (binary) feature'ının kullanılması
  önerilecektir — nihai karar Section 19'da verilecektir.
- **`living_area_per_bedroom`**'un minimum değeri (**≈49.1**), tam olarak Section 03'te
  bulduğumuz `bedrooms=33` anomalisine ait (1620 sqft / 33 = 49.1) — bu, türetilmiş
  feature'ların, orijinal anomalileri **büyütebileceğinin** iyi bir örneğidir.

### Critical Engineering Notes — Leakage Risk Review

| Feature | Leakage Risk |
|---|---|
| `house_age_at_sale`, `is_renovated_at_sale`, `years_since_renovation`, `has_basement`, `basement_ratio`, `living_area_per_bedroom`, `sale_month`, `sale_quarter` | **None** — tümü, Section 05'te onayladığımız "satış anında bilinen" ham sütunlardan (yalnızca `yr_built`, `yr_renovated`, `sqft_basement`, `sqft_living`, `bedrooms`, `date`) türetilmiştir. Hiçbiri `price`'ı veya satış-sonrası bir bilgiyi kullanmaz. |

### Hands-On Implementation — Seasonal Coverage Caveat


In [ ]:
print("sale_month value counts (training data):")
print(train_df["sale_month"].value_counts().sort_index())


### Critical Engineering Notes — External Validity of `sale_month`/`sale_quarter`

Training verisi **2014-05-02'den 2015-01-15'e** kadar sürüyor — bu nedenle **Şubat, Mart ve
Nisan aylarına ait HİÇBİR training örneği yok**. Bu iki ay, validation (`2015-01-16` →
`2015-03-25`) ve test (`2015-03-26` → `2015-05-27`) setlerinde **görülecektir**. Bu, `sale_month`
gibi bir feature'ın bu üç ay için **ekstrapolasyon** yapması gerektiği anlamına gelir — model
bu ayların mevsimsel etkisini **hiç görmeden** tahmin yapacaktır. Bu, kronolojik split'in doğal
bir sonucudur ve Section 29'da (Final Test Evaluation) hatırlanması gereken bir sınırlamadır.

### Student Exercise

`basement_ratio`'nun `has_basement=0` olan satırlarda her zaman tam olarak `0.0` olduğunu
doğrulayın. Bu iki feature'ın (biri binary, biri sürekli) birlikte kullanılmasının Section 18'de
ne tür bir multicollinearity riskine katkıda bulunabileceğini düşünün.


In [ ]:
# Student Exercise:
# TODO: train_df[train_df["has_basement"] == 0]["basement_ratio"] değerlerinin tamamının 0.0
#       olduğunu assert ile doğrulayın.
# TODO: has_basement ve basement_ratio arasındaki korelasyonu hesaplayıp yorumlayın.

# assert (train_df[train_df["has_basement"] == 0]["basement_ratio"] == 0.0).all()
# print(train_df["has_basement"].corr(train_df["basement_ratio"]))


### Checkpoint

- [x] Sekiz feature türetildi; her biri tanım, formül, kaynak sütun, tip, domain gerekçesi ve
  leakage riski ile belgelendi.
- [x] `house_age_at_sale`/`years_since_renovation`'daki negatif değerlerin Section 03'teki
  temporal anomalilerin doğal yansıması olduğu doğrulandı — **yeni bir sorun değil**.
- [x] `years_since_renovation`'ın yüksek eksiklik oranı (%95.47) nedeniyle Section 19'da
  muhtemelen dışarıda tutulacağı not edildi.
- [x] `sale_month`/`sale_quarter`'ın training verisinde kapsanmayan ayları (Şub-Nis) içerdiği ve
  bunun bir ekstrapolasyon riski oluşturduğu belgelendi.


---
## Section 16 — Numerical Transformations

### Learning Objective

Çarpık (skewed) sayısal feature'lara log transformation uygulamanın dağılım şekli üzerindeki
etkisini ölçmek; Section 10/14'te bulduğumuz nonlinear ilişkiler için quadratic terimlerin
**gerçekten** açıklayıcı güç eklediğini (yalnızca ham korelasyonla değil) doğrulamak.

### Hands-On Implementation — Log Transformation of Skewed Features


In [ ]:
skewed_features = ["sqft_living", "sqft_lot", "sqft_above", "sqft_basement", "sqft_living15", "sqft_lot15"]

log_transform_comparison = pd.DataFrame({
    "raw_skewness": [train_df[c].skew() for c in skewed_features],
    "log1p_skewness": [np.log1p(train_df[c]).skew() for c in skewed_features],
}, index=skewed_features).round(3)
log_transform_comparison


### Output Interpretation

`np.log1p()` (yani `log(1+x)`, `sqft_basement=0` gibi sıfır değerler için tanımlı) kullanıyoruz.
Sonuçlar çarpıcı: **`sqft_lot`'un skewness'i 10.80'den 0.93'e**, **`sqft_lot15`'inki 9.15'ten
0.95'e** düşüyor. `sqft_living`'in log-skewness'i (**-0.04**) artık neredeyse **tam simetrik**.
Bu, Section 19'da bu log-transformed versiyonların ham versiyonlara **tercih edilebileceğine**
dair güçlü bir ön kanıttır (nihai karar Section 19'da verilecektir).

### Hands-On Implementation — Quadratic Terms: A Correct vs Incorrect Diagnostic

Section 14'te `grade` ve `house_age_at_sale` için nonlinear (sırasıyla konveks ve U-şekilli)
örüntüler bulmuştuk. Naif bir yaklaşım, karesi alınmış bir terimin ($x^2$) `price` ile **ham
korelasyonuna** bakmak olurdu — ama bunun **yanıltıcı** olabileceğini göstereceğiz.


In [ ]:
train_df["house_age_sq"] = train_df["house_age_at_sale"] ** 2
train_df["grade_sq"] = train_df["grade"] ** 2

print("Naive (misleading) diagnostic — raw correlation of the SQUARED term alone with price:")
print(f"  corr(house_age_at_sale, price)    : {train_df['house_age_at_sale'].corr(train_df['price']):.4f}")
print(f"  corr(house_age_sq, price)         : {train_df['house_age_sq'].corr(train_df['price']):.4f}  <- looks weaker, misleading!")


### Critical Engineering Notes — Why the Naive Diagnostic Is Misleading

`house_age_sq`'nun `price` ile ham korelasyonu (**≈ -0.003**), `house_age_at_sale`'in kendi
korelasyonundan (**≈ -0.053**) **daha düşük** görünüyor — bu, "kare terim işe yaramıyor" gibi
yanlış bir sonuca götürebilir. **Ancak bu yanlış bir testtir**: $x^2$'nin tek başına korelasyonu,
$x$ **modelde zaten varken** $x^2$'nin **ek açıklayıcı güç** ekleyip eklemediğini ölçmez. Doğru
test, $x$ ve $x^2$'yi **birlikte** bir OLS modeline koyup, $R^2$'deki artışı ve $x^2$
katsayısının anlamlılığını incelemektir.

### Hands-On Implementation — The Correct Diagnostic: Nested OLS Comparison


In [ ]:
import statsmodels.formula.api as smf

model_age_linear = smf.ols("price ~ house_age_at_sale", data=train_df).fit()
model_age_quadratic = smf.ols("price ~ house_age_at_sale + house_age_sq", data=train_df).fit()

print(f"R² (age only)      : {model_age_linear.rsquared:.4f}")
print(f"R² (age + age²)    : {model_age_quadratic.rsquared:.4f}")
print(f"age² coefficient p-value: {model_age_quadratic.pvalues['house_age_sq']:.2e}")


In [ ]:
model_grade_linear = smf.ols("price ~ grade", data=train_df).fit()
model_grade_quadratic = smf.ols("price ~ grade + grade_sq", data=train_df).fit()

print(f"R² (grade only)     : {model_grade_linear.rsquared:.4f}")
print(f"R² (grade + grade²) : {model_grade_quadratic.rsquared:.4f}")
print(f"grade² coefficient p-value: {model_grade_quadratic.pvalues['grade_sq']:.2e}")


### Output Interpretation

Doğru test, tam tersi bir sonuç veriyor: `house_age_sq` eklendiğinde R², **0.0029'dan 0.0287'ye**
(≈10 katına) yükseliyor ve katsayısı **son derece anlamlı** (p ≈ 3.75e-88). `grade_sq` eklendiğinde
R², **0.450'den 0.516'ya** yükseliyor, o da son derece anlamlı. **Bu, Section 14'teki H2 ve
H4 hipotezlerini destekleyen ilk istatistiksel kanıttır** (henüz final model değil — bunlar basit
iki-değişkenli modellerdir, çok değişkenli modelde bu etkiler Section 22'de yeniden test
edilecektir).

**Önemli not (Statistical vs Practical Significance):** p-value'nun aşırı küçük olması
(`3.75e-88`), bu etkinin **büyüklüğünün** de büyük olduğu anlamına gelmez — bu ayrımı Section 22'de
ayrıntılı ele alacağız. Burada yalnızca **istatistiksel anlamlılığı** ve **R² artışını**
gösteriyoruz.

### Scaling Considerations

`statsmodels` OLS, `scikit-learn`'ün regularized modellerinin (Ridge/Lasso, LAB05'te
gördüğünüz) aksine, **ölçeklendirme (scaling) gerektirmez** — OLS katsayı tahminleri feature
ölçeğinden matematiksel olarak etkilenmez (yalnızca katsayının **birimi** değişir). Bu nedenle bu
capstone'da `StandardScaler` **kullanılmayacaktır** — ancak Section 18'de göreceğimiz gibi,
ölçek farkları **Condition Number** gibi sayısal kararlılık ölçütlerini etkileyebilir.

### Student Exercise

`sqft_living15` için de aynı naif-vs-doğru diagnostik karşılaştırmasını (karesi alınmış terimin
ham korelasyonu vs nested OLS R² karşılaştırması) uygulayın.


In [ ]:
# Student Exercise:
# TODO: train_df["sqft_living15_sq"] = train_df["sqft_living15"] ** 2 oluşturun.
# TODO: smf.ols("price ~ sqft_living15", ...) ve smf.ols("price ~ sqft_living15 + sqft_living15_sq", ...)
#       modellerini karşılaştırıp R² artışını ve p-value'yu yorumlayın.

# train_df["sqft_living15_sq"] = train_df["sqft_living15"] ** 2
# m1 = smf.ols("price ~ sqft_living15", data=train_df).fit()
# m2 = smf.ols("price ~ sqft_living15 + sqft_living15_sq", data=train_df).fit()
# print(m1.rsquared, m2.rsquared, m2.pvalues["sqft_living15_sq"])


### Checkpoint

- [x] Log transformation'ın 6 çarpık feature'ın skewness'ini önemli ölçüde azalttığı
  (`sqft_lot`: 10.80→0.93) doğrulandı.
- [x] Kare terimin **tek başına ham korelasyonuna** bakmanın yanıltıcı olabileceği, `house_age_sq`
  örneğiyle somut olarak gösterildi.
- [x] Doğru diagnostik (nested OLS, R² artışı + p-value) ile `house_age_sq` ve `grade_sq`'nin
  gerçek açıklayıcı güç eklediği doğrulandı (H2, H4 hipotezleri için ilk istatistiksel kanıt).
- [x] OLS için scaling'in gerekli olmadığı, ama Section 18'de Condition Number açısından
  önemli olacağı not edildi.


---
## Section 17 — Categorical Encoding

### Learning Objective

Her kategorik/ordinal feature için doğru encoding stratejisini seçmek; **dummy variable trap**'i
somut bir örnekle göstermek; `zipcode` gibi yüksek cardinality'li bir nominal feature'ı
yönetmek.

### Concept Explanation

- **Binary Encoding**: `waterfront`, `has_basement`, `is_renovated_at_sale` zaten `{0, 1}`
  biçiminde — ek bir encoding gerekmez.
- **Ordinal Representation**: `view` (0-4), `condition` (1-5), `grade` (1-13) — bu feature'lar
  **doğal bir sıralamaya** sahiptir. Section 16'da `grade`'in `price` ile **sayısal** bir
  ilişkisi (dahası, anlamlı bir quadratic terimi) olduğunu gösterdik — bu nedenle bu üçünü
  **One-Hot Encoding yapmadan, sayısal ordinal değerler olarak** tutacağız. Bu, 12+4+5=21 ek
  dummy değişken yerine yalnızca 3 sayısal feature kullanmamızı sağlar.
- **One-Hot Encoding (Nominal)**: `zipcode` (70 kategori) ve `sale_quarter` (4 kategori)
  **sıralaması olmayan (nominal)** feature'lardır — bunlar One-Hot Encoded olacaktır.
- **Reference Category / Dummy Variable Trap**: Bir kategorik feature'ın **tüm** kategorileri
  dummy olarak eklenip bir de intercept bulunduğunda, dummy'lerin toplamı her zaman `1`'e eşit
  olur — bu, intercept sütunuyla **tam doğrusal bağımlıdır** (exact multicollinearity). Çözüm:
  bir kategoriyi **referans (reference)** olarak bırakıp onu dummy olarak eklememek.

### Hands-On Implementation — Demonstrating the Dummy Variable Trap


In [ ]:
# Deliberately build the INCORRECT encoding: intercept + ALL zipcode dummies (no reference dropped).
all_zip_dummies = pd.get_dummies(train_df["zipcode"], prefix="zip").astype(float)
trap_design_matrix = all_zip_dummies.copy()
trap_design_matrix.insert(0, "Intercept", 1.0)

trap_rank = np.linalg.matrix_rank(trap_design_matrix.values)
trap_n_cols = trap_design_matrix.shape[1]

print(f"Design matrix shape: {trap_design_matrix.shape}")
print(f"Matrix rank: {trap_rank}")
print(f"Number of columns: {trap_n_cols}")
print(f"Rank-deficient by exactly: {trap_n_cols - trap_rank} column(s)")


### Output Interpretation

Intercept + **tüm 70** zipcode dummy'si ile oluşturulan 71 sütunluk tasarım matrisinin rank'i
**tam olarak 70** — yani matris **1 sütun eksik rank'e sahip (rank-deficient)**. Bu, 70
dummy'nin toplamının her satırda tam olarak `1` (yani Intercept sütunuyla özdeş) olmasından
kaynaklanır — bu **tam (perfect) bir multicollinearity** örneğidir, tahmini bir sonuç değil,
matematiksel bir zorunluluktur. Bu tasarım matrisiyle OLS **çözülemez** (katsayılar benzersiz
biçimde tahmin edilemez).

### Hands-On Implementation — The Correct Encoding (Reference Category Dropped)


In [ ]:
correct_zip_dummies = pd.get_dummies(train_df["zipcode"], prefix="zip", drop_first=True).astype(float)
correct_design_matrix = correct_zip_dummies.copy()
correct_design_matrix.insert(0, "Intercept", 1.0)

correct_rank = np.linalg.matrix_rank(correct_design_matrix.values)
correct_n_cols = correct_design_matrix.shape[1]

print(f"Design matrix shape: {correct_design_matrix.shape}")
print(f"Matrix rank: {correct_rank}")
print(f"Full column rank: {correct_rank == correct_n_cols}")


### Output Interpretation

`drop_first=True` ile bir zipcode kategorisini (alfabetik olarak ilkini) **referans** olarak
dışarıda tutunca, 70 sütunluk (Intercept + 69 dummy) matris **tam rank'e** ulaşıyor. Bu
capstone'da `statsmodels`'in formula API'si (`C(zipcode, Treatment(reference=...))`) bu referans
bırakma işlemini **otomatik olarak** yapar — Section 19'da OLS modelini kurarken bunu
kullanacağız; **en sık görülen zipcode'u (`98103`) referans kategori** olarak seçeceğiz (bu,
intercept'in yorumunu "en tipik lokasyondaki konut" olarak tutar).

### Hands-On Implementation — Unknown Category Check (Train vs Validation/Test)


In [ ]:
unknown_category_checks = {}
for col in ["zipcode", "grade", "view", "condition", "sale_quarter"]:
    train_categories = set(train_df[col].unique()) if col != "sale_quarter" else set(train_df["date"].dt.quarter.unique())
    unknown_category_checks[col] = train_categories

print("Categories observed in training data (CP10 input):")
for col, cats in unknown_category_checks.items():
    print(f"  {col}: {len(cats)} unique values")


### Critical Engineering Notes — Unknown Categories

Yukarıdaki kategoriler **training verisinden** kaydedildi. Section 19'un sonunda, bu kategorik
feature'lar `val_df`/`test_df`'e uygulanırken, eğer **training'de görülmemiş bir kategori**
(örn. yeni bir zipcode) karşılaşılırsa, bu modelin tahmin yapamayacağı bir durum oluşturur.
`statsmodels`'in formula API'si, bu durumla **otomatik olarak** başa çıkmaz (sklearn'ün
`OneHotEncoder(handle_unknown="ignore")`'unun aksine) — bu nedenle CP10 kontrolü, Section 19'da
**val/test setlerinde training'de görülmemiş bir kategori olup olmadığını açıkça
doğrulayacaktır**.

### Hands-On Implementation — High-Cardinality `zipcode`: A Design Trade-off


In [ ]:
print(f"zipcode: {train_df['zipcode'].nunique()} categories -> {train_df['zipcode'].nunique() - 1} dummy variables")
print(f"This adds {train_df['zipcode'].nunique() - 1} parameters to estimate from {len(train_df)} training rows.")
print(f"Average training rows per zipcode category: {len(train_df) / train_df['zipcode'].nunique():.1f}")


### Output Interpretation

`zipcode`'u One-Hot Encode etmek, modele **69 ek parametre** ekliyor — bu, Section 18'de
inceleyeceğimiz **Condition Number**'ı önemli ölçüde büyütebilir ve her bir zipcode katsayısının
standart hatasını artırabilir (özellikle ortalama **≈216 satır/zipcode** olduğu için, en nadir
zipcode'larda — örn. `98039`'da sadece 37 satır — bu katsayı daha az kesin tahmin edilecektir).
Bu, "zipcode'u dahil etmeyelim" anlamına **gelmez** (Section 11-12'de gördüğümüz gibi konumun
etkisi çok güçlü) — ama bu trade-off'u **açıkça kabul ediyoruz**.

### Student Exercise

`sale_quarter`'ı One-Hot Encode edip (`pd.get_dummies(train_df["sale_quarter"], drop_first=True)`),
kaç dummy değişken oluştuğunu ve bunların `Intercept` ile birlikte full column rank'e sahip
olup olmadığını doğrulayın.


In [ ]:
# Student Exercise:
# TODO: pd.get_dummies(train_df["sale_quarter"], prefix="q", drop_first=True) ile dummy'leri oluşturun.
# TODO: Intercept sütunu ekleyip np.linalg.matrix_rank() ile full column rank olduğunu doğrulayın.

# quarter_dummies = pd.get_dummies(train_df["sale_quarter"], prefix="q", drop_first=True).astype(float)
# quarter_design = quarter_dummies.copy()
# quarter_design.insert(0, "Intercept", 1.0)
# print(np.linalg.matrix_rank(quarter_design.values), quarter_design.shape[1])


### Checkpoint

- [x] Dummy variable trap, gerçek bir rank-deficiency (71 sütun, rank 70) ile **somut olarak**
  gösterildi.
- [x] Referans kategori bırakılarak (`drop_first=True`) full column rank'e ulaşıldığı
  doğrulandı.
- [x] `zipcode`/`grade`/`view`/`condition`/`sale_quarter` için training kategorileri kaydedildi
  (CP10'un girdisi olarak); val/test'te bilinmeyen kategori kontrolü Section 19'a aktarıldı.
- [x] `zipcode`'un yüksek cardinality'sinin getirdiği parametre sayısı/katsayı kesinliği
  trade-off'u açıkça belgelendi.


---
## Section 18 — Multicollinearity Analysis

**Bu bölüm zorunludur** (prompt'ta açıkça belirtilmiştir — capstone'un istatistiksel
çekirdeğinin bir parçasıdır).

### Learning Objective

Perfect multicollinearity ile high multicollinearity arasındaki farkı; VIF, Condition Number ve
Matrix Rank'in ne ölçtüğünü; ve multicollinearity'nin katsayı kararlılığı/standart hatalar
üzerindeki somut etkisini **gerçek verilerle** göstermek.

### Statistical Foundations

Variance Inflation Factor, bir feature'ın diğer tüm feature'lar tarafından ne kadar
"açıklandığını" ölçer:

$$VIF_j=\frac{1}{1-R_j^2}$$

Burada $R_j^2$, $x_j$'nin diğer tüm feature'lara karşı regresyonundan elde edilen R²'dir.
$R_j^2 \to 1$ olduğunda (yani $x_j$ diğerleri tarafından tam açıklanabildiğinde) $VIF_j \to
\infty$.

### Hands-On Implementation — Perfect Multicollinearity (The `sqft_above + sqft_basement = sqft_living` Identity)

Section 03'te bu kimliğin **21,613 satırın tamamında (%100)** tam olarak doğrulandığını
bulmuştuk. Şimdi bu kimliğin bir design matrix'te ne yaptığını gösterelim.


In [ ]:
import warnings
from statsmodels.stats.outliers_influence import variance_inflation_factor

perfect_collinearity_cols = ["sqft_living", "sqft_above", "sqft_basement", "bedrooms"]
X_perfect = train_df[perfect_collinearity_cols].copy()
X_perfect.insert(0, "const", 1.0)

perfect_rank = np.linalg.matrix_rank(X_perfect.values)
print(f"Design matrix shape: {X_perfect.shape}")
print(f"Matrix rank: {perfect_rank}  (expected full rank would be {X_perfect.shape[1]})")
print(f"Singular / rank-deficient: {perfect_rank < X_perfect.shape[1]}")


In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    perfect_vifs = [variance_inflation_factor(X_perfect.values, i) for i in range(X_perfect.shape[1])]
    for w in caught:
        print(f"WARNING: {w.category.__name__}: {str(w.message)[:150]}")

print()
for name, vif in zip(["const"] + perfect_collinearity_cols, perfect_vifs):
    print(f"  {name:15s}: VIF = {vif:.2e}")


### Output Interpretation

Tasarım matrisinin rank'i **4**, ama 5 sütunu var — matris **rank-deficient (singular)**. VIF
hesaplaması, `sqft_living`/`sqft_above`/`sqft_basement` için **≈1.0×10¹⁵** (pratikte sonsuz) bir
değer üretiyor ve `statsmodels` otomatik olarak **"design matrix is poorly conditioned"**
uyarısı veriyor. Bu, bir tahmin hatası değil — matematiksel bir zorunluluktur: `sqft_above +
sqft_basement`, `sqft_living`'i **tam olarak** yeniden üretir, bu üçü asla aynı OLS modelinde
**birlikte** anlamlı katsayılarla bulunamaz. **Çözüm: bu üçünden en fazla ikisini aynı anda
kullanmak** (Section 19'da `sqft_living`'i tutup `sqft_above`/`sqft_basement`'i çıkaracağız).

### Hands-On Implementation — High (but not Perfect) Multicollinearity & Coefficient Instability


In [ ]:
model_simple = smf.ols("price ~ sqft_living", data=train_df).fit()
model_with_collinear_pair = smf.ols("price ~ sqft_living + sqft_above", data=train_df).fit()

print("Model A: price ~ sqft_living")
print(f"  sqft_living coef = {model_simple.params['sqft_living']:.2f}, SE = {model_simple.bse['sqft_living']:.3f}")
print()
print("Model B: price ~ sqft_living + sqft_above  (corr(sqft_living, sqft_above) = 0.876)")
print(f"  sqft_living coef = {model_with_collinear_pair.params['sqft_living']:.2f}, SE = {model_with_collinear_pair.bse['sqft_living']:.3f}")
print(f"  sqft_above  coef = {model_with_collinear_pair.params['sqft_above']:.2f}, SE = {model_with_collinear_pair.bse['sqft_above']:.3f}")
print()
se_inflation = model_with_collinear_pair.bse['sqft_living'] / model_simple.bse['sqft_living']
print(f"Standard error inflation factor for sqft_living: {se_inflation:.2f}x")


### Output Interpretation

Bu, **high (perfect olmayan) multicollinearity**'nin somut bir örneğidir:

- `sqft_living`'in standart hatası, `sqft_above` eklendiğinde **≈2.08x artıyor** (2.27 → 4.72) —
  bu **inflated standard error**'un ta kendisidir: katsayı tahmini daha **kararsız (unstable)**
  hale geliyor.
- `sqft_above`'ın katsayısı **negatif (-14.59)** çıkıyor — oysa `sqft_above`'ın `price` ile ham
  korelasyonu **pozitiftir** (Section 11: ≈0.616). Bu **işaret değişimi (sign flip)**,
  multicollinearity'nin klasik bir belirtisidir: iki feature birbirine çok benzer bilgi
  taşıdığında, OLS bu bilgiyi ikisi arasında "keyfi" biçimde paylaştırabilir, sonuç katsayılar
  sezgisel olarak yorumlanamaz hale gelir.

**Önemli ayrım (Perfect vs High Multicollinearity):** Yukarıdaki `sqft_living`+`sqft_above`
modeli **hâlâ çözülebilir** (rank tam, VIF sonlu — aşağıda göreceğiz) — bu **yüksek ama
perfect olmayan** bir multicollinearity'dir. Önceki örnek (`sqft_living`+`sqft_above`+
`sqft_basement`) ise **perfect**'tir (rank-deficient, VIF sonsuz). Bu ikisi **aynı problem
değildir** — biri matematiksel olarak çözümsüz, diğeri çözülebilir ama güvenilmez.

### Hands-On Implementation — Condition Number: A Scale-Sensitive Metric


In [ ]:
candidate_features_no_redundancy = [
    "sqft_living", "bedrooms", "bathrooms", "floors", "grade", "condition",
    "view", "waterfront", "house_age_at_sale", "lat", "long",
]

X_raw = train_df[candidate_features_no_redundancy].copy()
X_raw.insert(0, "const", 1.0)
X_standardized = (train_df[candidate_features_no_redundancy] - train_df[candidate_features_no_redundancy].mean()) / train_df[candidate_features_no_redundancy].std()
X_standardized.insert(0, "const", 1.0)

print(f"Design matrix rank (raw scale)         : {np.linalg.matrix_rank(X_raw.values)} / {X_raw.shape[1]} (full rank)")
print(f"Condition Number (raw, un-standardized) : {np.linalg.cond(X_raw.values):,.1f}")
print(f"Condition Number (standardized features): {np.linalg.cond(X_standardized.values):,.2f}")


### Critical Engineering Notes — Condition Number vs VIF

Bu iki condition number arasındaki fark **çarpıcı**: ham ölçekte **≈2,338,450**, standartlaştırılmış
ölçekte ise **≈4.91**. **Aynı feature seti, aynı multicollinearity yapısı** — ama Condition
Number, feature'ların **ölçeğine (birimine)** son derece duyarlıdır (örn. `lat`≈47 ile
`sqft_living`≈2000 çok farklı büyüklük mertebelerinde). **VIF ise ölçekten bağımsızdır** —
aşağıda doğrulayacağız.

### Hands-On Implementation — VIF Is Scale-Invariant


In [ ]:
vif_raw = [variance_inflation_factor(X_raw.values, i) for i in range(1, X_raw.shape[1])]
vif_standardized = [variance_inflation_factor(X_standardized.values, i) for i in range(1, X_standardized.shape[1])]

vif_comparison = pd.DataFrame({
    "feature": candidate_features_no_redundancy,
    "VIF (raw scale)": vif_raw,
    "VIF (standardized)": vif_standardized,
}).sort_values("VIF (raw scale)", ascending=False)
vif_comparison


### Output Interpretation

VIF değerleri, ham ve standartlaştırılmış ölçekte **neredeyse birebir aynı** (küçük yuvarlama
farkları dışında) — bu, VIF'in matematiksel tanımının ölçekten bağımsız olduğunu doğruluyor. En
yüksek VIF'e sahip feature **`sqft_living`** (≈4.19) — bu, genellikle kullanılan **VIF > 5 veya
VIF > 10** eşiklerinin **altında**, yani bu feature seti (redundant `sqft_above`/
`sqft_basement` çıkarıldıktan sonra) multicollinearity açısından **kabul edilebilir** düzeyde.

### Critical Engineering Notes — VIF Eşiklerini Mekanik Olarak Uygulamama

**VIF > 5** veya **VIF > 10** gibi eşikler, literatürde yaygın **kural-of-thumb**'lardır, ama
**mekanik bir "sil" kuralı olarak uygulanmamalıdır**. Bu capstone'da VIF'i, hangi feature'ların
**birbirleriyle örtüştüğünü anlamak** için bir **teşhis aracı** olarak kullanıyoruz — nihai
feature seçimi (Section 19), VIF'in yanı sıra domain bilgisi, istatistiksel anlamlılık ve
validation performansını **birlikte** değerlendirecektir.

### Student Exercise

`sqft_living15`'i `candidate_features_no_redundancy` listesine ekleyip VIF tablosunu yeniden
hesaplayın. `sqft_living15`'in VIF'i, `sqft_living` ile olan korelasyonu (Section 11: ≈0.758)
göz önüne alındığında beklediğiniz gibi mi?


In [ ]:
# Student Exercise:
# TODO: candidate_features_no_redundancy listesine "sqft_living15" ekleyin.
# TODO: VIF tablosunu yeniden hesaplayıp sqft_living ve sqft_living15'in VIF değerlerini karşılaştırın.

# extended_features = candidate_features_no_redundancy + ["sqft_living15"]
# X_extended = train_df[extended_features].copy()
# X_extended.insert(0, "const", 1.0)
# vifs_extended = [variance_inflation_factor(X_extended.values, i) for i in range(1, X_extended.shape[1])]
# print(pd.Series(vifs_extended, index=extended_features).sort_values(ascending=False))


### Checkpoint

- [x] **Perfect multicollinearity** (`sqft_living`+`sqft_above`+`sqft_basement`), rank-deficiency
  (4/5) ve sonsuz VIF (≈10¹⁵) ile somut olarak gösterildi.
- [x] **High (perfect olmayan) multicollinearity**, `sqft_living`+`sqft_above` çiftinde standart
  hata enflasyonu (2.08x) ve katsayı işaret değişimi (sign flip) ile gösterildi.
- [x] **Condition Number**'ın ölçeğe duyarlı (2.3M ham vs 4.91 standartlaştırılmış), **VIF**'in
  ise ölçekten bağımsız olduğu doğrulandı.
- [x] Redundancy'den arındırılmış bir feature setinin VIF'lerinin kabul edilebilir düzeyde
  (<5) olduğu gösterildi.
- [x] VIF eşiklerinin mekanik silme kuralı olarak kullanılmayacağı, Section 19'un üç katmanlı
  feature selection'ının bir girdisi olacağı açıkça belirtildi.


---
## Section 19 — Feature Selection & Design Matrix

### Learning Objective

Session 4'ün tüm bulgularını (domain bilgisi, VIF/rank analizi, validation performansı) **üç
katmanlı bir feature selection süreciyle** birleştirip, Session 5'te kullanılacak **nihai,
full-column-rank** tasarım matrisini oluşturmak.

### Hands-On Implementation — Layer 1: Domain-Based Selection


In [ ]:
domain_based_exclusions = pd.DataFrame([
    {"Feature": "id, date", "Reason": "Identifier / raw timestamp, not a model feature (derived features used instead)"},
    {"Feature": "sqft_above, sqft_basement", "Reason": "Perfect multicollinearity with sqft_living (Section 18) — sqft_living retained, components dropped"},
    {"Feature": "years_since_renovation", "Reason": "95.47% missing by construction (Section 15); is_renovated_at_sale retained instead"},
    {"Feature": "sqft_living15, sqft_lot15", "Reason": "Unverified prediction-time availability (Section 01/05) — excluded for this baseline design; a documented limitation, not a proven leak"},
    {"Feature": "living_area_per_bedroom", "Reason": "Ratio of two features already in the model (sqft_living, bedrooms); high VIF when included (checked below)"},
    {"Feature": "has_basement", "Reason": "Redundant with basement_ratio (both derived from sqft_basement); basement_ratio retained as the richer continuous signal"},
])
domain_based_exclusions


### Hands-On Implementation — Anomaly Treatment Decision (from Section 03)

Section 03'te `bedrooms=33` kaydı (`id=2402100895`) için **"flag, exclude from training fit"**
stratejisi planlanmıştı. Bu kararı şimdi uyguluyoruz.


In [ ]:
anomalous_id = 2402100895
is_anomalous = train_df["id"] == anomalous_id

print(f"Rows flagged as anomalous (bedrooms=33): {is_anomalous.sum()}")
print(f"Rows with bedrooms == 0 (kept, just flagged, not excluded): {(train_df['bedrooms'] == 0).sum()}")

#  reset_index(drop=True) keeps row LABELS aligned with row POSITIONS -- this matters later
# (Section 25) when numpy arrays from get_influence() are matched back to specific rows.
train_model_df = train_df[~is_anomalous].copy().reset_index(drop=True)
print(f"\ntrain_df shape (before exclusion): {train_df.shape}")
print(f"train_model_df shape (after excluding the bedrooms=33 row): {train_model_df.shape}")


### Critical Engineering Notes

**`bedrooms=0` kayıtları (13 satır) hariç tutulmuyor** — Section 03'te bu kayıtlar için
"keep, flag" kararı verilmişti (stüdyo tipi konutlar olabileceği değerlendirmesiyle). Sadece
**tek bir** kayıt (`bedrooms=33`) training fit'inden çıkarılıyor. Bu karar Section 25'te
(Influence Diagnostics) **yeniden değerlendirilecektir** — eğer bu kaydın dahil edilmesi/
çıkarılması model sonuçlarını önemli ölçüde değiştiriyorsa bu orada ayrıca tartışılacaktır.

### Hands-On Implementation — Layer 2: Statistical Selection (Centering Polynomial Terms)

Section 18'de `grade`/`grade²` ve `house_age_at_sale`/`house_age_at_sale²` çiftlerinin VIF'inin
çok yüksek (≈99-101 ve ≈17-22) olduğunu bulmuştuk. Standart bir istatistiksel çözüm: polinom
terimi oluşturmadan önce **feature'ı merkezlemek (centering)**.


In [ ]:
GRADE_MEAN = train_model_df["grade"].mean()
AGE_MEAN = train_model_df["house_age_at_sale"].mean()

train_model_df["grade_centered"] = train_model_df["grade"] - GRADE_MEAN
train_model_df["grade_centered_sq"] = train_model_df["grade_centered"] ** 2
train_model_df["age_centered"] = train_model_df["house_age_at_sale"] - AGE_MEAN
train_model_df["age_centered_sq"] = train_model_df["age_centered"] ** 2

centered_check_cols = ["grade_centered", "grade_centered_sq", "age_centered", "age_centered_sq"]
X_centered_check = train_model_df[centered_check_cols].copy()
X_centered_check.insert(0, "const", 1.0)
vifs_centered = [variance_inflation_factor(X_centered_check.values, i) for i in range(1, X_centered_check.shape[1])]
print(pd.Series(vifs_centered, index=centered_check_cols))


### Output Interpretation

Centering, VIF'leri **≈99/101'den ≈1.6/1.3'e**, **≈17-22'den ≈1.6/1.2'ye** düşürüyor —
dramatik bir iyileşme. **Önemli:** `GRADE_MEAN` ve `AGE_MEAN`, **yalnızca training verisinden**
hesaplandı (CP08 kuralı: preprocessing training'e fit edilir) — bu sabit değerler, Section 19'un
sonunda `val_df`/`test_df`'e **yeniden hesaplanmadan, aynen** uygulanacaktır.

### Hands-On Implementation — Statistical Selection: Final VIF & Rank Check


In [ ]:
final_numeric_features = [
    "sqft_living", "bedrooms", "bathrooms", "floors", "grade_centered", "grade_centered_sq",
    "condition", "view", "waterfront", "age_centered", "age_centered_sq", "lat", "long",
    "basement_ratio", "is_renovated_at_sale",
]

X_final_numeric = train_model_df[final_numeric_features].copy()
X_final_numeric.insert(0, "const", 1.0)

final_rank = np.linalg.matrix_rank(X_final_numeric.values)
print(f"Rank: {final_rank} / {X_final_numeric.shape[1]} (full column rank: {final_rank == X_final_numeric.shape[1]})")

final_vifs = [variance_inflation_factor(X_final_numeric.values, i) for i in range(1, X_final_numeric.shape[1])]
print(pd.Series(final_vifs, index=final_numeric_features).sort_values(ascending=False))


### Output Interpretation

Tüm VIF'ler artık **5'in altında** (en yüksek: `sqft_living` ≈4.91) — bu, Layer 1 (domain) ve
Layer 2'nin (statistical, centering) birlikte uygulanmasıyla elde edilen **temiz bir numeric
feature seti**.

### Hands-On Implementation — Layer 3: Validation-Based Selection (`zipcode`/`sale_quarter` Trade-off)

Section 17'de `zipcode`'un **69 ek parametre** eklediğini, Condition Number'ı büyüteceğini not
etmiştik. Şimdi bu trade-off'u **validation performansıyla** ölçüyoruz: `zipcode`/
`sale_quarter` dahil edilmeden vs edilerek iki basit model fit edip validation hatasını
karşılaştırıyoruz.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error


def engineer_features(frame: pd.DataFrame, grade_mean: float, age_mean: float) -> pd.DataFrame:
    # Apply the SAME deterministic feature engineering formulas used on training data.
    # grade_mean/age_mean are passed in (learned from training only) -- never recomputed here.
    out = frame.copy()
    out["sale_year"] = out["date"].dt.year
    out["sale_quarter"] = out["date"].dt.quarter
    out["house_age_at_sale"] = out["sale_year"] - out["yr_built"]
    out["basement_ratio"] = out["sqft_basement"] / out["sqft_living"]
    out["is_renovated_at_sale"] = (out["yr_renovated"] > 0).astype(int)
    out["grade_centered"] = out["grade"] - grade_mean
    out["grade_centered_sq"] = out["grade_centered"] ** 2
    out["age_centered"] = out["house_age_at_sale"] - age_mean
    out["age_centered_sq"] = out["age_centered"] ** 2
    return out


val_model_df = engineer_features(val_df, GRADE_MEAN, AGE_MEAN)

model_core = LinearRegression().fit(train_model_df[final_numeric_features], train_model_df["price"])
pred_val_core = model_core.predict(val_model_df[final_numeric_features])

print(f"Core model (no zipcode/quarter) -> Val MAE : {mean_absolute_error(val_model_df['price'], pred_val_core):,.0f}")
print(f"Core model (no zipcode/quarter) -> Val RMSE: {root_mean_squared_error(val_model_df['price'], pred_val_core):,.0f}")


In [ ]:
reference_zipcode = train_model_df["zipcode"].value_counts().idxmax()
reference_quarter = 2  # most common quarter in training data (Q2)

train_zip_dummies = pd.get_dummies(train_model_df["zipcode"], prefix="zip").astype(float).drop(columns=[f"zip_{reference_zipcode}"])
val_zip_dummies = pd.get_dummies(val_model_df["zipcode"], prefix="zip").astype(float).reindex(columns=train_zip_dummies.columns, fill_value=0)

train_q_dummies = pd.get_dummies(train_model_df["sale_quarter"], prefix="q").astype(float).drop(columns=[f"q_{reference_quarter}"])
val_q_dummies = pd.get_dummies(val_model_df["sale_quarter"], prefix="q").astype(float).reindex(columns=train_q_dummies.columns, fill_value=0)

X_train_full = pd.concat([train_model_df[final_numeric_features].reset_index(drop=True),
                           train_zip_dummies.reset_index(drop=True), train_q_dummies.reset_index(drop=True)], axis=1)
X_val_full = pd.concat([val_model_df[final_numeric_features].reset_index(drop=True),
                         val_zip_dummies.reset_index(drop=True), val_q_dummies.reset_index(drop=True)], axis=1)

model_full = LinearRegression().fit(X_train_full, train_model_df["price"].reset_index(drop=True))
pred_val_full = model_full.predict(X_val_full)

print(f"Full model (+zipcode+quarter)   -> Val MAE : {mean_absolute_error(val_model_df['price'], pred_val_full):,.0f}")
print(f"Full model (+zipcode+quarter)   -> Val RMSE: {root_mean_squared_error(val_model_df['price'], pred_val_full):,.0f}")


### Output Interpretation

`zipcode`+`sale_quarter` eklenmesi, Validation MAE'yi **≈123,358'den ≈91,710'a (%25.7 azalma)**,
RMSE'yi **≈189,476'dan ≈145,162'ye (%23.4 azalma)** düşürüyor. Bu, Section 17-18'de belirlediğimiz
**parametre sayısı/Condition Number maliyetine** değer, **büyük ve ölçülebilir** bir validation
performansı kazancıdır — `zipcode`'u modele dahil etme kararı artık yalnızca domain sezgisine
değil, **somut validation kanıtına** dayanıyor.

### Hands-On Implementation — Unknown Category Validation (CP10)


In [ ]:
unseen_zip_in_val = set(val_model_df["zipcode"]) - set(train_model_df["zipcode"])
unseen_quarter_in_val = set(val_model_df["sale_quarter"]) - set(train_model_df["sale_quarter"])

print(f"Zipcodes in validation not seen in training: {unseen_zip_in_val if unseen_zip_in_val else 'None'}")
print(f"Sale quarters in validation not seen in training: {unseen_quarter_in_val if unseen_quarter_in_val else 'None'}")

assert not unseen_zip_in_val and not unseen_quarter_in_val
print("CP10 PASSED — categorical design matrix is consistent between train and validation.")


### Hands-On Implementation — Final Design Matrix Rank Validation (CP11)


In [ ]:
final_design_matrix = X_train_full.copy()
final_design_matrix.insert(0, "const", 1.0)

final_matrix_rank = np.linalg.matrix_rank(final_design_matrix.values)
print(f"Final design matrix shape: {final_design_matrix.shape}")
print(f"Rank: {final_matrix_rank}")

assert final_matrix_rank == final_design_matrix.shape[1]
print("CP11 PASSED — final design matrix has full column rank.")


### Critical Engineering Notes — Final Feature Set Summary

| Layer | Decision |
|---|---|
| Domain-based | `id`, `date`, `sqft_above`, `sqft_basement`, `years_since_renovation`, `sqft_living15`, `sqft_lot15`, `living_area_per_bedroom`, `has_basement` excluded |
| Statistical | `grade`/`house_age_at_sale` centered before squaring (VIF ≈100 → ≈1.6); all numeric VIFs confirmed <5; full rank confirmed |
| Validation-based | `zipcode` + `sale_quarter` retained — %25.7 Val MAE improvement justifies their parameter cost |

**Final feature set (15 numeric/binary + zipcode dummies + sale_quarter dummies):**
`sqft_living`, `bedrooms`, `bathrooms`, `floors`, `grade_centered`, `grade_centered_sq`,
`condition`, `view`, `waterfront`, `age_centered`, `age_centered_sq`, `lat`, `long`,
`basement_ratio`, `is_renovated_at_sale`, `C(zipcode)`, `C(sale_quarter)`.

### Student Exercise

`test_df`'e de `engineer_features(test_df, GRADE_MEAN, AGE_MEAN)` fonksiyonunu uygulayıp, test
setinde training'de görülmemiş bir `zipcode`/`sale_quarter` olup olmadığını (CP10 tarzı) kontrol
edin. (Test setini Section 29'a kadar yalnızca bu şema doğrulaması için kullanıyoruz — tahmin
veya değerlendirme için değil.)


In [ ]:
# Student Exercise:
# TODO: test_df'e engineer_features(test_df, GRADE_MEAN, AGE_MEAN) uygulayın.
# TODO: set(test_model_df["zipcode"]) - set(train_model_df["zipcode"]) ve aynısını sale_quarter
#       için kontrol edin.

# test_model_df = engineer_features(test_df, GRADE_MEAN, AGE_MEAN)
# print(set(test_model_df["zipcode"]) - set(train_model_df["zipcode"]))
# print(set(test_model_df["sale_quarter"]) - set(train_model_df["sale_quarter"]))


### Checkpoint

- [x] Üç katmanlı feature selection (domain / statistical / validation-based) tamamlandı ve her
  katmanın kararı gerekçelendirildi.
- [x] `bedrooms=33` anomalisi Section 03'teki plana uygun olarak training fit'inden çıkarıldı
  (`train_model_df`, 15099 satır).
- [x] Centering ile polinom terimlerinin VIF'i ≈100'den ≈1.6'ya düşürüldü.
- [x] `zipcode`/`sale_quarter`'ın dahil edilmesi, %25.7 Val MAE iyileşmesiyle **somut olarak**
  gerekçelendirildi.
- [x] **CP09** — Engineered features validated (negative/missing value patterns belgelendi).
- [x] **CP10** — Categorical design matrix consistent (val'de görülmemiş kategori yok).
- [x] **CP11** — Final design matrix has full column rank.

---

## Session 4 Özeti

Bu oturumda:

- Sekiz domain-based feature türettik ve her birinin leakage riskini değerlendirdik.
- Log transformation'ın skewness'i azalttığını, ama kare terimlerin **ham korelasyonla değil
  nested OLS R² testiyle** değerlendirilmesi gerektiğini gösterdik (H2/H4 için istatistiksel
  kanıt).
- Dummy variable trap'i gerçek bir rank-deficiency ile gösterdik, doğru encoding stratejisini
  (reference category) uyguladık.
- **Zorunlu Multicollinearity bölümünde**: perfect multicollinearity (rank-deficient, VIF→∞),
  high multicollinearity (SE 2.08x enflasyon, sign flip) ve Condition Number'ın ölçek
  hassasiyetini somut sayılarla gösterdik.
- Üç katmanlı bir feature selection süreciyle, **full-column-rank**, VIF'leri kontrol altında ve
  validation performansıyla gerekçelendirilmiş bir nihai tasarım matrisine ulaştık.

Session 5'te bu tasarım matrisi üzerinde **gerçek bir OLS modeli eğitecek** ve istatistiksel
varsayımlarını sistematik olarak test edeceğiz.


---
# SESSION 5 — OLS Regression & Statistical Diagnostics
**Toplam Süre: 150 dakika (140 dakika aktif + 10 dakika ara)**


---
## Section 20 — Baseline & Initial OLS

### Learning Objective

Section 19'da oluşturduğumuz final tasarım matrisiyle **gerçek bir OLS modeli** eğitmek; bunu
LAB05 metodolojisine uygun bir baseline karşılaştırmasına tabi tutmak.

### Hands-On Implementation — Baseline (`DummyRegressor`)


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

baseline_mean = DummyRegressor(strategy="mean").fit(train_model_df[["sqft_living"]], train_model_df["price"])
baseline_median = DummyRegressor(strategy="median").fit(train_model_df[["sqft_living"]], train_model_df["price"])

for name, model_b in [("mean", baseline_mean), ("median", baseline_median)]:
    pred = model_b.predict(val_model_df[["sqft_living"]])
    mae = mean_absolute_error(val_model_df["price"], pred)
    rmse = root_mean_squared_error(val_model_df["price"], pred)
    print(f"DummyRegressor({name}) -> Val MAE: {mae:,.0f} | Val RMSE: {rmse:,.0f}")


### Hands-On Implementation — Initial OLS Model

Section 19'da belirlediğimiz final feature seti ve `C()` formula sözdizimiyle (referans
kategoriler otomatik bırakılarak) OLS modelini kuruyoruz.


In [ ]:
reference_zip = train_model_df["zipcode"].value_counts().idxmax()
reference_quarter = 2

# base_ols_terms holds the right-hand side only, so later sections (Section 25, 27) can reuse it
# to build alternative specifications without retyping the full feature list.
base_ols_terms = (
    "sqft_living + bedrooms + bathrooms + floors + grade_centered + grade_centered_sq "
    "+ condition + view + waterfront + age_centered + age_centered_sq + lat + long + basement_ratio "
    f"+ is_renovated_at_sale + C(zipcode, Treatment(reference={reference_zip})) "
    f"+ C(sale_quarter, Treatment(reference={reference_quarter}))"
)
ols_formula = "price ~ " + base_ols_terms
print(ols_formula)


In [ ]:
initial_ols_model = smf.ols(ols_formula, data=train_model_df).fit()

print(f"Training R²     : {initial_ols_model.rsquared:.4f}")
print(f"Training Adj. R²: {initial_ols_model.rsquared_adj:.4f}")
print(f"N observations  : {int(initial_ols_model.nobs)}")
print(f"N parameters    : {int(initial_ols_model.df_model) + 1}")


In [ ]:
pred_val_ols = initial_ols_model.predict(val_model_df)
val_mae_ols = mean_absolute_error(val_model_df["price"], pred_val_ols)
val_rmse_ols = root_mean_squared_error(val_model_df["price"], pred_val_ols)
val_r2_ols = 1 - np.sum((val_model_df["price"] - pred_val_ols) ** 2) / np.sum((val_model_df["price"] - val_model_df["price"].mean()) ** 2)

print(f"Initial OLS -> Val MAE : {val_mae_ols:,.0f}")
print(f"Initial OLS -> Val RMSE: {val_rmse_ols:,.0f}")
print(f"Initial OLS -> Val R²  : {val_r2_ols:.4f}")


### Output Interpretation

Baseline'lar (`DummyRegressor`), hiçbir feature kullanmadan **≈217,000-232,000 USD** MAE
üretiyor. Initial OLS modeli, Val MAE'yi **≈91,710 USD**'ye düşürüyor — baseline'a kıyasla
**%60'tan fazla** bir iyileşme. Validation R² (**≈0.820**), training R²'ye (**≈0.836**) oldukça
yakın — bu, modelin validation setine **aşırı uydurulmadığının (overfit edilmediğinin)** ilk bir
işaretidir (Session 6'da bunu daha sistemli ele alacağız).

### Hands-On Implementation — Residuals & Predictions


In [ ]:
train_residuals = initial_ols_model.resid
train_fitted = initial_ols_model.fittedvalues

print(f"Residual mean : {train_residuals.mean():.2f}  (should be ≈0 for OLS by construction)")
print(f"Residual std  : {train_residuals.std():,.2f}")
print(f"Residual range: [{train_residuals.min():,.0f}, {train_residuals.max():,.0f}]")


### Critical Engineering Notes

OLS'in matematiksel yapısı gereği, training verisi üzerindeki residual'ların ortalaması **her
zaman tam olarak sıfırdır** (bu bir model başarı göstergesi değildir — OLS'in normal
equations'ından kaynaklanan bir **matematiksel zorunluluktur**). Residual'ların **dağılım
şekli** (normal mi, çarpık mı) ise Section 23'te ayrıca test edilecek bir **varsayımdır**.

### Student Exercise

`initial_ols_model.params`'daki katsayı sayısını (`len(initial_ols_model.params)`)
`X_train_full.shape[1] + 1` (Section 19'daki final tasarım matrisi + intercept) ile
karşılaştırarak iki yaklaşımın (formula API vs manuel dummy encoding) aynı sayıda parametre
ürettiğini doğrulayın.


In [ ]:
# Student Exercise:
# TODO: len(initial_ols_model.params) değerini yazdırın.
# TODO: Section 19'daki final_design_matrix.shape[1] ile karşılaştırın.

# print(len(initial_ols_model.params), final_design_matrix.shape[1])


### Checkpoint

- [x] Baseline (`DummyRegressor` mean/median) validation performansı ölçüldü.
- [x] Initial OLS modeli, Section 19'un final feature seti üzerinde eğitildi.
- [x] Val MAE'nin baseline'a kıyasla **%60'tan fazla** iyileştiği, training/validation R²'nin
  birbirine yakın olduğu doğrulandı.
- [x] Residual ortalamasının matematiksel olarak sıfır olduğu (bir "başarı" değil, bir OLS
  özelliği olduğu) açıklandı.


---
## Section 21 — OLS Regression Summary

### Learning Objective

`statsmodels` OLS `.summary()` çıktısındaki her alanı doğru biçimde okuyup yorumlamak.

### Hands-On Implementation — Full Summary Output


In [ ]:
print(initial_ols_model.summary())


### Output Interpretation — Alan Alan Açıklama

| Alan | Anlamı |
|---|---|
| **coef** | Tahmin edilen katsayı — diğer tüm feature'lar sabit tutulduğunda, bu feature'ın `price` üzerindeki marjinal etkisi (ölçeğine bağlı birimde). |
| **std err** | Katsayı tahmininin standart hatası — tahminin ne kadar kesin olduğunu gösterir (Section 18'de gördüğümüz gibi multicollinearity bunu büyütebilir). |
| **t** | `coef / std err` — katsayının sıfırdan kaç standart hata uzakta olduğu. |
| **P>\|t\|** | İlgili t-istatistiğine karşılık gelen p-value — "katsayı gerçekte sıfırsa, bu kadar uç bir t-değeri gözlemleme olasılığı". |
| **[0.025, 0.975]** | %95 güven aralığı — katsayının "gerçek" değerinin (aynı veri üretme sürecinden tekrar örneklem alınsaydı) %95 olasılıkla içinde olacağı aralık. |
| **R-squared** | Modelin açıkladığı varyans oranı (training verisinde). |
| **Adj. R-squared** | R²'nin, modeldeki parametre sayısı için **cezalandırılmış** hâli — daha fazla feature eklemek R²'yi her zaman artırır, Adj. R² artmayabilir. |
| **F-statistic** | Tüm katsayıların (intercept hariç) **birlikte** sıfır olup olmadığını test eden genel bir F-testi. |
| **Prob (F-statistic)** | Bu F-testinin p-value'su. |
| **AIC / BIC** | Model karmaşıklığını cezalandıran bilgi kriterleri — **düşük** değer daha iyi (Section 28'de modelleri karşılaştırırken kullanacağız). |
| **Durbin-Watson** | Residual'lar arasında ardışık (sıra-bağımlı) otokorelasyon olup olmadığını ölçer — `≈2` ideal, `<1` veya `>3` sorunlu. |
| **Jarque-Bera / Prob(JB)** | Residual normalliğini test eden bir istatistik (Section 23'te ayrıntılı ele alınacak). |
| **Cond. No. (Condition Number)** | Section 18'de tanıttığımız, tasarım matrisinin sayısal kararlılığının bir ölçütü. |

### Hands-On Implementation — Extracting Key Fields Programmatically


In [ ]:
print(f"R-squared          : {initial_ols_model.rsquared:.4f}")
print(f"Adj. R-squared     : {initial_ols_model.rsquared_adj:.4f}")
print(f"F-statistic        : {initial_ols_model.fvalue:,.2f}")
print(f"Prob (F-statistic) : {initial_ols_model.f_pvalue:.4g}")
print(f"AIC                : {initial_ols_model.aic:,.2f}")
print(f"BIC                : {initial_ols_model.bic:,.2f}")


### Output Interpretation

Bu model için **R² ≈ 0.836**, **Adj. R² ≈ 0.835** (87 feature + intercept, 15099 gözlem —
R²/Adj.R² farkının küçük olması, bu kadar çok parametreye kıyasla veri miktarının yeterli
olduğunu gösteriyor). **F-statistic ≈ 882.3**, **Prob(F) ≈ 0** — model, yalnızca intercept'ten
(yani her zaman ortalama fiyatı tahmin eden bir modelden) **istatistiksel olarak anlamlı
derecede** daha iyi.

### Critical Engineering Notes — R² vs Model Validity

**Yüksek R² (≈0.836), OLS'in varsayımlarının (linearity, homoscedasticity, residual normality)
sağlandığı anlamına GELMEZ.** Bir model, verideki varyansın büyük bir kısmını açıklayabilir ve
yine de hatalı varsayımlara sahip olabilir (örn. heteroscedastic hatalar, yanlış fonksiyonel
form). Section 23'te bu varsayımları **R²'den bağımsız olarak** test edeceğiz.

### Student Exercise

`initial_ols_model.params`'daki katsayılardan, mutlak değeri en büyük 5 katsayıyı (intercept ve
zipcode dummy'leri hariç) bulup, bunların Section 11-14'teki EDA bulgularıyla tutarlı olup
olmadığını yorumlayın.


In [ ]:
# Student Exercise:
# TODO: initial_ols_model.params Series'ini filtreleyip (zipcode/sale_quarter/Intercept hariç),
#       mutlak değere göre sıralayıp en büyük 5'ini yazdırın.

# non_dummy_params = initial_ols_model.params[~initial_ols_model.params.index.str.contains("zipcode|sale_quarter|Intercept")]
# print(non_dummy_params.reindex(non_dummy_params.abs().sort_values(ascending=False).index).head(5))


### Checkpoint

- [x] `statsmodels` OLS summary'deki her alan (coef, std err, t, P>|t|, CI, R², Adj. R²,
  F-statistic, AIC/BIC, Durbin-Watson, Jarque-Bera, Condition Number) tanımlandı ve bu modelin
  gerçek değerleriyle örneklendi.
- [x] R²'nin yüksek olmasının OLS varsayımlarının sağlandığını garanti etmediği açıkça
  belirtildi (Section 23'e yönlendirme).


---
## Section 22 — Statistical Hypothesis Testing

### Learning Objective

Section 14'teki hipotezleri (H1-H6), bireysel t-testleri, joint F-testleri ve güven aralıklarıyla
**resmi olarak test etmek**; istatistiksel ve pratik anlamlılık arasındaki farkı somut
örneklerle göstermek.

### Statistical Foundations

Her katsayı için null ve alternative hipotez:

$$H_0: \beta_j = 0 \quad \text{vs.} \quad H_1: \beta_j \neq 0$$

Bir grup katsayı için **joint F-test**:

$$H_0: \beta_{j_1}=\beta_{j_2}=\dots=\beta_{j_k}=0$$

### Hands-On Implementation — Individual t-Tests & Confidence Intervals (H1, H5)


In [ ]:
confidence_intervals = initial_ols_model.conf_int(alpha=0.05)

for feature in ["sqft_living", "condition", "age_centered", "bedrooms", "waterfront"]:
    coef = initial_ols_model.params[feature]
    p_value = initial_ols_model.pvalues[feature]
    ci_low, ci_high = confidence_intervals.loc[feature]
    print(f"{feature:15s}: coef={coef:>12,.2f}  p={p_value:.4g}  95% CI=({ci_low:,.2f}, {ci_high:,.2f})")


### Output Interpretation — H1 Testi (Living Area)

`sqft_living`'in katsayısı **176.62** (p ≈ 0, %95 CI yaklaşık (170.71, 182.53)) — **H1
doğrulandı**: diğer tüm feature'lar sabit tutulduğunda, 1 ek sqft yaşam alanı, ortalama **176.62
USD** ek fiyatla ilişkilidir. Bu, Section 11'deki ham korelasyonla (≈0.712) aynı **yönde**, ama
artık diğer feature'lar kontrol edildiğinde elde edilen **partial bir etkidir**.

### Output Interpretation — H5 Testi (Confounding: `condition`)

`condition`'ın katsayısı **36,528.22** (p ≈ 5.5×10⁻⁶⁴) — **istatistiksel olarak son derece
anlamlı ve pozitif**. Bu, Section 13'te `grade=7` alt kümesinde elle gözlemlediğimiz
confounding bulgusunu **doğrulamaktadır**: `grade` (ve diğer feature'lar) kontrol edildiğinde,
`condition`'ın gerçek etkisi ortaya çıkıyor — ham korelasyonun (≈0.039) aksine.

### Critical Engineering Notes — Beklenmeyen Bir Bulgu: `age_centered`

`age_centered`'ın katsayısı (**89.19**) **istatistiksel olarak anlamlı DEĞİL** (p ≈ 0.284, %95
CI (-74.02, 252.41) — sıfırı içeriyor). Ancak `age_centered_sq`'nin katsayısı (Section 20'de
hesaplanmıştı) **anlamlıdır** (p ≈ 0.0026). Bu, **H4'ü kısmen doğruluyor**: yaşın `price` ile
ilişkisi, bu çok değişkenli modelde **doğrusal bileşen üzerinden değil, yalnızca eğrilik (kare)
bileşeni üzerinden** istatistiksel olarak anlamlıdır. Bu, Section 16'daki basit iki-değişkenli
bulgudan (hem doğrusal hem kare terim anlamlıydı) **farklı** bir sonuçtur — çünkü burada `lat`,
`long`, `zipcode` gibi konum feature'ları da modelde ve bunlar `age`'in bir kısmı etkisini
"emiyor" olabilir (bazı bölgeler hem eski hem değerli olabilir).

### Hands-On Implementation — Joint F-Test (H3: Location)


In [ ]:
zipcode_terms = [c for c in initial_ols_model.params.index if c.startswith("C(zipcode")]
zipcode_restriction = ", ".join([f"{term} = 0" for term in zipcode_terms])

zipcode_f_test = initial_ols_model.f_test(zipcode_restriction)
print(f"Joint F-test (all {len(zipcode_terms)} zipcode dummies = 0):")
print(f"  F = {zipcode_f_test.fvalue:.2f}, p = {zipcode_f_test.pvalue:.4g}")


In [ ]:
quarter_terms = [c for c in initial_ols_model.params.index if c.startswith("C(sale_quarter")]
quarter_restriction = ", ".join([f"{term} = 0" for term in quarter_terms])

quarter_f_test = initial_ols_model.f_test(quarter_restriction)
print(f"Joint F-test (all {len(quarter_terms)} sale_quarter dummies = 0):")
print(f"  F = {quarter_f_test.fvalue:.2f}, p = {quarter_f_test.pvalue:.4g}")


### Output Interpretation

**Zipcode joint F-testi**: F ≈ 142.3, p ≈ 0 — **H3 güçlü biçimde doğrulandı**: 69 zipcode
dummy'sinin **birlikte** `price`'ı açıklama gücü istatistiksel olarak son derece anlamlı.

**Sale_quarter joint F-testi**: F ≈ 1.40, p ≈ **0.240** — **anlamlı DEĞİL**. Bu önemli bir
nüanstır: Section 19'da `zipcode`+`sale_quarter`'ı **birlikte** ekleyerek %25.7 Val MAE
iyileşmesi bulmuştuk, ama bu iyileşmenin **büyük kısmı `zipcode`'dan** kaynaklanıyor —
`sale_quarter`'ın **kendi başına** katkısı istatistiksel olarak anlamlı değil. Bu, **feature
grubu bazında yapılan bir validation testinin, grup içindeki her feature'ın ayrı ayrı anlamlı
olduğu anlamına gelmediğini** gösteren somut bir örnektir.

### Critical Engineering Notes — Statistical vs Practical Significance

`waterfront`'un katsayısı (**≈649,724**, p ≈ 0) hem **istatistiksel olarak** hem **pratik
olarak** anlamlı — bu kadar büyük bir USD etkisi, gerçek dünyada da önemlidir. Ancak büyük
örneklem boyutlarında (burada n≈15,099), **çok küçük, pratik olarak önemsiz etkiler de
istatistiksel olarak anlamlı çıkabilir** (düşük p-value). Bu capstone'da her katsayıyı
yorumlarken **hem p-value'ya hem katsayının gerçek büyüklüğüne (USD cinsinden)** bakıyoruz —
yalnızca birine değil.

### Critical Engineering Notes — Multiple Testing & Post-Selection Inference

Bu modelde **87 katsayı** için ayrı ayrı t-testleri yapılıyor. Çoklu test (multiple testing)
problemi gereği, bu kadar çok testten bazılarının **yalnızca şans eseri** anlamlı çıkma
olasılığı artar (örn. %5 anlamlılık düzeyinde, 87 testten ~4-5'i "şans eseri" anlamlı
çıkabilir). Ayrıca **post-selection inference** sorunu: Section 19'da feature seçimini **aynı
training verisi üzerinde** yaptık — bu nedenle buradaki p-value'lar, feature seçimi öncesinde
**bağımsız olarak belirlenmiş bir hipotez testinden daha az kesin** kabul edilmelidir. Bu
capstone, bu sınırlamayı **gizlemiyor** — ancak 69 zipcode dummy'si gibi büyük grupların joint
F-testi, tekil t-testlerden daha **sağlam** bir kanıttır.

### Student Exercise

`grade_centered` ve `grade_centered_sq`'nin her ikisinin de anlamlı olduğunu (Section 20'deki
p-value'lara bakarak) doğrulayın. Bu ikisinin **joint F-testini** de çalıştırıp, bireysel
testlerle tutarlı olup olmadığını kontrol edin.


In [ ]:
# Student Exercise:
# TODO: initial_ols_model.f_test("grade_centered = 0, grade_centered_sq = 0") çalıştırın.
# TODO: Sonucu, Section 20'deki bireysel p-value'larla karşılaştırın.

# grade_joint_test = initial_ols_model.f_test("grade_centered = 0, grade_centered_sq = 0")
# print(grade_joint_test.fvalue, grade_joint_test.pvalue)


### Checkpoint

- [x] H1 (sqft_living) ve H5 (condition confounding) bireysel t-testlerle doğrulandı.
- [x] H3 (location), 69 zipcode dummy'sinin joint F-testiyle **güçlü biçimde** doğrulandı
  (F≈142.3, p≈0).
- [x] `sale_quarter`'ın joint olarak anlamlı olmadığı (p≈0.240) — grup bazlı validation
  iyileşmesinin tek bir feature'ın anlamlılığını garanti etmediği — açıkça gösterildi.
- [x] Statistical vs Practical Significance ayrımı `waterfront` örneğiyle somutlaştırıldı.
- [x] Multiple testing ve post-selection inference sınırlamaları açıkça kabul edildi.


---
## Section 23 — OLS Assumptions & Residual Diagnostics

### Learning Objective

OLS'in temel varsayımlarını (linearity, homoscedasticity, residual normality, error
independence, correct specification) sistematik testlerle değerlendirmek; bir testin anlamlı
sonuç vermesinin "notebook hatası" değil, **gerçek bir model sınırlaması bulgusu** olduğunu
kabul etmek.

### Concept Explanation

- **Linearity**: `price`'ın, feature'ların doğrusal bir kombinasyonu olarak modellenebilmesi.
- **Homoscedasticity**: Hata teriminin varyansının, feature değerlerinden **bağımsız** olması.
- **Residual Normality**: Hata teriminin normal dağılması (büyük örneklemlerde katsayı
  tahminleri için zorunlu değildir, ama küçük-örneklem güven aralıkları için önemlidir).
- **Error Dependence**: Hataların birbirinden **bağımsız** olması (zaman serisi verilerinde
  ardışık korelasyon riski).
- **Conditional Exogeneity**: Feature'ların hata terimiyle **ilişkisiz** olması — **bu
  varsayım yalnızca residual plot'larla doğrulanamaz**, çünkü ölçülmeyen, feature'larla
  korele bir karıştırıcı (confounder) residual'larda **görünmeyebilir**.

### Hands-On Implementation — Residuals vs Fitted


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(train_fitted, train_residuals, color=BAR_COLOR, alpha=0.2, s=10)
ax.axhline(0, color=WARNING_COLOR, linewidth=2, linestyle="--")
ax.set_title("Residuals vs Fitted Values")
ax.set_xlabel("Fitted price")
ax.set_ylabel("Residual")
style_axes(ax)
plt.show()


### Output Interpretation

Residual'ların yayılımı, fitted değer arttıkça **görünür biçimde genişliyor** (huni/koni
şekli) — bu, **homoscedasticity varsayımının ihlal edildiğine** dair görsel bir işarettir
(Section 12'de `grade`'e göre fiyat yayılımının büyüdüğünü gördüğümüz bulgunun bir devamı).
Aşağıda bunu formal bir testle (Breusch-Pagan) doğrulayacağız.

### Hands-On Implementation — Residuals vs Key Predictor


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(train_model_df["sqft_living"], train_residuals, color=BAR_COLOR, alpha=0.2, s=10)
ax.axhline(0, color=WARNING_COLOR, linewidth=2, linestyle="--")
ax.set_title("Residuals vs sqft_living")
ax.set_xlabel("sqft_living")
ax.set_ylabel("Residual")
style_axes(ax)
plt.show()


### Output Interpretation

Benzer bir huni örüntüsü `sqft_living` için de görünüyor — büyük evlerde hata varyansı daha
yüksek. Bu, "büyük/pahalı evlerin fiyatını tahmin etmek doğası gereği daha belirsizdir"
şeklinde yorumlanabilir.

### Hands-On Implementation — Scale-Location Plot


In [ ]:
sqrt_abs_std_resid = np.sqrt(np.abs(initial_ols_model.get_influence().resid_studentized_internal))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(train_fitted, sqrt_abs_std_resid, color=BAR_COLOR, alpha=0.2, s=10)
ax.set_title("Scale-Location Plot")
ax.set_xlabel("Fitted price")
ax.set_ylabel("√|Standardized Residual|")
style_axes(ax)
plt.show()


### Output Interpretation

Scale-Location plot'ta düz, yatay bir eğilim **homoscedasticity**'ye işaret eder. Burada
eğilim **yükselen** — Residuals vs Fitted'daki huni örüntüsünü **doğruluyor**.

### Hands-On Implementation — Q-Q Plot (Residual Normality)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
qqplot(train_residuals, line="s", ax=ax, markerfacecolor=BAR_COLOR, markeredgecolor=BAR_COLOR, alpha=0.3)
ax.set_title("Q-Q Plot of Residuals")
style_axes(ax)
plt.show()


### Output Interpretation

Noktalar orta kısımda referans çizgisine yakın, ama **kuyruklarda belirgin biçimde ayrılıyor**
(özellikle sağ kuyrukta) — residual'lar **tam normal değil**, sağa çarpık ve ağır kuyruklu
görünüyor. Bu, Section 06'da `price`'ın kendisinin çok çarpık olduğunu bulduğumuz gerçeğiyle
tutarlı — raw-price OLS'in residual'ları da bu çarpıklığı kısmen taşıyor.

### Hands-On Implementation — Formal Tests: Breusch-Pagan & Jarque-Bera


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera, durbin_watson

bp_stat, bp_pvalue, bp_fvalue, bp_f_pvalue = het_breuschpagan(train_residuals, initial_ols_model.model.exog)
print(f"Breusch-Pagan LM statistic: {bp_stat:,.2f}, p-value: {bp_pvalue:.4g}")


### Output Interpretation — Breusch-Pagan Test

**H₀: Hata varyansı homoscedastic (feature'lardan bağımsız)**. LM istatistiği **≈2485.8**,
p-value **≈ 0** — H₀ **güçlü biçimde reddediliyor**. Bu, görsel bulguyu (huni şekli) **formal
olarak doğruluyor**: bu model **heteroscedastic hatalar üretiyor** — bu, Section 14'teki **H6
hipotezini doğrulayan ilk formal istatistiksel kanıttır**.


In [ ]:
jb_stat, jb_pvalue, jb_skew, jb_kurtosis = jarque_bera(train_residuals)
print(f"Jarque-Bera statistic: {jb_stat:,.2f}, p-value: {jb_pvalue:.4g}")
print(f"Residual skewness: {jb_skew:.3f}, kurtosis: {jb_kurtosis:.3f}")


### Output Interpretation — Jarque-Bera Test

**H₀: Residual'lar normal dağılıyor**. JB istatistiği **≈3,226,408**, p-value **≈ 0** — H₀
**güçlü biçimde reddediliyor**. Residual skewness (**≈3.76**) ve kurtosis (**≈74.2**) aşırı
yüksek — residual'lar **normal dağılımdan çok uzak**, hâlâ önemli bir sağ-çarpıklık taşıyor.

### Critical Engineering Notes — Target Normality vs Residual Normality (Yeniden)

Section 06'da `price`'ın kendi dağılımının normalliğinin **OLS'in bir ön koşulu olmadığını**
belirtmiştik — doğrudur. Ama burada test ettiğimiz **residual normality**, OLS'in **küçük
örneklemde geçerli t-testleri/güven aralıkları için varsayılan** gerçek bir koşuludur. Bu
testin reddedilmesi, **büyük örneklem boyutumuzda (n≈15,099) Merkezi Limit Teoremi sayesinde**
katsayı tahminlerinin dağılımının yine de yaklaşık normale yakın olmasını **sağlayabilir**, ama
bu **garanti edilmiş değildir** — bu bir sınırlama olarak not edilir.

### Hands-On Implementation — Durbin-Watson (Error Dependence)


In [ ]:
dw_statistic = durbin_watson(train_residuals)
print(f"Durbin-Watson statistic: {dw_statistic:.4f}")


### Output Interpretation — Durbin-Watson

Durbin-Watson istatistiği **≈1.974** — `2`'ye çok yakın, bu da **ardışık otokorelasyonun
ciddi bir sorun olmadığına** işaret ediyor. Bu şaşırtıcı olmayabilir: veri kronolojik olarak
sıralanmış olsa da, model `price`'ı zaman bazlı bir trend olarak değil, konut özellikleriyle
modelliyor — ardışık satışlar arasında güçlü bir seri-bağımlılık beklenmiyordu.

### Hands-On Implementation — Ramsey RESET Test (Model Specification)


In [ ]:
import warnings
from statsmodels.stats.diagnostic import linear_reset

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    reset_result = linear_reset(initial_ols_model, power=2, use_f=True)
    for w in caught:
        print(f"WARNING: {w.category.__name__}: {str(w.message)[:150]}")

print(f"\nRamsey RESET F-statistic: {reset_result.fvalue:.2f}, p-value: {reset_result.pvalue:.4g}")


### Critical Engineering Notes — A Numerical Stability Caveat (Scale Sensitivity, Again)

Dikkat edin: yukarıdaki hücre bir **"design matrix is rank-deficient"** uyarısı veriyor —
ancak bu, Section 18'de gördüğümüz **Condition Number'ın ölçeğe duyarlılığı** sorununun
**aynısıdır**, gerçek bir matematiksel rank problemi değildir. `linear_reset()`, test
regresyonuna `fitted_değer²` sütununu ekler; `fitted_değer` burada **yüzbinlerce USD**
mertebesinde olduğu için, karesi (**~10¹¹** mertebesinde) diğer sütunlardan (çoğu `0`-`1` veya
küçük sayılar) **astronomik derecede büyük ölçekte** — bu, `numpy`'ın SVD-tabanlı rank
hesaplamasının tolerans eşiğini yanıltıyor. Bunu, `fitted_değer²`'yi standardize ederek manuel
olarak doğrulayalım.


In [ ]:
import statsmodels.api as sm

fitted_sq_scaled = (train_fitted ** 2) / (train_fitted ** 2).std()
augmented_exog = np.column_stack([initial_ols_model.model.exog, fitted_sq_scaled])

print(f"Rank of augmented matrix (scaled fitted²): {np.linalg.matrix_rank(augmented_exog)} / {augmented_exog.shape[1]}")

augmented_model = sm.OLS(initial_ols_model.model.endog, augmented_exog).fit()
restricted_rss = np.sum(train_residuals ** 2)
unrestricted_rss = np.sum(augmented_model.resid ** 2)
f_stat_manual = ((restricted_rss - unrestricted_rss) / 1) / (unrestricted_rss / augmented_model.df_resid)

print(f"Manually re-verified RESET F-statistic: {f_stat_manual:.2f}")
print(f"(Original statsmodels result despite the warning: F = {reset_result.fvalue:.2f} — consistent, not a computation error.)")


### Output Interpretation — Ramsey RESET Test

Manuel olarak yeniden doğrulanan F-istatistiği (**≈5894**) statsmodels'in orijinal sonucuyla
(**≈5930.6**) **tutarlı** — yani uyarı, test sonucunu **geçersiz kılmıyor**, yalnızca
`numpy`'ın rank kontrolünün ölçek-hassasiyetine dair bir **uyarı gürültüsü**. Her iki hesaplama
da aynı sonuca işaret ediyor: **H₀ (model doğru şekilde specifiye edilmiş) güçlü biçimde
reddediliyor** (p ≈ 0). Bu, modelimizin **henüz yakalamadığı bazı nonlinear örüntüler**
olduğunu gösteriyor — Session 6'da (Section 27) alternatif specification'lar (interaction
terimleri, log-price modeli) bu bulguyu ele alacaktır.

### Critical Engineering Notes — Exogeneity Cannot Be Verified by Residual Plots Alone

Yukarıdaki tüm testler (Breusch-Pagan, Jarque-Bera, Durbin-Watson, RESET), **gözlenebilir**
varsayım ihlallerini tespit eder. Ancak **conditional exogeneity** (feature'ların hata
terimiyle ilişkisiz olması) varsayımı, bu testlerin **hiçbiriyle doğrulanamaz** — çünkü bu
varsayımın ihlali, tanım olarak **gözlenmeyen (omitted) bir değişkenden** kaynaklanır. Örneğin,
"mahalle prestiji" gibi veri kümesinde bulunmayan bir faktör, hem `grade` hem `price` ile
korele olabilir ve bu, `grade`'in katsayısını sistematik olarak çarpıtabilir — residual
plot'ları bunu **göstermez**. Bu, OLS'in **her zaman kabul etmemiz gereken** bir sınırlamadır.

### Student Exercise

`train_residuals`'ı `zipcode`'a göre gruplandırıp, hangi zipcode'ların en yüksek **ortalama
mutlak residual**'a sahip olduğunu bulun. Bu, Section 25'te (Influence Diagnostics) ayrıca
incelenecek bir ön sinyal olabilir.


In [ ]:
# Student Exercise:
# TODO: train_model_df["residual"] = train_residuals atayın.
# TODO: train_model_df.groupby("zipcode")["residual"].apply(lambda s: s.abs().mean()) ile
#       her zipcode için ortalama mutlak residual'ı hesaplayıp en yüksek 5'ini bulun.

# train_model_df["residual"] = train_residuals.values
# print(train_model_df.groupby("zipcode")["residual"].apply(lambda s: s.abs().mean()).sort_values(ascending=False).head(5))


### Checkpoint

- [x] Residuals vs Fitted / vs `sqft_living` / Scale-Location plot'ları, **heteroscedasticity**
  için görsel kanıt sağladı; **Breusch-Pagan testi** (p≈0) bunu formal olarak doğruladı (H6
  için istatistiksel kanıt).
- [x] Q-Q plot ve **Jarque-Bera testi** (p≈0), residual'ların normal olmadığını (skew≈3.76,
  kurtosis≈74.2) doğruladı — bu, target normality ile residual normality arasındaki farkla
  birlikte yorumlandı.
- [x] **Durbin-Watson** (≈1.97), ciddi bir ardışık otokorelasyon olmadığını gösterdi.
- [x] **Ramsey RESET testi** (p≈0), model specification'ında eksiklikler olduğuna işaret etti
  — bir numerik ölçek-hassasiyeti uyarısı araştırıldı ve test sonucunun geçerli olduğu manuel
  olarak doğrulandı.
- [x] **Conditional exogeneity**'nin residual plot'larla doğrulanamayacağı açıkça kabul edildi.


---
## Section 24 — Statistical Findings

### Learning Objective

Session 5'in tüm istatistiksel bulgularını, Session 6'daki model iyileştirme çalışmasına
doğrudan girdi sağlayacak bir **OLS Diagnostic Report**'ta birleştirmek.

### Hands-On Implementation — OLS Diagnostic Report


In [ ]:
ols_diagnostic_report = pd.DataFrame([
    {"Check": "R² / Adj. R²", "Result": f"{initial_ols_model.rsquared:.3f} / {initial_ols_model.rsquared_adj:.3f}", "Assumption Status": "N/A (fit quality, not an assumption)"},
    {"Check": "Overall F-test", "Result": f"F={initial_ols_model.fvalue:,.1f}, p≈0", "Assumption Status": "Model jointly significant"},
    {"Check": "Breusch-Pagan (homoscedasticity)", "Result": f"LM={bp_stat:,.1f}, p≈0", "Assumption Status": "VIOLATED — heteroscedastic errors"},
    {"Check": "Jarque-Bera (residual normality)", "Result": f"JB={jb_stat:,.0f}, p≈0", "Assumption Status": "VIOLATED — non-normal, right-skewed residuals"},
    {"Check": "Durbin-Watson (error independence)", "Result": f"DW={dw_statistic:.3f}", "Assumption Status": "OK — close to 2, no strong autocorrelation"},
    {"Check": "Ramsey RESET (specification)", "Result": f"F={reset_result.fvalue:.0f}, p≈0", "Assumption Status": "VIOLATED — functional form likely incomplete"},
    {"Check": "VIF (multicollinearity, Section 18-19)", "Result": "All numeric features < 5 after centering", "Assumption Status": "OK — after remediation"},
    {"Check": "Conditional exogeneity", "Result": "Not directly testable", "Assumption Status": "UNVERIFIED — inherent OLS limitation"},
])
ols_diagnostic_report


### Output Interpretation

Bu modelin istatistiksel profili: **güçlü genel açıklayıcılık** (R²≈0.836, F-testi son derece
anlamlı) ve **çözülmüş multicollinearity** (Section 18-19'daki çalışma sayesinde), ama **üç
varsayım ihlali**: heteroscedasticity, residual non-normality ve olası specification
eksikliği. Bu, "model işe yaramıyor" anlamına **gelmez** — Validation MAE/RMSE'nin
(Section 20) baseline'dan çok daha iyi olduğunu zaten gördük. Ama bu, **standart hataların ve
p-value'ların klasik OLS varsayımları altında hesaplandığı ölçüde tam güvenilir olmayabileceği**
anlamına gelir — bu konuyu Section 26'da (Robust Standard Errors) ele alacağız.

### Critical Engineering Notes — Bu Bulgular Session 6'yı Nasıl Yönlendiriyor?

| Bulgu | Session 6'daki Karşılık |
|---|---|
| Heteroscedasticity (Breusch-Pagan) | Section 26 — HC3 Robust Standard Errors |
| RESET testi (specification) | Section 27 — Interaction Model, Polynomial Model, Log-Target OLS |
| Residual non-normality + sağ çarpıklık | Section 27 — Log-Target OLS (Section 06'da log-price'ın normale daha yakın olduğunu bulmuştuk) |
| Yüksek leverage/influence olasılığı (Section 03'teki flag'li kayıtlar) | Section 25 — Influence Diagnostics |

### Student Exercise

Bu dört varsayım ihlalinden (heteroscedasticity, non-normality, RESET, exogeneity-unverified)
hangisinin, sizce, **validation/test performansı** için en riskli olduğunu (yalnızca
istatistiksel çıkarım için değil) 2-3 cümlede tartışın.


In [ ]:
# Student Exercise:
# TODO: Yukarıdaki dört bulgudan birini seçip, bunun yalnızca "p-value'ların güvenilirliği"
#       değil, "gerçek tahmin performansı" üzerindeki olası etkisini kendi cümlelerinizle yazın.


### Checkpoint

- [x] Session 5'in tüm istatistiksel bulguları tek bir **OLS Diagnostic Report**'ta
  toplandı.
- [x] Her varsayım ihlalinin Session 6'daki hangi spesifik bölümde ele alınacağı açıkça
  haritalandı.

---

## Session 5 Özeti

Bu oturumda:

- Baseline'lara (`DummyRegressor`) kıyasla **%60'tan fazla** iyileşme gösteren bir **Initial OLS
  modeli** eğittik (Val R²≈0.820, Val MAE≈91,710 USD).
- `statsmodels` OLS `.summary()`'deki her alanı (coef, std err, t, p, CI, R², F-test, AIC/BIC,
  Durbin-Watson, Jarque-Bera, Condition Number) gerçek değerlerle yorumladık.
- H1 (`sqft_living`) ve H5 (`condition` confounding) bireysel t-testlerle, H3 (`zipcode`) joint
  F-testiyle **doğrulandı**; `sale_quarter`'ın joint olarak anlamlı olmadığı bulundu —
  Statistical vs Practical Significance ayrımını `waterfront` örneğiyle somutlaştırdık.
- **Dört formal test** (Breusch-Pagan, Jarque-Bera, Durbin-Watson, Ramsey RESET) ile OLS
  varsayımlarını sistematik olarak değerlendirdik; üçünün (homoscedasticity, normality,
  specification) ihlal edildiğini **dürüstçe** raporladık — ve bir numerik ölçek-hassasiyeti
  uyarısını (RESET testinde) gerçek bir sorunla karıştırmadan doğru biçimde araştırdık.
- Tüm bulguları, Session 6'daki model iyileştirme çalışmasına doğrudan bağlanan bir **OLS
  Diagnostic Report**'ta birleştirdik.

Session 6'da bu bulgulara **doğrudan yanıt verecek**: Influence Diagnostics, Robust Standard
Errors, alternatif OLS specification'ları (interaction, polynomial, log-target) ve nihai model
seçimi/test değerlendirmesi.


---
# SESSION 6 — Model Refinement & Final Evaluation
**Toplam Süre: 150 dakika (140 dakika aktif + 10 dakika ara)**

Bu, capstone'un son oturumudur. Session 5'in bulgularına **doğrudan yanıt vereceğiz**.


---
## Section 25 — Influence Diagnostics

### Learning Objective

**Outlier**, **leverage** ve **influence** kavramlarını ayrı ayrı tanımlamak; Cook's Distance
ile en etkili gözlemi tespit etmek; bu gözlemin çıkarılmasının katsayılar üzerindeki etkisini
ölçen bir **sensitivity analysis** yapmak.

### Concept Explanation

- **Outlier**: Büyük bir **residual**'a sahip gözlem (model, bu noktayı kötü tahmin ediyor).
- **Leverage**: Bir gözlemin **feature uzayında** ne kadar "uç" (diğer gözlemlerden farklı)
  olduğu — `price`'a bakılmadan, yalnızca feature değerlerine dayanır.
- **Influence**: Bir gözlemin, **modelin katsayılarını** ne kadar değiştirdiği — hem outlier'lık
  hem leverage'ın **birleşimidir** (Cook's Distance bu ikisini birleştirir).

Bu üçü **farklı** kavramlardır: yüksek leverage'lı bir nokta, eğer residual'ı küçükse düşük
influence'a sahip olabilir; yüksek residual'lı bir nokta, eğer leverage'ı düşükse de düşük
influence'a sahip olabilir.

### Hands-On Implementation — Standardized & Studentized Residuals, Leverage


In [ ]:
influence = initial_ols_model.get_influence()
cooks_distance = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag
standardized_residuals = influence.resid_studentized_internal

print("Standardized residuals — describe:")
print(pd.Series(standardized_residuals).describe())


In [ ]:
n_high_std_resid = (np.abs(standardized_residuals) > 3).sum()
print(f"Observations with |standardized residual| > 3: {n_high_std_resid} ({n_high_std_resid/len(train_model_df):.2%})")


### Hands-On Implementation — Leverage & Cook's Distance


In [ ]:
n_params = int(initial_ols_model.df_model) + 1
leverage_threshold = 2 * n_params / len(train_model_df)
cooks_threshold = 4 / len(train_model_df)

n_high_leverage = (leverage > leverage_threshold).sum()
n_high_cooks = (cooks_distance > cooks_threshold).sum()

print(f"Leverage threshold (2p/n = {leverage_threshold:.5f}): {n_high_leverage} observations exceed it")
print(f"Cook's Distance threshold (4/n = {cooks_threshold:.6f}): {n_high_cooks} observations exceed it ({n_high_cooks/len(train_model_df):.2%})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(leverage, standardized_residuals, color=BAR_COLOR, alpha=0.2, s=10)
axes[0].axvline(leverage_threshold, color=WARNING_COLOR, linestyle="--", linewidth=1.5, label="Leverage threshold")
axes[0].set_title("Residuals vs Leverage")
axes[0].set_xlabel("Leverage")
axes[0].set_ylabel("Standardized Residual")
axes[0].legend(frameon=False, fontsize=8)
style_axes(axes[0])

axes[1].scatter(range(len(cooks_distance)), cooks_distance, color=BAR_COLOR, alpha=0.3, s=10)
axes[1].axhline(cooks_threshold, color=WARNING_COLOR, linestyle="--", linewidth=1.5, label="4/n threshold")
axes[1].set_title("Cook's Distance by Observation")
axes[1].set_xlabel("Observation index (chronological)")
axes[1].set_ylabel("Cook's Distance")
axes[1].legend(frameon=False, fontsize=8)
style_axes(axes[1])

plt.tight_layout()
plt.show()


### Output Interpretation

**715 gözlem (%4.74)** Cook's Distance eşiğini (`4/n`) aşıyor — bu oranın belirli bir eşikten
büyük olması beklenir (bu eşik, kesin bir "anomali" sınırı değil, yalnızca **göreceli** bir
tarama kuralıdır). Daha önemlisi: **tek bir gözlem**, Cook's Distance'ta diğerlerinden **çok
daha yüksek** bir değere sahip.

### Hands-On Implementation — The Single Most Influential Observation


In [ ]:
most_influential_idx = np.argmax(cooks_distance)
most_influential_row = train_model_df.loc[most_influential_idx]

print(f"Cook's Distance: {cooks_distance[most_influential_idx]:.4f}  (threshold: {cooks_threshold:.6f})")
print(f"Leverage        : {leverage[most_influential_idx]:.4f}")
print(f"Standardized residual: {standardized_residuals[most_influential_idx]:.2f}")
print()
print(most_influential_row[["id", "price", "sqft_living", "bedrooms", "bathrooms", "grade", "zipcode"]])
print(f"\nModel's fitted value for this observation: {initial_ols_model.fittedvalues[most_influential_idx]:,.0f}")


### Output Interpretation

Bu, Section 03/06'da zaten tanıdığımız bir kayıt: dataset'teki **en yüksek fiyatlı satış**
(**7,700,000 USD**, `grade=13` — maksimum grade, `sqft_living=12,050`). Model, bu konut için
yalnızca **≈3,816,098 USD** tahmin ediyor — **≈3.88 milyon USD** eksik tahmin. Bu hem çok yüksek
bir residual'a (standardized residual ≈26.3 — inanılmaz uç bir değer) hem de orta-yüksek bir
leverage'a sahip, bu da onu Cook's Distance'ta **tek başına baskın** kılıyor.

### Hands-On Implementation — Sensitivity Analysis: Refit Without This Point


In [ ]:
most_influential_id = int(most_influential_row["id"])
train_without_point = train_model_df[train_model_df["id"] != most_influential_id]

model_without_point = smf.ols("price ~ " + base_ols_terms, data=train_without_point).fit()

print("Coefficient comparison (with vs without the most influential point):")
for feature in ["sqft_living", "grade_centered", "grade_centered_sq", "condition", "waterfront"]:
    coef_with = initial_ols_model.params[feature]
    coef_without = model_without_point.params[feature]
    pct_change = (coef_without - coef_with) / abs(coef_with) * 100
    print(f"  {feature:18s}: with={coef_with:>12,.2f}  without={coef_without:>12,.2f}  change={pct_change:+.2f}%")

print()
print(f"R² with point: {initial_ols_model.rsquared:.4f}  |  R² without point: {model_without_point.rsquared:.4f}")


### Critical Engineering Notes — Sensitivity, Not Deletion

Katsayılar bu tek noktanın çıkarılmasıyla **%1.3 ile %6.4 arasında** değişiyor — bu **orta
düzeyde bir hassasiyettir** (modelin tamamen bu bir noktaya bağlı olmadığını, ama onun gözardı
edilebilir olmadığını gösterir). **Bu kaydı silmiyoruz** — bu gerçek, geçerli bir satıştır (bir
veri hatası olduğuna dair kanıt yok), yalnızca dataset'in **lüks segment** uç noktasını temsil
ediyor. Bu bulgu, Section 27'de **"Refined OLS"** adını vereceğimiz bir alternatif
specification'ın motivasyonu olacaktır — oradaki karşılaştırma, bu noktayı training'den
çıkarmanın validation performansını nasıl etkiledigini **ölçecektir** (nihai karar orada
verilecektir, burada yalnızca hassasiyeti ölçüyoruz).

### Student Exercise

Section 03'te işaretlediğimiz `yr_built > sale_year` anomalisine sahip 12 kayıttan birinin
Cook's Distance değerini bulup, bunun dataset ortalamasından (yukarıdaki `.describe()`
çıktısı) ne kadar farklı olduğunu yorumlayın.


In [ ]:
# Student Exercise:
# TODO: train_model_df["age_centered"] + AGE_MEAN < 0 olan (yr_built > sale_year anomalisine
#       sahip) satırların index'lerini bulup, bu index'lere karşılık gelen cooks_distance
#       değerlerini yazdırın.

# anomalous_age_mask = (train_model_df["house_age_at_sale"] < 0).values
# print(pd.Series(cooks_distance[anomalous_age_mask]).describe())


### Checkpoint

- [x] Outlier, leverage ve influence kavramları ayrı ayrı tanımlandı ve bu modelde somut
  sayılarla örneklendi.
- [x] En yüksek Cook's Distance'a sahip gözlem (dataset'teki en yüksek fiyatlı satış, 7.7M USD)
  tespit edildi.
- [x] Bu noktanın çıkarılmasının katsayılarda %1-6 aralığında değişime yol açtığı — orta
  düzeyde bir hassasiyet — ölçüldü.
- [x] Bu kaydın **silinmediği**, yalnızca Section 27'deki bir sensitivity specification'ına
  girdi olarak kullanılacağı açıkça belirtildi.


---
## Section 26 — Robust Standard Errors

### Learning Objective

Section 23'te bulduğumuz heteroscedasticity'ye bir **yanıt** olarak HC3 robust standard
error'ları uygulamak; bunun katsayıları **değiştirmediğini**, yalnızca **inference'ı**
(standart hata, güven aralığı, p-value) değiştirdiğini somut olarak göstermek.

### Statistical Foundations

Klasik OLS standart hataları, homoscedasticity varsayımı altında türetilir:
$Var(\hat\beta) = \sigma^2(X^TX)^{-1}$. Heteroscedasticity varsa, bu formül **yanlış** standart
hatalar üretir. **HC3** (White'ın heteroscedasticity-consistent estimator'ının bir
varyasyonu), $\sigma^2$'yi her gözlem için **farklı** tahmin ederek bu sorunu düzeltir —
katsayı tahminlerini **değiştirmez**, yalnızca onların **varyans tahminini** düzeltir.

### Hands-On Implementation — Classical vs HC3


In [ ]:
robust_ols_model = smf.ols(ols_formula, data=train_model_df).fit(cov_type="HC3")

coefficients_identical = np.allclose(initial_ols_model.params.values, robust_ols_model.params.values)
print(f"Coefficients identical between classical and HC3 models: {coefficients_identical}")


In [ ]:
se_comparison = pd.DataFrame({
    "coef": initial_ols_model.params,
    "SE_classical": initial_ols_model.bse,
    "SE_HC3": robust_ols_model.bse,
    "p_classical": initial_ols_model.pvalues,
    "p_HC3": robust_ols_model.pvalues,
})
se_comparison["SE_ratio_HC3_over_classical"] = se_comparison["SE_HC3"] / se_comparison["SE_classical"]

key_features = ["sqft_living", "grade_centered", "grade_centered_sq", "condition", "bedrooms", "age_centered", "waterfront"]
se_comparison.loc[key_features].round(4)


### Output Interpretation

Katsayılar **tamamen aynı** (`coefficients_identical = True`) — HC3, OLS'in **nokta
tahminlerini değiştirmez**. Ancak standart hatalar **çarpıcı biçimde büyüyor**:
`waterfront`'un SE'si **≈4.25x**, `sqft_living`'inki **≈3.47x**, `grade_centered`'inki **≈2.37x**
artıyor — bu, Section 23'teki Breusch-Pagan bulgusuyla (heteroscedasticity tespit edilmişti)
**tutarlıdır**: klasik standart hatalar, bu veri için **gerçekte olduğundan daha küçük (aşırı
iyimser)** görünüyordu.

### Hands-On Implementation — Does Any Conclusion Change?


In [ ]:
for feature in key_features:
    sig_classical = se_comparison.loc[feature, "p_classical"] < 0.05
    sig_hc3 = se_comparison.loc[feature, "p_HC3"] < 0.05
    status = "SAME conclusion" if sig_classical == sig_hc3 else "CONCLUSION CHANGES"
    print(f"{feature:18s}: significant (classical)={sig_classical}  significant (HC3)={sig_hc3}  -> {status}")


### Output Interpretation

Standart hatalar önemli ölçüde büyümesine rağmen, incelediğimiz **hiçbir** feature için
anlamlılık sonucu **değişmiyor** (`age_centered` zaten her iki yöntemde de anlamsız,
diğerleri her iki yöntemde de anlamlı kalıyor) — bu, bulgularımızın heteroscedasticity'ye karşı
**nispeten kararlı (robust)** olduğunu gösteriyor. Ancak bu **her zaman böyle olacağı
anlamına gelmez** — daha marjinal (p-value'su 0.05'e yakın) bir katsayı için HC3, farklı bir
sonuca yol açabilirdi.

### Critical Engineering Notes — What HC3 Does NOT Fix

**HC3, aşağıdaki sorunları ÇÖZMEZ:**

- **Heteroscedasticity'nin kendisi** hâlâ mevcuttur — HC3 yalnızca bunun **inference** üzerindeki
  etkisini düzeltir, hata varyansının feature'lara bağlı değişmesini **ortadan kaldırmaz**.
- **Endogeneity / Omitted-Variable Bias**: Eğer modelde olmayan bir değişken (örn. "mahalle
  prestiji") hem bir feature'la hem `price`'la korele ise, HC3 bu **yanlılığı (bias) düzeltmez**
  — yalnızca doğru varyansla **yanlış** bir katsayıyı daha "güvenilir" göstermiş olabilir. Bu,
  Section 23'teki **conditional exogeneity** sınırlamasının burada da geçerli olduğu
  anlamına gelir.
- **Model Specification Hataları**: Section 23'teki RESET testinin işaret ettiği nonlinearity
  eksiklikleri, HC3 ile **çözülmez** — bunun için Section 27'deki alternatif specification'lara
  ihtiyaç var.

### Student Exercise

`age_centered`'ın klasik p-value'sunu (≈0.284) HC3 p-value'suyla (yukarıdaki tabloda) karşılaştırın.
Her iki durumda da bu feature'ın anlamlı olmadığı sonucuna varıyor musunuz?


In [ ]:
# Student Exercise:
# TODO: se_comparison.loc["age_centered"] satırını yazdırıp p_classical ve p_HC3 değerlerini
#       karşılaştırın.

# print(se_comparison.loc["age_centered"])


### Checkpoint

- [x] HC3 katsayıları **değiştirmediği**, yalnızca standart hataları değiştirdiği doğrulandı
  (`np.allclose` ile kanıtlandı).
- [x] Standart hata enflasyonunun (1.0x-4.25x) Section 23'teki Breusch-Pagan bulgusuyla tutarlı
  olduğu gösterildi.
- [x] İncelenen feature'lar için anlamlılık sonucunun değişmediği doğrulandı — ama bunun genel
  bir garanti olmadığı not edildi.
- [x] HC3'ün endogeneity/omitted-variable bias veya model specification hatalarını **çözmediği**
  açıkça belirtildi.


---
## Section 27 — Alternative OLS Specifications

### Learning Objective

Session 5'teki bulgulara (influence, RESET testi, residual çarpıklığı) **doğrudan yanıt veren**
beş alternatif OLS specification'ı kurmak ve bunları Initial OLS ile **validation verisinde**
karşılaştırmak.

### Hands-On Implementation — Specification 1: Refined OLS (Influence-Based)

Section 25'te en etkili gözlemi (7.7M USD'lik satış) tespit etmiştik. Bu specification, o
gözlemi **training fit'inden** çıkarır (validation seti dokunulmaz).


In [ ]:
refined_ols_model = smf.ols(ols_formula, data=train_without_point).fit()
pred_val_refined = refined_ols_model.predict(val_model_df)

print(f"Refined OLS -> Val MAE: {mean_absolute_error(val_model_df['price'], pred_val_refined):,.0f}")
print(f"Refined OLS -> Val RMSE: {root_mean_squared_error(val_model_df['price'], pred_val_refined):,.0f}")


### Hands-On Implementation — Specification 2: Feature Engineered OLS (`sqft_living15` Sensitivity Check)

Section 01/05/19'da `sqft_living15`'in prediction-time erişilebilirliğinin **doğrulanmamış**
olduğu için dışarıda tutulduğunu belirtmiştik. Şimdi bunu **ekleyip** gerçekten bir fark
yaratıp yaratmadığını test ediyoruz.


In [ ]:
feature_engineered_formula = ols_formula + " + sqft_living15"
feature_engineered_model = smf.ols(feature_engineered_formula, data=train_model_df).fit()
pred_val_fe = feature_engineered_model.predict(val_model_df)

print(f"Feature Engineered OLS (+sqft_living15) -> Val MAE: {mean_absolute_error(val_model_df['price'], pred_val_fe):,.0f}")
print(f"Feature Engineered OLS (+sqft_living15) -> Val RMSE: {root_mean_squared_error(val_model_df['price'], pred_val_fe):,.0f}")
print(f"sqft_living15 coefficient p-value: {feature_engineered_model.pvalues['sqft_living15']:.4f}")


### Output Interpretation

`sqft_living15` eklenmesi Val MAE/RMSE'yi **pratik olarak hiç değiştirmiyor** ve katsayısı
**anlamlı değil** (p≈0.79). Bu, Section 19'daki dışlama kararını **empirik olarak doğruluyor**:
bu feature'ı eklemiş olsaydık bile anlamlı bir fark yaratmayacaktı — hem prediction-time
belirsizliği hem de düşük açıklayıcı gücü nedeniyle dışarıda tutulması gerekçelendirilmiş
oluyor.

### Hands-On Implementation — Specification 3: Interaction Model (`grade` × `condition`)

Section 13/22'de `condition`'ın etkisinin `grade`'e bağlı olarak değişebileceğini (confounding)
bulmuştuk. Bir interaction terimi bu etkileşimi doğrudan modelleyebilir.


In [ ]:
train_model_df["condition_centered"] = train_model_df["condition"] - 3
val_model_df["condition_centered"] = val_model_df["condition"] - 3

interaction_formula = ols_formula + " + grade_centered:condition_centered"
interaction_model = smf.ols(interaction_formula, data=train_model_df).fit()
pred_val_interaction = interaction_model.predict(val_model_df)

print(f"Interaction OLS (grade × condition) -> Val MAE: {mean_absolute_error(val_model_df['price'], pred_val_interaction):,.0f}")
print(f"Interaction OLS -> Val RMSE: {root_mean_squared_error(val_model_df['price'], pred_val_interaction):,.0f}")
print(f"Interaction term p-value: {interaction_model.pvalues['grade_centered:condition_centered']:.4g}")


### Output Interpretation

Interaction terimi **son derece anlamlı** (p≈7.4×10⁻³⁷) ve Val MAE'de küçük bir iyileşme
sağlıyor (≈91,710 → ≈90,877). Bu, `grade` ve `condition`'ın etkisinin **birbirine bağlı**
olduğunu destekliyor — ama **unutmayın**: bu, "grade ve condition birbirini nedensel olarak
etkiliyor" anlamına gelmez, yalnızca `price`'a olan **birleşik etkilerinin** doğrusal
toplamdan farklı olduğu anlamına gelir.

### Hands-On Implementation — Specification 4: Polynomial Model (`sqft_living²`)

Section 10'da `sqft_living`'in büyük ölçüde doğrusal olduğunu bulmuştuk, ama en uç değerlerde
LOWESS'in hafifçe ayrıldığını da not etmiştik. Bunu test edelim.


In [ ]:
SQFT_LIVING_MEAN = train_model_df["sqft_living"].mean()
train_model_df["sqft_living_centered"] = train_model_df["sqft_living"] - SQFT_LIVING_MEAN
train_model_df["sqft_living_centered_sq"] = train_model_df["sqft_living_centered"] ** 2
val_model_df["sqft_living_centered"] = val_model_df["sqft_living"] - SQFT_LIVING_MEAN
val_model_df["sqft_living_centered_sq"] = val_model_df["sqft_living_centered"] ** 2

polynomial_formula = ols_formula.replace("sqft_living +", "sqft_living_centered + sqft_living_centered_sq +")
polynomial_model = smf.ols(polynomial_formula, data=train_model_df).fit()
pred_val_poly = polynomial_model.predict(val_model_df)

print(f"Polynomial OLS (+sqft_living²) -> Val MAE: {mean_absolute_error(val_model_df['price'], pred_val_poly):,.0f}")
print(f"Polynomial OLS -> Val RMSE: {root_mean_squared_error(val_model_df['price'], pred_val_poly):,.0f}")
print(f"sqft_living² coefficient p-value: {polynomial_model.pvalues['sqft_living_centered_sq']:.4g}")


### Output Interpretation

`sqft_living²` terimi **son derece anlamlı** (p < 10⁻¹⁸⁸) ve en büyük Val MAE iyileşmesini
sağlıyor (≈91,710 → ≈88,857) — bu, modelimizin önceden kaçırdığı gerçek bir nonlinear örüntü
olduğunu doğruluyor.

### Hands-On Implementation — Specification 5: Log-Target OLS

Section 06/23'te `price`'ın ham dağılımının çok çarpık olduğunu ve residual'ların normal
olmadığını bulmuştuk. Log-target bir model deneyelim.


In [ ]:
train_model_df["log_price"] = np.log(train_model_df["price"])
log_target_formula = ols_formula.replace("price ~", "log_price ~")
log_target_model = smf.ols(log_target_formula, data=train_model_df).fit()

print(f"Log-target model R² (log scale): {log_target_model.rsquared:.4f}")


### Hands-On Implementation — Back-Transformation & Retransformation Bias

$E[price] \neq \exp(E[\log(price)])$ olduğu için, tahminleri `exp()` ile geri dönüştürmek
**sistematik olarak düşük tahmine (downward bias)** yol açar — buna **retransformation bias**
denir. Bunu iki yöntemle düzeltiyoruz: (1) hata teriminin normal dağıldığı varsayımına dayanan
klasik düzeltme, (2) **Duan's Smearing Estimator** — normallik varsaymayan, nonparametrik bir
düzeltme.


In [ ]:
pred_log_val = log_target_model.predict(val_model_df)

# (1) Classical correction, assumes normally distributed residuals.
normal_correction_factor = np.exp(log_target_model.mse_resid / 2)

# (2) Duan's smearing estimator — nonparametric, uses TRAINING residuals only (no normality assumption).
smearing_factor = np.mean(np.exp(log_target_model.resid))

pred_val_uncorrected = np.exp(pred_log_val)
pred_val_normal = np.exp(pred_log_val) * normal_correction_factor
pred_val_smeared = np.exp(pred_log_val) * smearing_factor

print(f"Normal correction factor : {normal_correction_factor:.5f}")
print(f"Duan smearing factor     : {smearing_factor:.5f}")
print()
for name, pred in [("Uncorrected exp()", pred_val_uncorrected), ("Normal correction", pred_val_normal), ("Duan smearing", pred_val_smeared)]:
    mae = mean_absolute_error(val_model_df["price"], pred)
    rmse = root_mean_squared_error(val_model_df["price"], pred)
    print(f"  {name:20s}: Val MAE={mae:,.0f}  Val RMSE={rmse:,.0f}")


### Output Interpretation

Hiçbir düzeltme yapmamak (`Uncorrected exp()`), en yüksek (en kötü) MAE/RMSE'yi veriyor —
retransformation bias **gerçek ve ölçülebilir**. Normal-dağılım düzeltmesi ile Duan's Smearing
arasındaki fark burada **ihmal edilebilir** (çarpanlar ≈1.0168 vs ≈1.0165) — bu, Section 23'te
residual'ların normal olmadığını bulmuş olsak da, bu **özel düzeltme faktörü** için sapmanın
pratikte küçük olduğunu gösteriyor. Daha genel bir kural olarak, residual'lar normal değilken
**Duan's Smearing tercih edilmelidir** (normallik varsaymadığı için teorik olarak daha
güvenlidir) — bu capstone'da **Duan's Smearing**'i kullanacağız.

Log-target model, **≈76,605 USD Val MAE** ile **tüm specification'lar arasında en düşük
hatayı** veriyor — Initial OLS'in (≈91,710) **%16.5 altında**.

### Student Exercise

Log-target modelin Breusch-Pagan ve Jarque-Bera test sonuçlarını (Section 23'teki kod
hücrelerini bu modele uygulayarak) hesaplayıp, raw-price modelin sonuçlarıyla karşılaştırın.


In [ ]:
# Student Exercise:
# TODO: het_breuschpagan(log_target_model.resid, log_target_model.model.exog) çalıştırın.
# TODO: jarque_bera(log_target_model.resid) çalıştırın.
# TODO: Sonuçları Section 23'teki raw-price model sonuçlarıyla (LM≈2485.8, JB≈3,226,408) karşılaştırın.

# bp_log = het_breuschpagan(log_target_model.resid, log_target_model.model.exog)
# jb_log = jarque_bera(log_target_model.resid)
# print(bp_log[:2], jb_log[:2])


### Checkpoint

- [x] Beş alternatif specification (Refined, Feature Engineered, Interaction, Polynomial,
  Log-Target) kuruldu ve her biri Initial OLS ile validation verisinde karşılaştırıldı.
- [x] `sqft_living15` eklemenin (Feature Engineered) pratikte fark yaratmadığı — Section 19'un
  dışlama kararını doğrulayan — bulgusu elde edildi.
- [x] Retransformation bias somut sayılarla gösterildi; Duan's Smearing (normallik
  varsaymayan) yöntemi, normal-dağılım düzeltmesine karşı değerlendirildi ve tercih edildi.
- [x] Log-Target OLS, tüm specification'lar arasında en düşük Val MAE'yi (≈76,605 USD) verdi.


---
## Section 28 — Model Comparison & Selection

### Learning Objective

Altı OLS specification'ını tek bir karşılaştırma tablosunda birleştirmek; **validation
verisine dayanarak** (test verisine bakmadan) nihai modeli seçmek.

### Hands-On Implementation — Full Comparison Table


In [ ]:
def val_r2(pred):
    return 1 - np.sum((val_model_df["price"] - pred) ** 2) / np.sum((val_model_df["price"] - val_model_df["price"].mean()) ** 2)


comparison_rows = [
    {"Model": "Initial OLS", "Val_MAE": mean_absolute_error(val_model_df["price"], initial_ols_model.predict(val_model_df)),
     "Val_RMSE": root_mean_squared_error(val_model_df["price"], initial_ols_model.predict(val_model_df)),
     "Val_R2": val_r2(initial_ols_model.predict(val_model_df)), "AIC": initial_ols_model.aic, "BIC": initial_ols_model.bic,
     "AIC_Comparable": "Yes (raw price scale)"},
    {"Model": "Refined OLS (influence-adjusted)", "Val_MAE": mean_absolute_error(val_model_df["price"], pred_val_refined),
     "Val_RMSE": root_mean_squared_error(val_model_df["price"], pred_val_refined),
     "Val_R2": val_r2(pred_val_refined), "AIC": refined_ols_model.aic, "BIC": refined_ols_model.bic,
     "AIC_Comparable": "Yes (raw price scale)"},
    {"Model": "Feature Engineered (+sqft_living15)", "Val_MAE": mean_absolute_error(val_model_df["price"], pred_val_fe),
     "Val_RMSE": root_mean_squared_error(val_model_df["price"], pred_val_fe),
     "Val_R2": val_r2(pred_val_fe), "AIC": feature_engineered_model.aic, "BIC": feature_engineered_model.bic,
     "AIC_Comparable": "Yes (raw price scale)"},
    {"Model": "Interaction (grade × condition)", "Val_MAE": mean_absolute_error(val_model_df["price"], pred_val_interaction),
     "Val_RMSE": root_mean_squared_error(val_model_df["price"], pred_val_interaction),
     "Val_R2": val_r2(pred_val_interaction), "AIC": interaction_model.aic, "BIC": interaction_model.bic,
     "AIC_Comparable": "Yes (raw price scale)"},
    {"Model": "Polynomial (+sqft_living²)", "Val_MAE": mean_absolute_error(val_model_df["price"], pred_val_poly),
     "Val_RMSE": root_mean_squared_error(val_model_df["price"], pred_val_poly),
     "Val_R2": val_r2(pred_val_poly), "AIC": polynomial_model.aic, "BIC": polynomial_model.bic,
     "AIC_Comparable": "Yes (raw price scale)"},
    {"Model": "Log-Target OLS (Duan smearing)", "Val_MAE": mean_absolute_error(val_model_df["price"], pred_val_smeared),
     "Val_RMSE": root_mean_squared_error(val_model_df["price"], pred_val_smeared),
     "Val_R2": val_r2(pred_val_smeared), "AIC": log_target_model.aic, "BIC": log_target_model.bic,
     "AIC_Comparable": "NO — different scale (log-price), see note below"},
]

model_comparison_df = pd.DataFrame(comparison_rows).round(2)
model_comparison_df


### Critical Engineering Notes — Raw-Price vs Log-Price AIC/BIC Are NOT Comparable

Tablodaki `Log-Target OLS` satırının AIC/BIC değeri (**çok daha negatif**) **yanıltıcı
biçimde "çok daha iyi"** görünüyor — ama bu **doğrudan karşılaştırılamaz**. AIC/BIC,
log-likelihood'a dayanır ve log-likelihood, bağımlı değişkenin **ölçeğine/dağılımına**
bağlıdır: `log(price)` değerleri (≈11-16 aralığında) ile `price` değerleri (≈75,000-7,700,000
aralığında) tamamen farklı bir olabilirlik yüzeyi üretir. **Raw-price ve log-price modellerinin
AIC/BIC'lerini doğrudan karşılaştırmıyoruz** — bu capstone'un modelleri karşılaştırma kriteri
olarak **Val MAE/RMSE/R²**'yi (orijinal USD ölçeğine geri dönüştürülmüş) kullanıyoruz; bunlar
her iki model türü için de **aynı ölçekte ve karşılaştırılabilir**.

### Output Interpretation — Model Comparison Summary

| Model | Val MAE | Val RMSE | Val R² |
|---|---|---|---|
| Initial OLS | ≈91,710 | ≈145,162 | ≈0.8203 |
| Refined OLS | ≈90,762 | ≈144,659 | ≈0.8215 |
| Feature Engineered | ≈91,699 | ≈145,144 | ≈0.8203 |
| Interaction | ≈90,877 | ≈144,633 | ≈0.8216 |
| Polynomial | ≈88,857 | ≈145,187 | ≈0.8202 |
| **Log-Target (Duan smearing)** | **≈76,605** | **≈135,785** | **≈0.8427** |

**Log-Target OLS, Val MAE/RMSE'de tüm specification'lardan açıkça daha iyi** — bu üstünlük
küçük bir fark değil, MAE'de **≈%16.5'lik** bir iyileşme. İlginç bir not: **Polynomial**
specification, Val MAE'yi önemli ölçüde iyileştiriyor (≈91,710→≈88,857) ama Val R²'yi **neredeyse
hiç değiştirmiyor** (≈0.8203→≈0.8202) — bu, R²'nin (kareli hataya dayandığı için) dataset'teki
en yüksek fiyatlı, en etkili gözlemler (Section 25'teki 7.7M USD'lik satış gibi) tarafından
domine edildiğini, MAE'nin ise (mutlak hataya dayandığı için) tipik-büyüklükteki evlerdeki
iyileşmeyi daha iyi yansıttığını gösteriyor — **tek bir metriğe güvenmemek gerektiğinin** bir
başka örneği. Log-Target OLS ayrıca Section 23'te tespit ettiğimiz heteroscedasticity ve
residual non-normality sorunlarını da (tam olarak çözmese de) **önemli ölçüde azaltıyor** (bir
sonraki hücrede doğrulayacağız).

### Hands-On Implementation — Does Log-Target Also Improve Diagnostics?


In [ ]:
log_bp_stat, log_bp_pvalue, _, _ = het_breuschpagan(log_target_model.resid, log_target_model.model.exog)
log_jb_stat, log_jb_pvalue, log_skew, log_kurt = jarque_bera(log_target_model.resid)

print("Diagnostic comparison: Raw-price (Initial OLS) vs Log-Target OLS")
print(f"  Breusch-Pagan LM : {bp_stat:,.1f} (raw)  vs  {log_bp_stat:,.1f} (log)")
print(f"  Jarque-Bera stat : {jb_stat:,.0f} (raw)  vs  {log_jb_stat:,.0f} (log)")
print(f"  Residual skewness: {jb_skew:.3f} (raw)   vs  {log_skew:.3f} (log)")
print(f"  Residual kurtosis: {jb_kurtosis:.3f} (raw)  vs  {log_kurt:.3f} (log)")


### Output Interpretation

Log-target model, Breusch-Pagan istatistiğini **≈2,485.8'den ≈1,263.3'e** (yaklaşık yarıya)
düşürüyor — heteroscedasticity **hâlâ istatistiksel olarak anlamlı** (tamamen ortadan
kalkmıyor), ama **önemli ölçüde azalıyor**. Residual skewness **≈3.76'dan ≈-0.27'ye**
düşüyor — bu **çarpıcı bir iyileşme**, residual'lar artık neredeyse simetrik. Kurtosis
**≈74.2'den ≈6.28'e** düşüyor — hâlâ normal dağılımın (`kurtosis=3`) üzerinde, ama **çok daha
yakın**. **Bu varsayım ihlallerinin tamamen ortadan kalktığını iddia etmiyoruz** — yalnızca
**önemli ölçüde azaldığını**, hem validation performansı hem de istatistiksel varsayımlar
açısından Log-Target'ı **en savunulabilir seçim** yaptığını gösteriyoruz.

### Hands-On Implementation — Final Model Selection


In [ ]:
FINAL_MODEL_NAME = "Log-Target OLS (Duan smearing correction)"
FINAL_RETRANSFORM_FACTOR = smearing_factor

print(f"Selected final model: {FINAL_MODEL_NAME}")
print(f"Selection basis: validation MAE/RMSE/R² (NOT test data)")
print(f"Final retransformation factor (learned from training residuals only): {FINAL_RETRANSFORM_FACTOR:.5f}")


### Critical Engineering Notes

Bu seçim **yalnızca validation verisine** dayanmaktadır — test verisi bu karara **hiç
dahil edilmedi**. `FINAL_RETRANSFORM_FACTOR`, **training residual'larından** hesaplandı (val/
test'ten değil) — bu, CP08 (preprocessing yalnızca training'e fit edilir) kuralının log-target
modeller için doğal bir uzantısıdır.

### Student Exercise

`model_comparison_df` tablosundaki `Val_R2` sütununu kullanarak, hangi iki modelin birbirine en
yakın performansa sahip olduğunu bulun. Bu iki model arasındaki fark, sizce **pratik olarak**
önemli mi, yoksa ihmal edilebilir mi?


In [ ]:
# Student Exercise:
# TODO: model_comparison_df'yi Val_R2'ye göre sıralayıp en yakın iki modeli bulun.
# TODO: Bu iki modelin Val_MAE farkının (USD cinsinden) pratik önemini yorumlayın.

# print(model_comparison_df.sort_values("Val_R2", ascending=False))


### Checkpoint

- [x] Altı specification, tutarlı bir karşılaştırma tablosunda birleştirildi.
- [x] Raw-price ve log-price AIC/BIC'lerinin **doğrudan karşılaştırılamayacağı** açıkça
  belirtildi ve bunun yerine Val MAE/RMSE/R² kullanıldı.
- [x] **Log-Target OLS**, hem validation performansı (%16.5 MAE iyileşmesi) hem de azaltılmış
  (ama tam ortadan kalkmamış) varsayım ihlalleri temelinde **nihai model** olarak seçildi.
- [x] Final seçim **yalnızca validation verisine** dayandırıldı — test verisi kullanılmadı.


---
## Section 29 — Final Test Evaluation

### Learning Objective

Seçilmiş nihai modeli (**Log-Target OLS, Duan smearing**), test verisinde **bir kez** ve
**bağımsız olarak** değerlendirmek.

### Critical Rule

**Bu, bu capstone'da test verisinin kullanıldığı TEK yerdir.** Test sonuçlarını gördükten sonra
model veya specification seçimini **değiştirmiyoruz**.

### Hands-On Implementation — Preparing the Test Set (Schema Validation Only)


In [ ]:
test_model_df = engineer_features(test_df, GRADE_MEAN, AGE_MEAN)

unseen_zip_test = set(test_model_df["zipcode"]) - set(train_model_df["zipcode"])
unseen_quarter_test = set(test_model_df["sale_quarter"]) - set(train_model_df["sale_quarter"])

print(f"Test set shape: {test_model_df.shape}")
print(f"Zipcodes in test not seen in training: {unseen_zip_test if unseen_zip_test else 'None'}")
print(f"Sale quarters in test not seen in training: {unseen_quarter_test if unseen_quarter_test else 'None'}")

assert not unseen_zip_test and not unseen_quarter_test
print("Schema check PASSED — test set can be safely scored with the training-fitted model.")


### Hands-On Implementation — One-Time Final Evaluation


In [ ]:
from sklearn.metrics import median_absolute_error

pred_log_test = log_target_model.predict(test_model_df)
pred_test_final = np.exp(pred_log_test) * FINAL_RETRANSFORM_FACTOR

test_mae = mean_absolute_error(test_model_df["price"], pred_test_final)
test_mse = np.mean((test_model_df["price"] - pred_test_final) ** 2)
test_rmse = root_mean_squared_error(test_model_df["price"], pred_test_final)
test_r2 = 1 - np.sum((test_model_df["price"] - pred_test_final) ** 2) / np.sum((test_model_df["price"] - test_model_df["price"].mean()) ** 2)
test_median_ae = median_absolute_error(test_model_df["price"], pred_test_final)
test_abs_errors = np.abs(test_model_df["price"] - pred_test_final)
test_p90_ae = np.percentile(test_abs_errors, 90)

print(f"Test MAE              : {test_mae:,.0f}")
print(f"Test MSE              : {test_mse:,.0f}")
print(f"Test RMSE             : {test_rmse:,.0f}")
print(f"Test R²               : {test_r2:.4f}")
print(f"Test Median Abs Error : {test_median_ae:,.0f}")
print(f"Test P90 Abs Error    : {test_p90_ae:,.0f}")


### Output Interpretation

Test R² (**≈0.842**), validation R²'ye (**≈0.843**) **son derece yakın** — model validation
setine aşırı uydurulmamış, zaman içinde (test seti validation'dan kronolojik olarak **sonraki**
bir dönemi kapsıyor) genellemesini **korumuş**. Test MAE (**≈86,227 USD**), validation MAE'den
(**≈76,605 USD**) biraz daha yüksek — bu, Section 04'te not ettiğimiz "test dönemi farklı bir
sipariş/satış karması içerebilir" riskiyle tutarlı bir sapma.

**Median Absolute Error (≈52,144 USD)**, ortalama MAE'den (≈86,227) **önemli ölçüde düşük** —
bu, hata dağılımının **sağa çarpık** olduğunu gösteriyor: çoğu tahmin oldukça iyi, ama az
sayıda büyük hata ortalamayı yukarı çekiyor (Section 25'teki 7.7M USD'lik ev gibi örnekler).
**P90 Absolute Error (≈185,399 USD)**, tahminlerin %90'ının bu hata payının altında olduğunu
gösteriyor — bu, ortalama tek bir sayıdan daha **zengin bir hata profili** sunuyor.

### Hands-On Implementation — Error Segmentation


In [ ]:
test_segmentation = test_model_df.copy()
test_segmentation["abs_error"] = test_abs_errors

zipcode_error_segmentation = (
    test_segmentation.groupby("zipcode")["abs_error"].agg(["mean", "count"]).sort_values("mean", ascending=False)
)
zipcode_error_segmentation.head(5)


### Output Interpretation

En yüksek ortalama mutlak hataya sahip 5 zipcode (`98112`, `98039`, `98004`, `98040`, `98105`),
Section 11'de en yüksek medyan fiyata sahip olarak tespit ettiğimiz **aynı lüks bölgelerdir**.
Bu, Section 23/25'te bulduğumuz heteroscedasticity ve influence bulgularıyla **tamamen
tutarlı**: model, **yüksek fiyatlı, lüks segmentteki konutları** sistematik olarak daha büyük
mutlak hatayla tahmin ediyor — bu bir sürpriz değil, önceki tüm bulgularımızın **doğal bir
sonucu**.

### Important

**Test sonuçlarını gördük — model veya specification seçimini şimdi değiştirmiyoruz.** Bu,
seçilen modelin nihai, bağımsız performans raporudur.

### Checkpoint

- [x] Test verisinde training'de görülmemiş kategori olmadığı doğrulandı.
- [x] Final model (**Log-Target OLS, Duan smearing**) test verisinde **bir kez** değerlendirildi:
  MAE≈86,227, RMSE≈149,573, R²≈0.842.
- [x] Test ve validation performansının birbirine yakın olduğu (overfitting yok) doğrulandı.
- [x] Hata dağılımının sağa çarpık olduğu (Median AE ≪ Mean MAE) ve en yüksek hatanın lüks
  zipcode'larda yoğunlaştığı (Section 11/23/25 ile tutarlı) gösterildi.
- [x] Test sonuçlarından sonra model seçiminin **değiştirilmediği** açıkça teyit edildi.


---
## Section 30 — Final Capstone Report

### Learning Objective

Capstone'un tüm bulgularını, teknik olarak savunulabilir, kanıta dayalı tek bir rapora
toplamak.

### Business Problem

King County'deki geçmiş konut satışlarından (2014-2015) öğrenerek, konut özelliklerine dayalı
bir **Historical Property Valuation** modeli geliştirmek. Bu, gerçek zamanlı bir production
valuation sistemi **değildir**.

### Dataset Assessment

- **21,613 gözlem, 21 kolon** — Kaggle'ın belirttiği profille (gerçek CSV okunarak) tam olarak
  doğrulandı.
- **21,436 benzersiz property ID** (177 tekrarlanan satış — aynı mülkün farklı tarihlerde
  yeniden satılması).
- **0 gerçek missing value, 0 exact duplicate row.**
- Tarih aralığı: **2014-05-02 → 2015-05-27**.

### Data Quality

- `bedrooms=33` (1 kayıt) ve `bedrooms/bathrooms=0` (13/10 kayıt) **flag'lendi, otomatik
  silinmedi**; `bedrooms=33` yalnızca training fit'inden çıkarıldı (Section 19), diğerleri
  korundu.
- `yr_built`/`yr_renovated` > `sale_year` (12+6 kayıt) — her biri tam olarak **+1 yıl** sapma,
  muhtemelen "off-plan" satışlar; **silinmedi, flag'lendi**.
- `yr_renovated=0` ve `sqft_basement=0`, **missing value değil**, structural zero olarak doğru
  yorumlandı.
- `sqft_above + sqft_basement = sqft_living` kimliği **%100 doğrulandı** — Section 18'in
  temelini oluşturdu.

### EDA Findings

- `price`, aşırı sağa çarpık (skewness≈4.26, kurtosis≈39.5); log-transform bunu önemli ölçüde
  azaltıyor (skewness≈0.43).
- `sqft_living` (r≈0.71) ve `grade` (r≈0.67), `price` ile en güçlü doğrusal ilişkiye sahip.
- `zipcode`'un ham korelasyonu ≈0 olmasına karşın, en yüksek/en düşük medyan fiyatlı zipcode
  arasında **9x fark** var — korelasyonun nominal değişkenler için yanıltıcı olabileceğinin
  somut bir kanıtı.
- `condition`-`price` ilişkisinde gerçek bir **confounding** örneği bulundu: `grade` kontrol
  edilmeden `condition`'ın etkisi gizli kalıyor.
- Altı hipotez (H1-H6), gerçek EDA bulgularına dayanarak kayda geçirildi ve sonraki
  section'larda **istatistiksel olarak test edildi**.

### Feature Engineering

Sekiz domain-based feature türetildi (`house_age_at_sale`, `is_renovated_at_sale`,
`years_since_renovation`, `has_basement`, `basement_ratio`, `living_area_per_bedroom`,
`sale_month`, `sale_quarter`); her biri leakage riski açısından değerlendirildi (**hepsi
"None"**). `grade`/`house_age_at_sale`, kare terimi oluşturmadan önce **merkezlendi**
(centering) — VIF'i ≈100'den ≈1.6'ya düşürdü.

### Feature Selection

Üç katmanlı süreç: (1) **Domain-based** — perfect-collinear ve yüksek-missingness feature'lar
dışlandı; (2) **Statistical** — centering ile VIF kontrol altına alındı, full column rank
doğrulandı; (3) **Validation-based** — `zipcode`+`sale_quarter`'ın dahil edilmesi %25.7 Val MAE
iyileşmesiyle gerekçelendirildi.

### Multicollinearity Analysis

**Perfect multicollinearity** (`sqft_living`+`sqft_above`+`sqft_basement`), rank-deficiency
(4/5) ve VIF≈10¹⁵ ile gösterildi. **High multicollinearity**, SE enflasyonu (2.08x) ve katsayı
işaret değişimiyle gösterildi. **Condition Number**'ın ölçeğe duyarlı (2.3M ham / 4.91
standartlaştırılmış), **VIF**'in ölçekten bağımsız olduğu kanıtlandı. Nihai modelde tüm numeric
feature VIF'leri **<5**.

### OLS Statistical Results

Initial OLS: **R²≈0.836, Adj. R²≈0.835, F≈882.3 (p≈0)**. `sqft_living`, `condition`, `grade`
(ve karesi), `waterfront`, `bedrooms` bireysel olarak anlamlı; `zipcode` joint F-testiyle
(F≈142.3) güçlü biçimde anlamlı; `sale_quarter` joint olarak **anlamlı değil** (p≈0.24).

### Assumption Diagnostics

- **Homoscedasticity: VIOLATED** (Breusch-Pagan p≈0).
- **Residual Normality: VIOLATED** (Jarque-Bera p≈0, skewness≈3.76).
- **Error Independence: OK** (Durbin-Watson≈1.97).
- **Specification: eksiklikler var** (Ramsey RESET p≈0; bir numerik ölçek-hassasiyeti uyarısı
  araştırılıp test sonucunun geçerli olduğu doğrulandı).
- **Conditional exogeneity: doğrulanamaz** — bu, OLS'in kalıcı bir sınırlamasıdır.

### Influence Analysis

En etkili gözlem (dataset'in en yüksek fiyatlı satışı, 7.7M USD) Cook's Distance ile tespit
edildi; çıkarılması katsayılarda %1-6 değişime yol açtı (orta düzeyde hassasiyet) — **silinmedi**,
bir sensitivity specification'ına (Refined OLS) girdi oldu.

### Model Comparison

Altı specification karşılaştırıldı (Initial, Refined, Feature Engineered, Interaction,
Polynomial, Log-Target). **Log-Target OLS (Duan smearing)**, Val MAE'de **%16.5 iyileşme**
(≈91,710→≈76,605 USD) ve azaltılmış (ama ortadan kalkmamış) heteroscedasticity/non-normality
ile **nihai model** olarak seçildi. Raw-price/log-price AIC'lerinin doğrudan
karşılaştırılamayacağı not edildi.

### Final Test Performance

**Test MAE≈86,227, RMSE≈149,573, R²≈0.842, Median AE≈52,144, P90 AE≈185,399** (USD). Test
R², validation R²'ye (≈0.843) çok yakın — overfitting kanıtı yok. En yüksek hata, en yüksek
fiyatlı (lüks) zipcode'larda yoğunlaşıyor.

### Limitations

- Bu, gerçek bir production valuation sistemi değildir; 2014-2015 King County verisine özgüdür.
- `sqft_living15`/`sqft_lot15`'in prediction-time erişilebilirliği **doğrulanmamıştır** (bu
  nedenle dışarıda tutuldu).
- Heteroscedasticity ve residual non-normality, log-target ile **azaltıldı ama ortadan
  kalkmadı**.
- Conditional exogeneity **doğrulanamaz** — gözlenmeyen karıştırıcı değişkenler (örn. "mahalle
  prestiji") katsayıları etkileyebilir.
- Feature seçimi training verisi üzerinde yapıldı — **post-selection inference** sınırlaması
  geçerlidir.
- `sale_month`/`sale_quarter`, training verisinde kapsanmayan ayları (Şubat-Nisan) validation/
  test'te ekstrapole ediyor.

### Conclusions

Bu capstone, **güçlü bir validation/test performansı** (R²≈0.84) ile **dürüstçe raporlanmış
istatistiksel sınırlamaları** birleştiren bir OLS modeli üretti. En düşük hataya sahip modelin
(Log-Target OLS) seçimi, hem **validation kanıtına** hem de **varsayım ihlallerinin azalmasına**
dayanıyor — ama bu, modelin tüm OLS varsayımlarını karşıladığı veya production'a hazır olduğu
anlamına **gelmiyor**. Katılımcılar, bu capstone boyunca her bulguyu (korelasyon, multicollinearity,
hipotez testi, residual diagnostics, influence, model karşılaştırması) **gerçek veriyle
hesaplayıp doğrulamayı** ve sonuçları **teknik olarak savunmayı** öğrendi.

### Checkpoint

- [x] Final Capstone Report, master prompt'ta istenen 14 başlığın tümünü kapsayacak şekilde
  tamamlandı.
- [x] Tüm bulgular, notebook içinde gerçekten hesaplanmış sayılara dayanıyor — hiçbiri uydurma
  değil.

---

## Session 6 Özeti

Bu son oturumda:

- **Influence Diagnostics** ile dataset'in en etkili gözlemini (7.7M USD'lik satış) tespit edip
  bir sensitivity analysis yaptık.
- **HC3 Robust Standard Errors**'ın katsayıları değiştirmediğini, ama standart hataları (1.0x-
  4.25x) büyüttüğünü gösterdik — ve bunun endogeneity/specification sorunlarını **çözmediğini**
  vurguladık.
- **Beş alternatif OLS specification** kurduk; `sqft_living15` eklemenin fark yaratmadığını
  (Section 19 kararını doğrulayarak), interaction ve polynomial terimlerin anlamlı katkı
  sağladığını, ve **Log-Target OLS**'in en iyi validation performansını verdiğini bulduk.
- Raw-price/log-price AIC'lerinin karşılaştırılamayacağını açıkça belirterek, **Val MAE/RMSE/R²**
  temelinde nihai modeli (**Log-Target OLS, Duan smearing**) seçtik.
- Bu modeli test verisinde **bir kez** değerlendirdik (R²≈0.842) ve sonuçlardan sonra seçimi
  değiştirmedik.
- Tüm capstone'u, 14 başlıklı bir **Final Capstone Report**'ta topladık.

---

## Capstone Tamamlandı

Bu, **LAB 05 Capstone Project: Multiple Linear Regression**'ın sonudur. Katılımcılar, veri
kalitesi değerlendirmesinden, EDA'ya, feature engineering'e, multicollinearity analizine, OLS
istatistiksel çıkarımına, residual diagnostics'e, influence analizine ve nihai model seçimine
kadar **tam bir uygulamalı regresyon analizi iş akışını** gerçek veriyle, gerçek sayılarla ve
dürüst sınırlama raporlamasıyla tamamladı.
